# 「미지의 전북」 뉴스·SNS 관심대상 재분석 노트북 (v2, 과업지시서 v2 반영)

**목적.** 전북 14개 시군 관련 뉴스·인스타그램·유튜브 자료에서 *지역에 연결된 구체적인 관심대상*(장소·먹거리·행사·기업·산업·정책·사회 현상)을 찾습니다. 관심의 규모(등장 콘텐츠 수·비중·출처 다양성), SNS 반응(조회수·좋아요·댓글, 상대 반응 점수), 변화 유형(HOT·RISING·STEADY·SEASONAL·EVENT/VIRAL)을 근거 원문과 함께 제시합니다.

**과업지시서 v2의 핵심 원칙(이 노트북에서 지키는 것)**
- 빈도의 기본 단위는 *고유 콘텐츠 수(DF)* 입니다. 반복군(뉴스 기사군, SNS 동일 본문·출처 반복)은 보조 지표로 나란히 둡니다.
- 단독 지역명·범용어·정당명·인명·기관명은 핵심 순위에서 빼고 보조 정보로 보존합니다. 관련된 구체적인 사업·행사·공간은 유지합니다.
- 시군은 '수집 당시 시군'이 아니라 *콘텐츠 원문과 대상 이름이 가리키는 시군*으로 귀속합니다. 근거가 없으면 '지역미확정'으로 둡니다.
- 2023년 1~10월은 수집 방식이 다른 보완 수집분입니다. 분석을 **A(전체 관측자료 기반 발굴)**, **B(2024~2026 비교 가능한 층의 주 추세)**, **C(2023 포함 민감도)** 로 나눕니다.
- 결측과 실제 0을 구분하고, 원시 반응 수치를 매체끼리 더하지 않습니다. 기준값은 `CONFIG`에 모으고 '초기 기준'으로 표시합니다(발주자 승인 전).

**실행 방법.** 위에서부터 차례로 실행합니다. 마지막 단계에서 `output/`에 29개 시트 Excel, 전체 원장 CSV(UTF-8 BOM)·Parquet, manifest를 저장합니다.

| 단계 | 내용 | 과업지시서 |
|---|---|---|
| 0~1 | 환경·파라미터, 원자료 불러오기(입력 해시) | 4, 20장 |
| 2 | 공통 형식 표준화(원본ID, 형식, 수집구간·수집층, 검토등급) | 3, 5장 |
| 3 | 자료 점검 | 4장 |
| 4 | 전처리(띄어쓰기 복원, 정제, 기사군·반복군, 지역 관련성) | 5, 8장 |
| 5~6 | 후보 추출, 표현 통합·판정 → 콘텐츠×관심대상 원장 | 6, 7, 9장 |
| 7 | 인명·기관 단독 제외, 지역 연결 불명 식품 처리, 6개 분야 분류 | 6, 11장 |
| 8 | 콘텐츠 마스터·수집층·비교그룹 | 4, 17장 |
| 9 | 시군 귀속·근거 원장(키워드·지역·관계 등급) | 8~10장 |
| 10 | SNS 반응 상태·상대 반응 점수 | 13~14장 |
| 11 | 공통 집계 엔진, 매체별·시군별·분야별·연도계절·월별 집계 | 12, 15, 18장 |
| 12 | HOT·RISING·STEADY·SEASONAL·EVENT/VIRAL | 16장 |
| 13 | 매체 비교, 대표 콘텐츠 | 18, 19장 |
| 14 | 수집 편향 민감도(A/B/C), 기타 민감도 | 17장 |
| 15 | 검토 목록, 검수(자동 검산·검수 표본), 변경 이력 | 21~22, 1장 |
| 16 | 저장(29개 시트 + 원장 파일 + manifest) | 19, 20장 |

**폴더 구조**
```
keyword_analysis/
├── 미지의전북_키워드분석.ipynb
├── 뉴스기사_최종유지_271159건.csv                      ← 원자료(수정하지 않음)
├── 전북SNS_수집결과_20230101_20261031_revised3.xlsx  ← 원자료(수정하지 않음)
├── dict/     ← 사전(불용어·보조어·지역명·표현통합·분야규칙·지정인명·지역연결필요식품 등)
├── cache/    ← 중간 결과 캐시(삭제하면 다시 계산)
└── output/   ← 결과물
```

> **해석 원칙.** 뉴스 보도량은 언론의 주목 신호이고, SNS 언급·반응은 *수집된 자료 안*의 생산·소비 신호입니다. 빈도 상위를 도민 선호로, 조회수 합계를 고유 관심자 수로, 시군 간 데이터량 차이를 관심도 우열로 해석하지 않습니다.

## 0. 환경 설정

In [ ]:
# ── 표준 라이브러리 ──────────────────────────────────────────────
import os, re, json, sys, time, platform, unicodedata, warnings
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

# ── 외부 라이브러리 ──────────────────────────────────────────────
import numpy as np
import pandas as pd
from kiwipiepy import Kiwi            # 한국어 형태소 분석기
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

# 그래프 한글 글꼴 (macOS: AppleGothic, Windows: Malgun Gothic, Linux: NanumGothic)
for _font in ["AppleGothic", "Malgun Gothic", "NanumGothic"]:
    if any(_font in f.name for f in matplotlib.font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = _font
        break
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# ── 경로 ─────────────────────────────────────────────────────────
BASE_DIR  = Path.cwd()                      # 노트북이 있는 폴더(keyword_analysis)에서 실행
RAW_NEWS  = BASE_DIR / "뉴스기사_최종유지_271159건.csv"
RAW_SNS   = BASE_DIR / "전북SNS_수집결과_20230101_20261031_revised3.xlsx"   # 2023.1.1~10.30 보완 수집분 포함, 정제(revised2)·추가 수집 병합(revised3)
DICT_DIR  = BASE_DIR / "dict"
CACHE_DIR = BASE_DIR / "cache"
OUT_DIR   = BASE_DIR / "output"
for _d in (CACHE_DIR, OUT_DIR):
    _d.mkdir(exist_ok=True)

# ── 분석 파라미터 ────────────────────────────────────────────────
# 모든 기준값을 이곳에 모았습니다. 값을 바꾸면 이후 단계에 그대로 반영됩니다.
# '잠정'으로 표시한 값은 자료 분포를 확인한 뒤 의뢰자와 협의해 확정할 값입니다.
CONFIG = {
    # ── 기간(과업지시서 3장) ─────────────────────────────────────
    "COLLECTION_DATE": "2026-10-31",                 # SNS 기본 구간 수집일(보완 구간 수집일은 2026-11-05)
    "COMMON_START": "2023-01-01",                    # 세 매체 공통 비교기간 시작
    "COMMON_END":   "2026-10-26",                    # 세 매체 공통 비교기간 종료(뉴스 종료일)
    "SNS_REF_END":  "2026-10-30",                    # SNS 10/27~10/30은 별도 참고 집계만(주 결과와 섞지 않음)
    "SUPP_SEGMENT_END": "2023-10-30",                # 2023 보완 수집 구간 끝(수집기준 시트의 실제 경계)
    "YEAR_WINDOW":  ("01-01", "10-26"),              # 연도 비교 동기간(2026 관측 종료일에 맞춤)
    "YEAR_WINDOW_SENS": ("01-01", "09-30"),          # 민감도: 완전한 달(1~9월)
    "CUR_YEAR": 2026,
    "BASE_YEAR": 2025,                               # RISING 기본 비교연도(2026 대 2025 동기간)
    "AUX_BASE_YEARS": [2024, 2025],                  # 트랙 B 보조 기준: 연도별 비중의 동일가중 평균
    "C_BASE_YEARS": [2023, 2024, 2025],              # 트랙 C(2023 포함 민감도) 기준
    "SEASONS": {"봄": [3, 4, 5], "여름": [6, 7, 8], "가을": [9, 10, 11], "겨울": [12, 1, 2]},

    # ── 전처리 ──────────────────────────────────────────────────
    "NEWS_DUP_WINDOW_DAYS": 3,        # 정규화한 제목이 같고 이 일수 안이면 같은 기사군(보조 지표)
    "NEWS_EXCLUDE_REVIEW": False,     # True면 '원문확인' 판정 기사 제외
    "SNS_RELEVANCE": "standard",      # strict: 본문/위치 단서만, standard: +다른 지역 태그, loose: 전체
    "YT_DESC_MAXLEN": 1500,           # 유튜브 설명란은 앞부분만 사용(주소·링크·협찬 문구가 뒤에 몰려 있음)
    "YT_BOILERPLATE_TAG_SHARE": 0.5,  # 영상 10~29개 채널: 영상의 50% 이상에 반복되는 태그는 채널 상투 태그
    "YT_BOILERPLATE_TAG_SHARE_BIG": 0.15,  # 영상 30개 이상 채널: 15% 이상(방송사 SEO 태그 묶음)

    # ── 후보 추출 ──────────────────────────────────────────────
    "NGRAM_MIN_DF": {"뉴스": 20, "인스타": 10, "유튜브": 5},
    "NGRAM_ASSOC": 0.25,
    "MAX_UNIT_LEN": 25,
    "SOURCE_BOILER_MIN": 10,          # 같은 채널·작성자 콘텐츠가 이 수 이상일 때 '반복 생산' 점검
    "SOURCE_BOILER_SHARE": 0.5,
    "SOURCE_BOILER_SHARE_BIG": 0.25,
    "REGION_COMPOUND_HEADS": ["새만금"],
    "REGION_COMPOUND_MIN_DF": {"뉴스": 10, "인스타": 5, "유튜브": 3},
    "CLASSIFY_MIN_N": 3,              # 이 건수 이상 등장한 후보는 모두 6개 분야 분류
    "AD_STUFF_MIN_VARIANTS": 4,       # 한 게시물 해시태그에 '지역명+같은 표현'이 이 수 이상이면 광고·홍보성 나열 콘텐츠
    "AD_STEM_AUTO_MIN": 10,           # 나열 콘텐츠가 이 수 이상인 표현은 광고·홍보성 표현으로 자동 추가

    # ── 지지조건·유형 판정(과업지시서 15~16장 초기값) ─────────────
    "MIN_SUPPORT_CONTENTS": 5,        # 고반응·RISING 최소 고유 콘텐츠 수
    "MIN_SUPPORT_SOURCES": 3,         # SNS 최소 고유 계정/채널 수
    "NEWS_MIN_PRESS": 2,              # 뉴스 최소 언론사 수(미달은 단일 언론사 후보로 별도 표시)
    "HOT_TOP_K": 30,
    "HOT_MIN_COUNT": {"뉴스": 50, "인스타": 30, "유튜브": 15},
    "HOT_MIN_DIVERSITY": {"뉴스": 3, "인스타": 10, "유튜브": 5},
    "RISING_MIN_RATIO": 1.5,          # 비중비 하한
    "RISING_MIN_PER1000_DELTA": 1.0,  # 천건당언급 증가 하한
    "RISING_MIN_MONTHS": 2,           # 현재 관측월 하한
    "STEADY_MIN_YEARS": 3, "STEADY_MIN_PER_YEAR": 3, "STEADY_MIN_TOTAL": 20, "STEADY_MAX_CV": 0.5,
    "SEASONAL_MIN_YEARS": 2, "SEASONAL_MIN_PER_SEASON": 3, "SEASONAL_MIN_INDEX": 1.5,
    "EVENT_TOP1_SHARE": 0.5,          # 반응량 상위 1개 콘텐츠 기여율
    "EVENT_MONTH_SHARE": 0.5,         # 연간 대상 콘텐츠의 한 달 집중률
    "TOP_SOURCE_MAX_SHARE": 0.5,      # 최대 출처 점유율이 이 이상이면 '단일 출처 의존'

    # ── SNS 반응(과업지시서 13~14장) ────────────────────────────
    "MIN_VIEWS_FOR_RATE": 100,        # 좋아요/조회수 비율 계산 최소 조회수
    "PCTL_MIN_GROUP": 30,             # 백분위 비교집단 최소 유효 N
    "AUTHOR_MIN_SAME_FORMAT": 10,     # 작성자 평소 반응 비교 최소 콘텐츠 수
    "HIDDEN_MIN_SCORED": 5,           # 숨은 관심대상: 반응점수가 있는 콘텐츠가 이 수 미만이면 '소수 게시물 의존'
    "HIDDEN_MAX_SOURCE_SHARE": 0.3,   # 숨은 관심대상: 한 계정 점유율이 이 이상이면 '특정 계정 의존'
    "HIDDEN_MIN_INDEP_TEXTS": 5,      # 숨은 관심대상: 같은 안내문 반복을 하나로 본 독립 문구가 이 수 미만이면 보류
    "HIDDEN_MIN_SOURCES": 5,          # 숨은 관심대상: 고유 계정이 이 수 미만이면 '특정 계정 의존' 

    # ── 보고용 상위 개수(19장) ─────────────────────────────────
    "REPORT_TOP_N": 30,               # 시군별·분야별 상위
    "TYPE_TOP_N": 20,                 # 유형별 상위
    "CROSS_TOP_K": 100,

    "PREV_VERSION": "v2_20261106_2353",   # 2차 검토 변경 내역 비교 기준(이전 결과 폴더)
    "SEED": 20261106,                 # 표본 추출 seed(검수 표본)
}
MEDIA = ["뉴스", "인스타", "유튜브"]
SIGUN = ["전주시","군산시","익산시","정읍시","남원시","김제시","완주군","진안군",
         "무주군","장수군","임실군","순창군","고창군","부안군"]
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M")
ANALYSIS_VERSION = f"v2.0-{RUN_TS}"   # 모든 집계표의 분석버전 열

In [ ]:
def environment_info():
    """재현 정보(파이썬·라이브러리 버전, 실행 시각, 원자료 크기)를 표로 반환합니다.

    결과는 마지막 단계에서 '재현정보' 시트로 저장됩니다.
    """
    import kiwipiepy, openpyxl
    rows = [
        ("실행시각", RUN_TS),
        ("python", sys.version.split()[0]),
        ("platform", platform.platform()),
        ("pandas", pd.__version__),
        ("numpy", np.__version__),
        ("kiwipiepy", kiwipiepy.__version__),
        ("openpyxl", openpyxl.__version__),
        ("matplotlib", matplotlib.__version__),
        ("뉴스 원자료", f"{RAW_NEWS.name} ({RAW_NEWS.stat().st_size/1e6:.1f} MB)"),
        ("SNS 원자료", f"{RAW_SNS.name} ({RAW_SNS.stat().st_size/1e6:.1f} MB)"),
        ("xlsxwriter", __import__("xlsxwriter").__version__),
        ("pyarrow", __import__("pyarrow").__version__),
        ("random seed", CONFIG["SEED"]),
    ]
    return pd.DataFrame(rows, columns=["항목", "값"])

ENV_INFO = environment_info()
ENV_INFO

## 1. 원자료 불러오기

- 뉴스: CSV 1개(271,159행)
- SNS: XLSX의 `인스타`, `유튜브`, `유튜브_타임라인` 시트. 수집 범위를 해석하는 데 필요한 `수집기준`, `인스타태그범위`, `유튜브검색기준` 시트도 함께 읽습니다.
- 의뢰서 범위에 따라 **댓글 본문 시트는 분석하지 않습니다**(댓글 수만 사용).

XLSX는 읽는 데 15초쯤 걸리므로, 처음 한 번 읽은 뒤 `cache/`에 parquet으로 저장해 다음부터 빨리 불러옵니다.

In [ ]:
def _cache_path(src: Path, tag: str) -> Path:
    """원본 파일의 수정시각을 이름에 넣은 캐시 경로를 만듭니다. 원본이 바뀌면 자동으로 새로 읽습니다."""
    return CACHE_DIR / f"{tag}_{int(src.stat().st_mtime)}.parquet"


def load_news(path: Path = RAW_NEWS) -> pd.DataFrame:
    """뉴스 CSV를 문자열 그대로 읽고 추적용 '원자료_행번호'를 붙입니다.

    뉴스 자료에는 URL이나 고유 기사ID가 없으므로 의뢰서 10.2에 따라 ID를 새로 만들지 않습니다.
    대신 CSV의 행번호(헤더=1행, 첫 기사=2행)를 보존해 언론사·날짜·제목과 함께 원자료를 추적합니다.
    """
    cp = _cache_path(path, "news_raw")
    if cp.exists():
        return pd.read_parquet(cp)
    df = pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[""])
    df.insert(0, "원자료_행번호", np.arange(2, len(df) + 2))
    df.to_parquet(cp)
    return df


def load_sns(path: Path = RAW_SNS) -> dict:
    """SNS 엑셀에서 분석에 쓰는 시트만 읽어 {시트명: DataFrame} 딕셔너리로 반환합니다.

    모든 열을 문자열로 읽은 뒤(앞자리 0, ID 손상 방지) 숫자 변환은 2단계에서 따로 합니다.
    """
    sheets = ["인스타", "유튜브", "유튜브_타임라인", "수집기준", "인스타태그범위", "유튜브검색기준", "요약"]
    out, missing = {}, []
    for s in sheets:
        cp = _cache_path(path, f"sns_{s}")
        if cp.exists():
            out[s] = pd.read_parquet(cp)
        else:
            missing.append(s)
    if missing:
        raw = pd.read_excel(path, sheet_name=missing, dtype=str)
        for s, df in raw.items():
            df.columns = [str(c) for c in df.columns]
            df.to_parquet(_cache_path(path, f"sns_{s}"))
            out[s] = df
    # openpyxl이 셀 안 줄바꿈(CR)을 '_x000D_' 문자열로 남긴 것을 정리(2차 검토: '진안X000D' 같은 후보 방지)
    for df in out.values():
        for c in df.columns:
            if pd.api.types.is_string_dtype(df[c]) or df[c].dtype == object:
                df[c] = df[c].str.replace("_x000D_", "", regex=False)
    return out


def file_sha256(path: Path) -> str:
    """원본 파일의 SHA-256(작업 전후 원본이 바뀌지 않았는지 검산하는 데 씀)."""
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


INPUT_HASH = {p.name: file_sha256(p) for p in (RAW_NEWS, RAW_SNS)}
print("입력 해시:", {k: v[:16] + "…" for k, v in INPUT_HASH.items()})

t0 = time.time()
news_raw = load_news()
sns_raw = load_sns()
print(f"불러오기 완료 {time.time()-t0:.1f}초")
print("뉴스:", news_raw.shape)
for k, v in sns_raw.items():
    print(f"{k}: {v.shape}")

In [ ]:
# 수집기준 시트: 이후 해석(수집 범위·편향)의 근거가 되므로 그대로 확인합니다.
sns_raw["수집기준"].head(12)

## 2. 공통 형식으로 표준화

세 매체를 **같은 열 구조**로 맞춥니다. 합치려는 것이 아니라 *같은 함수로 따로 분석*하려는 것입니다.

| 공통 열 | 뉴스 | 인스타 | 유튜브 |
|---|---|---|---|
| `content_id` | `N`+원자료 행번호 | 게시물ID | 영상ID |
| `date` | 검색표시날짜(실제 발행일 확인 불가 → 대리 날짜) | 작성일(KST) | 게시일(KST) |
| `title` / `body` | 기사제목 / 미리보기 | – / 본문 | 제목 / 설명 |
| `tags` | – | 해시태그 | 태그 |
| `source` (다양성 지표) | 언론사 | 작성자(해시) | 채널명 |
| `sigun_all` | 검색어목록 → 시군 | 시군+관련시군 | 시군+관련시군 |
| 반응 | – | 좋아요·댓글·조회수(릴스) | 조회수·좋아요·댓글 |

In [ ]:
SIGUN_BASE = {s[:-1]: s for s in SIGUN}           # '전주' → '전주시'


def to_num(s: pd.Series) -> pd.Series:
    """문자열 숫자를 실수로 바꿉니다. 빈 값·변환 불가 값은 0이 아니라 결측(NaN)으로 둡니다."""
    return pd.to_numeric(s.astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")


def split_sigun(main, related) -> list:
    """'시군'과 '관련시군'(쉼표 구분)을 합쳐 중복 없는 시군 목록으로 만듭니다. 순서는 유지합니다."""
    out = []
    for v in [main] + (str(related).split(",") if isinstance(related, str) else []):
        v = str(v).strip()
        if v and v != "nan" and v not in out:
            out.append(v)
    return out


def news_keywords_to_sigun(kw_list: list) -> list:
    """뉴스 '검색어목록'(예: ["전북","전주"])을 시군 목록으로 바꿉니다.

    시군명이 없고 광역 검색어(전북, 전북특별자치도)만 있으면 SNS 표기와 맞춰 '전북 공통'으로 둡니다.
    """
    out = [SIGUN_BASE[k] for k in kw_list if k in SIGUN_BASE]
    out = list(dict.fromkeys(out))
    return out if out else ["전북 공통"]


def standardize_news(df: pd.DataFrame) -> pd.DataFrame:
    """뉴스 원자료를 공통 형식으로 바꿉니다."""
    kws = df["검색어목록"].fillna("[]").map(json.loads)
    import hashlib
    fields = df.drop(columns=["원자료_행번호"]).fillna("").astype(str).agg("\x1f".join, axis=1)
    out = pd.DataFrame({
        "media": "뉴스",
        "content_id": "N" + df["원자료_행번호"].astype(str),
        # 원본ID: 원본 파일 해시 앞 12자 + CSV 논리 레코드 번호(헤더 다음 첫 기사 = 1). 물리 줄 번호가 아님
        "원본ID": INPUT_HASH[RAW_NEWS.name][:12] + ":" + (df["원자료_행번호"] - 1).astype(str),
        "논리레코드번호": df["원자료_행번호"] - 1,
        "원본필드해시": [hashlib.sha1(s.encode("utf-8")).hexdigest()[:16] for s in fields],
        "원자료_행번호": df["원자료_행번호"],
        "date": pd.to_datetime(df["검색표시날짜"], errors="coerce"),
        "title": df["기사제목"].fillna(""),
        "body": df["미리보기"].fillna(""),
        "tags": "",
        "timeline": "",
        "location": "",
        "source": df["언론사"],
        "search_keywords": kws,
        "sigun_all": kws.map(news_keywords_to_sigun),
        "type": "기사",
        "review_flag": df["통합재판정"],            # 유지(잠정) / 원문확인 / 제외후보
        "url": "",
        "원본시군": kws.map(lambda L: ",".join(L)),
        "원본관련시군": "",
        "수집층": "뉴스(단일 수집)",
        "검토등급": "", "검토유형": "",
    })
    return out


def standardize_instagram(df: pd.DataFrame) -> pd.DataFrame:
    """인스타 시트를 공통 형식으로 바꿉니다.

    좋아요를 숨긴 게시물('좋아요숨김'=예)은 좋아요 칸에 자리표시값(대부분 3)만 들어 있으므로 결측으로 처리합니다.
    조회수는 릴스에만 제공되며, 사진·캐러셀의 조회수는 0이 아니라 '미제공(결측)'입니다.
    """
    likes = to_num(df["좋아요"]).where(df["좋아요숨김"] != "예")
    out = pd.DataFrame({
        "media": "인스타",
        "content_id": df["게시물ID"],
        "date": pd.to_datetime(df["작성일(KST)"], errors="coerce"),
        "title": "",
        "body": df["본문"].fillna(""),
        "tags": df["해시태그"].fillna(""),
        "timeline": "",
        "location": df["위치"].fillna(""),
        "source": df["작성자(해시)"],
        "sigun_all": [split_sigun(a, b) for a, b in zip(df["시군"], df["관련시군"])],
        "type": df["미디어유형"],                    # 사진 / 캐러셀 / 릴스
        "likes": likes,
        "likes_hidden": df["좋아요숨김"] == "예",
        "comments": to_num(df["댓글수"]),
        "views": to_num(df["조회수(릴스)"]),
        "collect_tag": df["수집태그"].fillna(""),
        "url": df["링크"],
        "원본ID": "IG:" + df["게시물ID"],
        "원본시군": df["시군"].fillna(""),
        "원본관련시군": df["관련시군"].fillna(""),
        "검토등급": df["검토등급"].fillna("") if "검토등급" in df else "",
        "검토유형": df["검토유형"].fillna("") if "검토유형" in df else "",
    })
    return out


def standardize_youtube(df: pd.DataFrame, timeline: pd.DataFrame) -> pd.DataFrame:
    """유튜브 시트를 공통 형식으로 바꾸고, 타임라인 텍스트를 영상ID로 붙입니다.

    타임라인 행은 별도 영상으로 세지 않습니다(의뢰서 2.2). 같은 영상의 타임라인 항목을 ' | '로 이어
    'timeline' 열 하나로 만들고, 제목에 드러나지 않는 방문지·음식 등을 보완하는 데만 씁니다.
    """
    tl = timeline.copy()
    tl["텍스트"] = tl["장소명·텍스트"].fillna("").map(clean_timeline_item)
    tl = tl[tl["텍스트"] != ""]
    tl_join = tl.groupby("영상ID")["텍스트"].apply(lambda s: " | ".join(dict.fromkeys(s)))
    out = pd.DataFrame({
        "media": "유튜브",
        "content_id": df["영상ID"],
        "date": pd.to_datetime(df["게시일(KST)"], errors="coerce"),
        "title": df["제목"].fillna(""),
        "body": df["설명"].fillna(""),
        "tags": df["태그"].fillna(""),
        "timeline": df["영상ID"].map(tl_join).fillna(""),
        "location": "",
        "source": df["채널명"].fillna("").str.strip(),
        "sigun_all": [split_sigun(a, b) for a, b in zip(df["시군"], df["관련시군"])],
        # 형식: 일반 / 숏츠(180초 이하) / 라이브(라이브=예이면 길이와 무관하게 라이브로 재판정, 근거는 '형식근거' 열)
        "type": np.where(df["라이브"].fillna("") == "예", "라이브", df["구분"]),
        "형식근거": np.where(df["라이브"].fillna("") == "예", "라이브=예 → 라이브(원 구분: " + df["구분"].fillna("") + ")", "원본 구분"),
        "length_sec": to_num(df["길이(초)"]),
        "views": to_num(df["조회수"]),
        "likes": to_num(df["좋아요"]),
        "comments": to_num(df["댓글수"]),
        "collect_mode": df["수집방식"],              # 기본 / 추가(시범 수집분)
        "search_query": df["검색어"].fillna(""),
        "url": df["링크"],
        "원본ID": "YT:" + df["영상ID"],
        "원본시군": df["시군"].fillna(""),
        "원본관련시군": df["관련시군"].fillna(""),
        "검토등급": df["검토등급"].fillna("") if "검토등급" in df else "",
        "검토유형": df["검토유형"].fillna("") if "검토유형" in df else "",
    })
    return out


_TL_GENERIC = re.compile(
    r"^(인트로|intro|outro|아웃트로|엔딩|ending|오프닝|opening|시작|마무리|끝|하이라이트|예고|"
    r"day\s?\d+|\d+\s?일차|첫째\s?날|둘째\s?날|셋째\s?날|넷째\s?날|\d+부|part\s?\d+|구독|광고|"
    r"클로징|closing|outro|정리|소개|요약)$", re.I)


def clean_timeline_item(s: str) -> str:
    """타임라인 항목 하나를 정리합니다. '인트로', 'Day1'처럼 대상이 아닌 구간 이름은 빈 문자열로 만듭니다."""
    s = re.sub(r"[^\w\s가-힣&·\-]", " ", str(s))
    s = re.sub(r"\s+", " ", s).strip()
    if not s or _TL_GENERIC.match(s):
        return ""
    return s


news = standardize_news(news_raw)
ig = standardize_instagram(sns_raw["인스타"])
yt = standardize_youtube(sns_raw["유튜브"], sns_raw["유튜브_타임라인"])
SUPP_END = pd.Timestamp(CONFIG["SUPP_SEGMENT_END"]) + pd.Timedelta(days=1)
for d in (news, ig, yt):
    d["year"] = d["date"].dt.year
    d["month"] = d["date"].dt.to_period("M").astype(str)
    if d["media"].iat[0] != "뉴스":
        # 수집구간: 원자료 '수집기준' 시트의 두 구간(작성일 기준). 2023.1.1~10.30은 2026-11-05에 따로 받은 보완 수집분
        d["수집구간"] = np.where(d["date"] < SUPP_END, "2023.1.1~10.30(보완)", "2023.10.31~2026.10.30(기본)")
    else:
        d["수집구간"] = "뉴스(단일)"
# 수집층: 보완/기본 구간 × (유튜브) 기본/추가(시범) 수집. 유튜브 '추가'는 시범 수집분이며 2023 보완 수집과 다른 개념
ig["수집층"] = "IG_" + np.where(ig["수집구간"].str.contains("보완"), "보완(인기·릴스탭 중심)", "기본(최근 게시물)")
yt["수집층"] = "YT_" + np.where(yt["수집구간"].str.contains("보완"), "보완(조회수순)", "기본(검색기본순)") + "_" + yt["collect_mode"].fillna("")
print("뉴스", news.shape, "| 인스타", ig.shape, "| 유튜브", yt.shape)
print("타임라인이 붙은 유튜브 영상:", (yt["timeline"] != "").sum())

## 3. 데이터 현황·품질 점검 (산출물 ①)

결측, 중복, 기간, 매체·연도·시군 분포를 점검하고, **수집량 변화가 실제 관심 변화인지 수집 방식 때문인지** 판단할 근거를 정리합니다.

In [ ]:
def basic_quality(df: pd.DataFrame, id_col="content_id", check_cols=()) -> pd.DataFrame:
    """매체 하나의 기본 품질표(행 수, 고유 ID 수, 중복 ID, 기간, 주요 열의 결측 수·결측률)를 만듭니다."""
    rows = [
        ("원자료 행 수", len(df), ""),
        ("고유 ID 수", df[id_col].nunique(), ""),
        ("중복 ID 행", int(df[id_col].duplicated().sum()), ""),
        ("날짜 변환 실패", int(df["date"].isna().sum()), ""),
        ("최초 일자", str(df["date"].min().date()), ""),
        ("최종 일자", str(df["date"].max().date()), ""),
    ]
    for c in check_cols:
        if c not in df:
            continue
        s = df[c]
        miss = s.isna() | (s.astype(str).str.strip() == "")
        rows.append((f"결측: {c}", int(miss.sum()), f"{miss.mean():.1%}"))
    return pd.DataFrame(rows, columns=["항목", "값", "비율"])


QC = {
    "뉴스": basic_quality(news, check_cols=["title", "body", "source"]),
    "인스타": basic_quality(ig, check_cols=["body", "tags", "location", "likes", "comments", "views"]),
    "유튜브": basic_quality(yt, check_cols=["title", "body", "tags", "likes", "comments", "views", "timeline"]),
}
qc_table = pd.concat({k: v.set_index("항목") for k, v in QC.items()}, axis=1)
qc_table

In [ ]:
def year_distribution() -> pd.DataFrame:
    """매체×연도 수록 건수(원자료 기준). 2026년은 부분 연도(뉴스 10/26, SNS 10/30까지)입니다."""
    t = pd.concat([d[["media", "year"]] for d in (news, ig, yt)])
    out = t.pivot_table(index="year", columns="media", aggfunc="size", fill_value=0)[MEDIA]
    out.index.name = "연도"
    return out


def sigun_distribution() -> pd.DataFrame:
    """매체×시군 건수. 여러 시군에 걸친 콘텐츠는 *각 시군에 1번씩* 세므로 합계가 전체 건수보다 클 수 있습니다."""
    parts = []
    for d in (news, ig, yt):
        e = d[["media", "sigun_all"]].explode("sigun_all")
        parts.append(e)
    t = pd.concat(parts)
    out = t.pivot_table(index="sigun_all", columns="media", aggfunc="size", fill_value=0)[MEDIA]
    out.index.name = "시군(복수 허용)"
    return out.sort_values("뉴스", ascending=False)


def monthly_volume() -> pd.DataFrame:
    """매체별 월간 수록 건수. 수집량이 급격히 변한 달을 찾는 데 씁니다."""
    t = pd.concat([d[["media", "month"]] for d in (news, ig, yt)])
    return t.pivot_table(index="month", columns="media", aggfunc="size", fill_value=0)[MEDIA]


YEAR_DIST, SIGUN_DIST, MONTH_VOL = year_distribution(), sigun_distribution(), monthly_volume()
display(YEAR_DIST)
display(SIGUN_DIST)

ax = MONTH_VOL.loc["2023-01":].plot(figsize=(12, 4), logy=True, title="매체별 월간 수록 건수(로그 축)")
ax.set_xlabel(""); plt.tight_layout(); plt.show()

In [ ]:
def instagram_collection_bias(tag_range: pd.DataFrame) -> pd.DataFrame:
    """'인스타태그범위' 시트에서 수집이 *쪽수 상한으로 잘린* 태그를 수집구간별로 찾습니다.

    SNS 원자료는 두 구간을 따로 수집했습니다.
    - 2023.1.1~10.30(보완 수집): 해시태그의 '인기 게시물(릴스 탭)'만 받았고 '최근 게시물'은 받지 않았습니다.
    - 2023.10.31~2026.10.30: '최근 게시물'을 쪽수 상한까지 받았습니다.
    '가장 이른 작성일'이 그 구간 시작일보다 30일 넘게 늦고 쪽수 상한으로 표시된 태그는 오래된 게시물을
    끝까지 받지 못한 것입니다. 이런 태그가 많을수록 최근 월에 게시물이 몰리므로, 건수 대신 비중으로 비교합니다.
    """
    t = tag_range.dropna(subset=["수집구간"]).copy()
    t["구간시작"] = pd.to_datetime(t["수집구간"].str.extract(r"^(\d{4})\.(\d{1,2})\.(\d{1,2})")
                                  .apply(lambda r: f"{r[0]}-{r[1]}-{r[2]}", axis=1), errors="coerce")
    t["가장 이른 작성일"] = pd.to_datetime(t["가장 이른 작성일"], errors="coerce")
    t["남은 게시물"] = to_num(t["남은 게시물"])
    t["상한으로_잘림"] = (t["최근 게시물(쪽수·받은 범위)"].fillna("") + t["인기 게시물(쪽수·받은 범위)"].fillna("")).str.contains("상한")
    t["초기기간_누락"] = t["가장 이른 작성일"] > t["구간시작"] + pd.Timedelta(days=30)
    t["판정"] = np.where(t["상한으로_잘림"] & t["초기기간_누락"], "과거 게시물 일부 미수집(최근 편중)", "")
    return t.drop(columns="구간시작").sort_values(["수집구간", "남은 게시물"], ascending=[True, False])


IG_TAG_BIAS = instagram_collection_bias(sns_raw["인스타태그범위"])
n_bias = (IG_TAG_BIAS["판정"] != "").sum()
share_bias = IG_TAG_BIAS.loc[IG_TAG_BIAS["판정"] != "", "남은 게시물"].sum() / IG_TAG_BIAS["남은 게시물"].sum()
print(f"과거 게시물이 잘린 태그: {n_bias}개 / 전체 {len(IG_TAG_BIAS)}개, 해당 태그 게시물 비중(중복 포함) {share_bias:.1%}")
display(IG_TAG_BIAS.groupby("수집구간").agg(태그수=("태그", "size"), 남은게시물=("남은 게시물", "sum"),
                                         잘린태그=("판정", lambda s: (s != "").sum())))
IG_TAG_BIAS[IG_TAG_BIAS["판정"] != ""].head(15)[["수집구간", "시군", "태그", "남은 게시물", "가장 이른 작성일", "최근 게시물(쪽수·받은 범위)"]]

In [ ]:
def youtube_collection_note(df: pd.DataFrame) -> pd.DataFrame:
    """유튜브 수집방식(기본/추가)×연도 분포.

    기본 수집은 '검색어×연도마다 관련성 순 상위 50개'라서 연도별 영상 수에 *구조적인 상한*이 있습니다.
    '추가'(시범 수집분)는 2023~2025년에만 있으므로 연도 비교에서는 기본 수집분만 씁니다(CONFIG['YT_TREND_BASIS_ONLY']).
    """
    return pd.crosstab(df["collect_mode"], df["year"], margins=True, margins_name="합계")


def news_date_note() -> pd.DataFrame:
    """뉴스 날짜 열의 성격과 한계를 기록합니다(의뢰서 2.1)."""
    return pd.DataFrame([
        ("날짜 열", "검색표시날짜"),
        ("형식", "YYYY-MM-DD 단일 형식, 변환 실패 0건" if news["date"].notna().all() else "변환 실패 있음"),
        ("실제 발행일 여부", "원자료만으로는 확인 불가 → 대리 날짜로 사용(포털 검색 결과 표시일)"),
        ("분석 텍스트", "기사제목+미리보기(본문 전체 아님)"),
        ("언론사 수", f"{news['source'].nunique()}개: " + ", ".join(news['source'].value_counts().index)),
        ("통합재판정", ", ".join(f"{k} {v:,}" for k, v in news["review_flag"].value_counts().items())),
    ], columns=["항목", "내용"])


display(youtube_collection_note(yt))
display(news_date_note())

**점검 결과 해석 메모** *(실행 후 수치를 확인해 보고서에 옮기세요)*
- 인스타 2026년 수록량(특히 9~10월)은 해시태그별 '최근 글' 쪽수 상한 때문에 최근 게시물이 많이 들어온 것입니다. **건수가 아니라 기간별 등장 비중**으로 비교하고, 9~10월을 뺀 1~8월 구간으로 민감도를 점검합니다(9단계).
- **2023.1.1~10.30 보완 수집분은 수집 방식이 다릅니다.** 인스타는 해시태그 '인기 게시물(릴스 탭)'만, 유튜브는 검색어별 *조회수 순* 상위 50개를 받았습니다(이후 구간은 최근 게시물·관련성 순). 그래서 RISING은 2023~2025년을 합친 과거 구간과 비교하되, 2023년을 뺀 2024~2025년 기준으로도 조건이 유지되는지 `민감도_유지`로 표시합니다.
- 유튜브는 검색어×연도별 상위 50개 구조여서 연도별 영상 수 자체는 관심 규모를 뜻하지 않습니다. 이 역시 비중으로 비교합니다.
- 뉴스는 6개 지역 언론사에 한정되고, 제목·미리보기만으로 분석했습니다.

## 4. 전처리

### 4.1 뉴스 검색어 띄어쓰기 복원
원자료에서는 포털 검색 결과의 검색어 강조 흔적으로 **검색어 바로 뒤에 공백이 끼어 있습니다**(예: `전주 시는`, `익산 시의회`, `전북 자치도`, `전주 의`). 그대로 두면 '전주시', '익산시의회' 같은 고유명칭이 깨지므로, 그 행의 `검색어목록`에 있는 검색어 뒤의 공백 1칸만 지웁니다.

> 원래 띄어 썼던 표현(예: `전주 한옥마을`)도 붙게 되지만(`전주한옥마을`), 표현 통합 단계에서 같은 대상으로 묶이므로 집계에는 영향이 없습니다.

In [ ]:
def restore_keyword_spacing(text: str, keywords: list) -> str:
    """검색어 뒤에 잘못 들어간 공백을 지웁니다.

    Parameters
    ----------
    text : 기사제목 또는 미리보기
    keywords : 그 기사의 검색어목록(예: ["전북", "전주"])

    긴 검색어부터 처리해 '전북특별자치도'가 '전북'보다 먼저 복원되도록 합니다.
    한글·영문·숫자나 문장부호가 뒤따를 때만 공백을 지웁니다.
    """
    if not text or not keywords:
        return text
    for kw in sorted(set(keywords), key=len, reverse=True):
        text = re.sub(re.escape(kw) + r" (?=[가-힣A-Za-z0-9,.·…!?)\]’”'\"])", kw, text)
    return text


# 예시 확인
for t, k in [("익산 시의회 손진영 의원, 익산 시 수목 관리", ["익산"]), ("전북 , 소비쿠폰 찾아가는 신청", ["전북"])]:
    print(t, "→", restore_keyword_spacing(t, k))

news["title"] = [restore_keyword_spacing(t, k) for t, k in zip(news["title"], news["search_keywords"])]
news["body"] = [restore_keyword_spacing(t, k) for t, k in zip(news["body"], news["search_keywords"])]

### 4.2 텍스트 정제

In [ ]:
_URL = re.compile(r"https?://\S+|www\.\S+")
_MASK = re.compile(r"@\*+|\*{3,}")                         # 비식별 처리된 @아이디·이메일
_EMAIL = re.compile(r"\S+@\S+\.\S+")
_HASHTAG = re.compile(r"#[^\s#]+")
_TIMESTAMP = re.compile(r"\b\d{1,2}:\d{2}(:\d{2})?\b")
_NEWS_BOILER = [
    re.compile(r"[가-힣]{2,4}\s?기자"),                    # 홍길동 기자
    re.compile(r"/\s?[가-힣]{2,4}\s?=?\s?$"),               # /고창=
    re.compile(r"사진\s?=\s?\S+"),                          # 사진=전주시
    re.compile(r"\[[^\]]{1,20}\]"),                         # [전북일보], [앵커]
    re.compile(r"\([^)]{0,15}제공\)"),                      # (전주시 제공)
    re.compile(r"\.{2,}|…"),
]


def clean_common(text: str) -> str:
    """매체 공통 정제: 유니코드 정규화(NFKC), URL·이메일·비식별 기호·시각 표기 제거, 공백 정리."""
    if not isinstance(text, str) or not text:
        return ""
    t = unicodedata.normalize("NFKC", text)
    t = _URL.sub(" ", t)
    t = _EMAIL.sub(" ", t)
    t = _MASK.sub(" ", t)
    t = _TIMESTAMP.sub(" ", t)
    t = re.sub(r"[​-‏﻿]", "", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()


_STAFF_NAME = re.compile(
    r"(취재\s?기자|기자|아나운서|앵커|리포터|촬영|편집|취재|영상취재|그래픽|pd|PD)\s*[:：]?\s*[가-힣]{2,4}(\s?[,·/]\s?[가-힣]{2,4})*"
    r"|[가-힣]{2,4}\s?(기자|아나운서|앵커|리포터|PD)"
    r"|(뉴스|news|NEWS)\s*[가-힣]{2,4}\s?입니다")


def clean_youtube_text(text: str) -> str:
    """유튜브 제목·설명 정제: 공통 정제에 더해 '정다윤 기자', '촬영: 홍길동'처럼 *제작진 이름*을 지웁니다.

    방송사 뉴스 영상 설명란에는 기자·아나운서 이름이 반복되어 관심대상 순위에 들어오기 때문입니다.
    """
    return re.sub(r"\s+", " ", _STAFF_NAME.sub(" ", clean_common(text))).strip()


def clean_news_text(text: str) -> str:
    """뉴스 정제: 공통 정제에 더해 기자명, 사진 출처, 언론사 머리표 같은 상투 문구를 지웁니다."""
    t = clean_common(text)
    for p in _NEWS_BOILER:
        t = p.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()


_BOILER_LINE = re.compile(
    r"(주소|위치\s*[:：]|영업\s*시간|운영\s*시간|라스트\s*오더|브레이크\s*타임|휴무|정기휴일|전화|문의|예약|주차|"
    r"카카오톡|카톡|이메일|e-?mail|홈페이지|페이스북|인스타그램|블로그|네이버|구독|좋아요|알림\s*설정|"
    r"ⓒ|©|copyright|all\s*rights|무단\s*전재|재배포|ai\s*학습|제보|협찬\s*문의|광고\s*문의|비즈니스\s*문의|"
    r"(취재|기자|촬영|편집|그래픽|앵커|영상)\s*[:：/]|영상\s*제공|화면\s*제공|신청\s*방법|신청\s*기간|모집\s*기간|모집\s*대상|참가비|선착순|당첨자|qr|music\s*provided|bgm)", re.I)


def drop_boiler_lines(text: str) -> str:
    """SNS 본문·유튜브 설명란에서 연락처·주소·영업시간·저작권·구독 요청·이벤트 신청 안내 같은 *안내 문구 줄*을 지웁니다.

    이런 줄은 관심대상이 아니라 게시 형식에서 반복되는 문구라서 빈도 상위를 차지합니다('홈페이지', '1층', '무단전재' 등).
    줄바꿈 단위로 판단하므로 정제(공백 합치기) *전에* 적용합니다.
    """
    if not isinstance(text, str) or not text:
        return ""
    return "\n".join(l for l in text.splitlines() if not _BOILER_LINE.search(l) and not is_stuffed_line(l))


def is_stuffed_line(line: str, min_tokens: int = 8, min_share: float = 0.6) -> bool:
    """키워드 도배 줄인지 판정합니다. 예: '익산석재.김제석재.부안석재.고창석재.…'

    구분자(. , · / | 공백)로 나눈 조각이 min_tokens개 이상이고, 조각 끝 2글자가 같은 것이 min_share 이상이면
    검색 노출용으로 지명을 나열한 줄로 보고 지웁니다.
    """
    parts = [p for p in re.split(r"[.,·/|\s]+", line) if len(p) >= 3]
    if len(parts) < min_tokens:
        return False
    top = Counter(p[-2:] for p in parts).most_common(1)[0][1]
    return top / len(parts) >= min_share


def extract_hashtags(text: str) -> list:
    """문자열에서 해시태그를 뽑아 '#'과 끝 문장부호를 떼고 반환합니다. 같은 태그는 한 번만 남깁니다."""
    if not isinstance(text, str):
        return []
    tags = [re.sub(r"[^\w가-힣]+$", "", h[1:]) for h in _HASHTAG.findall(unicodedata.normalize("NFKC", text))]
    return list(dict.fromkeys(t for t in tags if t))


def split_yt_tags(text: str) -> list:
    """유튜브 '태그' 칸(쉼표 구분)을 목록으로 만듭니다.

    방송사 채널은 태그 끝에 '아나운서, 홍길동, 김철수'처럼 진행자 이름을 붙이므로, '아나운서·앵커·기자' 태그
    *뒤에 오는* 2~4글자 한글 태그는 인명으로 보고 뺍니다. 태그 칸에 섞인 URL도 지웁니다.
    """
    if not isinstance(text, str) or not text.strip():
        return []
    out, after_staff = [], False
    for t in unicodedata.normalize("NFKC", text).split(","):
        t = _URL.sub("", t).strip()
        if not t:
            continue
        if t in ("아나운서", "앵커", "기자", "리포터"):
            after_staff = True
            continue
        if after_staff and re.fullmatch(r"[가-힣]{2,4}", t):
            continue
        after_staff = False
        out.append(t)
    return list(dict.fromkeys(out))


def collect_staff_names(texts) -> set:
    """유튜브 설명란에서 기자·제작진·영상제공자 이름을 모읍니다(태그·본문에서 이 이름을 빼는 데 씀).

    잡는 꼴: '임홍진 기자', 'MBC뉴스 조수영입니다', '영상제공: 남원시(강석현), 완주군(김회성)', '촬영: 홍길동'.
    인명처럼 보이는 3글자(성씨로 시작)이고 2번 이상 나온 이름만 남겨 '사진기자' 같은 일반어가 섞이지 않게 합니다.
    """
    pats = [re.compile(r"([가-힣]{3})\s?(기자|아나운서|앵커|리포터|PD)"),
            re.compile(r"(?:뉴스|news)\s*([가-힣]{3})\s?입니다", re.I),
            re.compile(r"(?:취재|촬영|편집|영상취재|그래픽)\s*[:：]?\s*([가-힣]{3})")]
    credit = re.compile(r"(?:영상|화면)\s*제공\s*[:：]?([^\n]*)")
    names = Counter()
    for t in texts:
        if not isinstance(t, str):
            continue
        for p in pats:
            names.update(m.group(1) for m in p.finditer(t))
        for c in credit.finditer(t):
            names.update(re.findall(r"\(([가-힣]{3})\)", c.group(1)))
    return {n for n, c in names.items() if c >= 2 and n[0] in _SURNAMES_STAFF}


_SURNAMES_STAFF = set("김이박최정강조윤장임한오서신권황안송류유홍전고문양손배백허남심노하곽성차주우구민진나지엄채원천방공현함변염여추도소석선설마길연위표명기반왕금옥육인맹제모탁국어은편용예경봉")
YT_STAFF = collect_staff_names(yt["body"])
print(f"유튜브 제작진·영상제공자 이름 {len(YT_STAFF)}개 수집(태그·대상에서 제외):", ", ".join(sorted(YT_STAFF)[:15]), "…")

# 뉴스: 제목 + 미리보기
news["text"] = (news["title"].map(clean_news_text) + " . " + news["body"].map(clean_news_text)).str.strip(" .")
news["tag_list"] = [[] for _ in range(len(news))]

# 인스타: 본문에서 해시태그를 떼어 내 따로 보관(본문 형태소 분석과 태그 분석이 겹치지 않도록)
ig["tag_list"] = [list(dict.fromkeys(extract_hashtags(a) + extract_hashtags(b))) for a, b in zip(ig["tags"], ig["body"])]
ig["text"] = (ig["body"].map(drop_boiler_lines).map(lambda s: _HASHTAG.sub(" ", s)).map(clean_common) + " . " + ig["location"].map(clean_common)).str.strip(" .")

# 유튜브: 제목 + 설명(앞부분) + 타임라인, 태그는 따로
yt["tag_list"] = [[t for t in dict.fromkeys(split_yt_tags(a) + extract_hashtags(b) + extract_hashtags(c)) if t not in YT_STAFF]
                  for a, b, c in zip(yt["tags"], yt["title"], yt["body"])]
yt["text"] = (yt["title"].map(lambda s: _HASHTAG.sub(" ", s)).map(clean_youtube_text) + " . "
              + yt["body"].map(drop_boiler_lines).str[:CONFIG["YT_DESC_MAXLEN"]].map(lambda s: _HASHTAG.sub(" ", s)).map(clean_youtube_text) + " . "
              + yt["timeline"].map(clean_common)).str.strip(" .")

print(news[["title", "text"]].head(3).to_string()[:600])

In [ ]:
def remove_channel_boilerplate_tags(df: pd.DataFrame, share: float, min_videos: int = 10,
                                    share_big: float = 0.25, big_videos: int = 30):
    """유튜브 채널이 모든 영상에 반복해서 붙이는 상투 태그(예: '전주MBC, 뉴스데스크, 속보')를 지웁니다.

    영상이 min_videos개 이상인 채널에서, 그 채널 영상의 share 비율(영상이 big_videos개 이상이면 share_big) 이상에
    붙은 태그를 '채널 상투 태그'로 봅니다. 방송사 채널은 아나운서 이름까지 포함한 SEO 태그 묶음을 일부 영상에만
    붙이는 경우가 있어 큰 채널에는 더 낮은 기준을 씁니다.
    이런 태그는 관심대상이 아니라 채널 표식이므로 빈도에서 빼고, 제거 목록을 결과로 남깁니다.
    """
    removed, bp_by_channel = [], {}
    for ch, g in df.groupby("source"):
        if len(g) < min_videos:
            continue
        cnt = Counter(t for tl in g["tag_list"] for t in set(tl))
        thr = share_big if len(g) >= big_videos else share
        bp = {t for t, c in cnt.items() if c / len(g) >= thr}
        if bp:
            bp_by_channel[ch] = bp
            removed += [(ch, len(g), t, cnt[t]) for t in sorted(bp)]
    new_lists = [[t for t in tl if t not in bp_by_channel.get(ch, ())] for ch, tl in zip(df["source"], df["tag_list"])]
    return new_lists, pd.DataFrame(removed, columns=["채널명", "채널 영상 수", "제거 태그", "붙은 영상 수"])


yt["tag_list"], YT_BOILER = remove_channel_boilerplate_tags(yt, CONFIG["YT_BOILERPLATE_TAG_SHARE"],
                                                             share_big=CONFIG["YT_BOILERPLATE_TAG_SHARE_BIG"])
print(f"채널 상투 태그 {len(YT_BOILER)}건 제거 ({YT_BOILER['채널명'].nunique()}개 채널)")
YT_BOILER.head(10)

### 4.3 뉴스 중복·전재 처리
- **완전 중복:** 같은 언론사, 같은 날짜, 같은 제목(점검 결과 0건)
- **기사군:** 기호·공백을 지운 제목이 같고 `NEWS_DUP_WINDOW_DAYS`일 안에 실린 기사는 같은 사건을 다룬 **동일 기사·전재**로 보고 한 기사군으로 묶습니다. 뉴스 빈도는 **기사군 수**로 세고, 언론사 다양성은 기사군에 속한 모든 언론사로 계산합니다.
- 제목이 조금씩 다른 유사 기사는 묶지 않았습니다(한계로 기록).

In [ ]:
def mark_news_duplicates(df: pd.DataFrame, window_days: int) -> pd.DataFrame:
    """정규화한 제목과 날짜 간격으로 뉴스 기사군(dup_group)을 붙입니다.

    처리 순서: (1) 제목에서 한글·영문·숫자만 남겨 norm_title을 만들고, (2) norm_title별로 날짜순 정렬한 뒤,
    (3) 앞 기사와의 간격이 window_days를 넘으면 새 기사군을 시작합니다.
    제목이 8자 미만이면('[포토]', '인사' 등) 잘못 묶일 수 있어 기사마다 따로 기사군을 줍니다.

    Returns
    -------
    dup_group, dup_size(기사군 크기), dup_press_n(기사군 언론사 수) 열이 추가된 DataFrame
    """
    d = df.copy()
    d["norm_title"] = d["title"].str.replace(r"[^0-9A-Za-z가-힣]", "", regex=True)
    d = d.sort_values(["norm_title", "date"])
    new_grp = (d["norm_title"] != d["norm_title"].shift()) | (d["date"].diff().dt.days > window_days) \
              | (d["norm_title"].str.len() < 8)
    d["dup_group"] = "G" + new_grp.cumsum().astype(str)
    g = d.groupby("dup_group")
    d["dup_size"] = g["content_id"].transform("size")
    d["dup_press_n"] = g["source"].transform("nunique")
    return d.sort_index()


news = mark_news_duplicates(news, CONFIG["NEWS_DUP_WINDOW_DAYS"])
NEWS_DUP_SUMMARY = pd.DataFrame([
    ("원자료 기사 수", len(news)),
    ("완전 중복(언론사·날짜·제목 동일)", int(news.duplicated(["source", "date", "title"]).sum())),
    ("기사군 수(빈도 산정 기준)", news["dup_group"].nunique()),
    ("2건 이상 묶인 기사군", int((news.groupby("dup_group").size() > 1).sum())),
    ("묶여서 줄어든 기사 수", len(news) - news["dup_group"].nunique()),
    ("여러 언론사가 같은 제목으로 실은 기사군", int((news.groupby("dup_group")["source"].nunique() > 1).sum())),
], columns=["항목", "건수"])
NEWS_DUP_SUMMARY

### 4.4 SNS 지역 관련성 점검
수집 태그·검색어가 붙었다는 사실만으로 전북 관련 콘텐츠로 확정하지 않습니다(의뢰서 4.3). 콘텐츠마다 지역 단서의 수준을 표시합니다.

| 수준 | 기준 |
|---|---|
| A | 본문·위치(유튜브는 제목·설명·타임라인)에 시군명, 전북, 대표 지명 등 지역 단서가 있음 |
| B | 본문에는 없지만 수집 태그 *이외의* 해시태그·태그에 지역 단서가 있음 |
| C | 수집 태그 말고는 지역 단서가 없음 |

`CONFIG["SNS_RELEVANCE"]`: `strict`=A, `standard`=A+B(기본값), `loose`=A+B+C

In [ ]:
def build_region_cue_pattern() -> re.Pattern:
    """지역 단서 정규식: 시군명, 광역명, 수집 태그로 쓰인 대표 지명·축제·특산물, 영문 지명."""
    words = list(SIGUN_BASE.keys()) + ["전북", "전라북도", "새만금", "jeonbuk", "jeollabuk"]
    words += ["고군산", "선유도", "미륵사", "왕궁리", "내장산", "광한루", "지평선", "삼례", "마이산", "덕유산",
              "반딧불", "임실치즈", "순창고추장", "강천산", "선운사", "청보리", "변산", "채석강", "내소사", "격포",
              "곰소", "한옥마을", "원광대", "군산대", "전북대", "장안산", "모악산", "대둔산", "운일암", "구천동"]
    words += ["jeonju", "gunsan", "iksan", "jeongeup", "namwon", "gimje", "wanju", "jinan", "muju",
              "jangsu", "imsil", "sunchang", "gochang", "buan"]
    return re.compile("|".join(sorted(map(re.escape, words), key=len, reverse=True)), re.I)


REGION_CUE = build_region_cue_pattern()


def relevance_level(text: str, tags: list, collect_tags: str = "") -> str:
    """콘텐츠 하나의 지역 단서 수준(A/B/C)을 판정합니다. 기준은 위 표와 같습니다."""
    if REGION_CUE.search(text or ""):
        return "A"
    ct = {t.strip().lstrip("#") for t in re.split(r"[,\s]+", collect_tags or "") if t.strip()}
    if any(REGION_CUE.search(t) for t in tags if t not in ct):
        return "B"
    return "C"


ig["relevance"] = [relevance_level(t, tl, ct) for t, tl, ct in zip(ig["text"], ig["tag_list"], ig["collect_tag"])]
yt["relevance"] = [relevance_level(t, tl) for t, tl in zip(yt["text"], yt["tag_list"])]
news["relevance"] = "A"   # 뉴스는 원자료 단계에서 관련성 판정(최종판정=유지)을 마쳤으므로 그대로 사용

REL_TABLE = pd.concat({
    "인스타": ig["relevance"].value_counts(), "유튜브": yt["relevance"].value_counts()
}, axis=1).fillna(0).astype(int).sort_index()
display(REL_TABLE)
print("C 수준 인스타 예시:")
for s in ig.loc[ig["relevance"] == "C", "body"].head(3):
    print(" -", s[:120].replace("\n", " "))

In [ ]:
def apply_validity(df: pd.DataFrame) -> pd.Series:
    """CONFIG 기준에 따라 분석에 포함할 콘텐츠(유효 콘텐츠) 여부를 True/False로 돌려줍니다.

    - 공통: 날짜가 있고 공통 관측기간 이전 자료도 포함(뉴스 장기 분석용). 기간 제한은 집계 단계에서 적용
    - SNS: 지역 관련성 수준 필터
    - 뉴스: NEWS_EXCLUDE_REVIEW=True면 '원문확인' 판정 제외
    """
    ok = df["date"].notna()
    mode = CONFIG["SNS_RELEVANCE"]
    if df["media"].iat[0] in ("인스타", "유튜브"):
        allowed = {"strict": {"A"}, "standard": {"A", "B"}, "loose": {"A", "B", "C"}}[mode]
        ok &= df["relevance"].isin(allowed)
    if df["media"].iat[0] == "뉴스" and CONFIG["NEWS_EXCLUDE_REVIEW"]:
        ok &= df["review_flag"] != "원문확인"
    return ok


def mark_sns_text_duplicates(df: pd.DataFrame, min_len: int = 30) -> pd.Series:
    """SNS에서 본문이 사실상 같은 게시물(같은 홍보문·이벤트 공지를 여러 번 올린 경우)을 하나의 묶음으로 표시합니다.

    해시태그·기호·공백을 지운 본문 앞 200자가 같으면 같은 묶음입니다. 본문이 min_len자 미만이면
    잘못 묶일 수 있어 게시물마다 따로 둡니다. 반환값은 빈도 산정 단위(count_key)로 씁니다.
    """
    norm = df["body"].map(lambda s: _HASHTAG.sub("", s or "")).str.replace(r"[^0-9A-Za-z가-힣]", "", regex=True).str[:200]
    key = np.where(norm.str.len() >= min_len, "T" + norm, "C" + df["content_id"])
    return pd.Series(pd.factorize(key)[0], index=df.index).map(lambda i: f"{df['media'].iat[0][0]}{i}")


for d in (news, ig, yt):
    d["valid"] = apply_validity(d)
    # 기본 빈도 단위(과업지시서 12장): 고유 콘텐츠(content_id). 반복군(rep_key)은 보조 지표로 함께 집계
    #   뉴스 반복군 = 기사군(같은 제목, 3일 이내), 인스타 = 동일 본문 묶음, 유튜브 = 영상(타임라인 행은 영상 1건)
    d["count_key"] = d["content_id"]
    if d["media"].iat[0] == "뉴스":
        d["rep_key"] = d["dup_group"]
    elif d["media"].iat[0] == "인스타":
        d["rep_key"] = mark_sns_text_duplicates(d)
    else:
        d["rep_key"] = d["content_id"]
print("인스타 동일 본문 반복군으로 묶이는 게시물:", len(ig) - ig["rep_key"].nunique())

VALID_SUMMARY = pd.DataFrame({
    m: {"원자료": len(d), "유효 콘텐츠": int(d["valid"].sum()), "반복군 수": d.loc[d["valid"], "rep_key"].nunique()}
    for m, d in zip(MEDIA, (news, ig, yt))
}).T
VALID_SUMMARY

## 5. 관심대상 후보 추출

**방법.** Kiwi 형태소 분석기로 명사를 찾은 뒤 다음 세 종류의 의미 단위를 만듭니다.

1. **어절 내 명사 연쇄**: 띄어 쓰지 않은 명사 묶음을 한 단위로 봅니다. 예: `군산시간여행축제`, `전주비빔밥`, `임실치즈테마파크`
2. **어절을 잇는 복합명사**: 조사 없이 이어진 명사 어절(예: `새만금 국제공항`)은 *자주 함께 나오고 결합이 강할 때만* 한 단위로 잇습니다(`NGRAM_MIN_DF`, `NGRAM_ASSOC`). 명사를 기계적으로 이어 붙이는 것(예: `예산 확보 추진`)을 막기 위해서입니다.
3. **해시태그·유튜브 태그**: 작성자가 직접 붙인 단위를 그대로 씁니다.

유튜브 타임라인은 2.에서 본문 텍스트에 포함했으므로 같은 방식으로 추출됩니다.

> 이 단계는 시간이 가장 오래 걸립니다(전체 약 수 분). 결과는 `cache/`에 저장되어 다시 실행하면 바로 불러옵니다.

In [ ]:
def load_word_list(name: str) -> set:
    """dict/ 폴더의 한 줄 한 단어 텍스트 사전을 읽습니다. '#'으로 시작하는 줄은 설명입니다."""
    lines = (DICT_DIR / name).read_text(encoding="utf-8").splitlines()
    return {normalize_unit(l) for l in lines if l.strip() and not l.startswith("#")}


def normalize_unit(s: str) -> str:
    """후보 단위의 표기를 통일합니다.

    NFKC 정규화 → 영문 소문자화 → 한글·영문·숫자 이외 문자 제거(띄어쓰기 포함) →
    앞에 붙은 연도·회차 표기 제거(예: '2026군산시간여행축제' → '군산시간여행축제', '제27회' 제거)
    """
    s = unicodedata.normalize("NFKC", str(s)).lower()
    s = re.sub(r"[^0-9a-z가-힣]", "", s)
    s = re.sub(r"^((19|20)\d{2}(년)?|제?\d{1,3}(회|차|기|번째))+", "", s)
    s = re.sub(r"(?<=[가-힣])\d+$", "", s)                      # '전주시의회5' → '전주시의회'
    s = re.sub(r"(?<=[가-힣])[a-z]{4,}$", "", s)                 # '한옥마을jeonjuhanok' → '한옥마을'(영문 4자 이상 꼬리)
    return s


def make_kiwi() -> Kiwi:
    """Kiwi 분석기를 만들고 dict/사용자사전.csv의 고유명사를 등록합니다(지명·축제명이 쪼개지지 않도록)."""
    kw = Kiwi(num_workers=-1)      # -1 = 사용 가능한 모든 CPU 코어(0이면 단일 스레드)
    ud = pd.read_csv(DICT_DIR / "사용자사전.csv")
    for w, tag in zip(ud["단어"], ud["품사"]):
        kw.add_user_word(str(w), tag, 0.0)
    return kw


kiwi = make_kiwi()
NOUN_TAGS = {"NNG", "NNP", "SL", "SN", "SH"}
XSN_KEEP_EXCLUDE = {"들", "적", "님", "씨", "등", "께"}
print([ (t.form, t.tag) for t in kiwi.tokenize("군산시간여행축제와 새만금 국제공항, 전주비빔밥 맛집") ])

In [ ]:
def _is_noun(tok) -> bool:
    """명사 연쇄에 포함할 토큰인지 판단합니다. 일반·고유명사, 외국어, 숫자, 한자와 일부 명사 파생 접미사를 포함합니다."""
    if tok.tag in NOUN_TAGS:
        return True
    if tok.tag == "XSN" and tok.form not in XSN_KEEP_EXCLUDE:
        return True
    if tok.tag == "XPN":
        return True
    return False


def extract_runs(tokens, text: str):
    """형태소 분석 결과에서 '어절 내 명사 연쇄'와 '어절을 잇는 인접 연쇄 쌍·3연쇄'를 뽑습니다.

    Returns
    -------
    runs : 어절 내 명사 연쇄의 원문 표기 목록(예: ['군산시간여행축제', '새만금', '국제공항'])
    ngrams : 공백 하나로만 떨어져 이어진 연쇄들의 튜플 목록(예: [('새만금','국제공항')])
             앞 연쇄 뒤에 조사·어미 등이 붙으면 끊어진 것으로 보고 잇지 않습니다.
    """
    runs, cur = [], []
    for t in tokens:
        if _is_noun(t):
            if cur and t.start == cur[-1].end:           # 공백 없이 붙어 있음 → 같은 어절의 명사 연쇄
                cur.append(t)
            else:
                if cur:
                    runs.append(cur)
                cur = [t]
        else:
            if cur:
                runs.append(cur)
                cur = []
            runs.append(None)                             # 명사가 아닌 토큰 = 연결 끊김 표시
    if cur:
        runs.append(cur)

    surfaces, ngrams, chain = [], [], []
    prev_end = None
    for r in runs:
        if r is None:
            chain, prev_end = [], None
            continue
        s = text[r[0].start:r[-1].end]
        surfaces.append(s)
        gap = text[prev_end:r[0].start] if prev_end is not None else None
        if gap is not None and gap.strip() == "" and len(gap) >= 1:
            chain.append(s)
        else:
            chain = [s]
        if len(chain) >= 2:
            ngrams.append(tuple(chain[-2:]))
        if len(chain) >= 3:
            ngrams.append(tuple(chain[-3:]))
        prev_end = r[-1].end
    return surfaces, ngrams


def tokenize_media(df: pd.DataFrame, tag: str) -> pd.DataFrame:
    """매체 하나의 모든 유효 콘텐츠를 형태소 분석해 콘텐츠별 단위 목록을 만듭니다(캐시 사용).

    Returns
    -------
    DataFrame[content_id, runs(list), ngrams(list of tuple)]
    """
    import hashlib
    sig = hashlib.md5(("|".join(df["content_id"]) + "\x1e".join(df["text"])).encode()).hexdigest()[:10]   # 텍스트 전체 기준
    cp = CACHE_DIR / f"units_{tag}_{len(df)}_{sig}.pkl"     # 콘텐츠·텍스트가 바뀌면 캐시 이름도 바뀜
    if cp.exists():
        return pd.read_pickle(cp)
    texts = df["text"].fillna("").tolist()
    t0 = time.time()
    res = []
    for text, toks in zip(texts, kiwi.tokenize(texts)):
        res.append(extract_runs(toks, text))
    out = pd.DataFrame({"content_id": df["content_id"].values,
                        "runs": [r[0] for r in res], "ngrams": [r[1] for r in res]})
    out.to_pickle(cp)
    print(f"  {tag}: {len(df):,}건 형태소 분석 {time.time()-t0:.0f}초")
    return out


# 예시
_t = "새만금 국제공항 건설 예산 확보를 추진한다. 군산시간여행축제 개막"
print(extract_runs(kiwi.tokenize(_t), _t))

In [ ]:
UNITS = {}
for m, d in zip(MEDIA, (news, ig, yt)):
    UNITS[m] = tokenize_media(d[d["valid"]], m)
    print(m, "완료:", len(UNITS[m]))

## 6. 표현 통합·필터링 → 콘텐츠×관심대상 표

각 단위를 정규화한 뒤 다음 순서로 판정합니다.

| 판정 | 기준 | 처리 |
|---|---|---|
| 정당명 | `dict/제외_정당명.txt`의 정당명(더불어민주당, 민주당, 국민의힘 …)이 들어간 단위 | **분석에서 완전히 제외**(순위·영역·HOT·RISING 모두), 제외 건수만 따로 기록 |
| 새만금 복합어 | `새만금` 단독은 지역명으로 빼고, 뒤에 오는 말과 묶인 복합어(새만금국제공항, 새만금이차전지 …)만 후보로 남김 | 후보 |
| 지역명 | `dict/지역명.txt`에 정확히 일치(전주, 전주시, 전북특별자치도 …), 하위 행정구역(완산구, 전주시완산구, ○○읍·면·동) | 순위에서 제외, 시군 정보로만 보존 |
| 불용어 | `dict/불용어.txt`, 숫자만, 1글자 | 제외 |
| 보조어 | `dict/보조어.txt`(관광·경제·지원·축제 …), *시군명+일반어*(전주맛집, 군산여행), SNS 상투 태그, 화이트리스트에 없는 2글자 단일 일반명사·영문 일반어, **14개 시군에 고르게 퍼진 일반명사(범용어 자동 판정)** | 주요 순위에서 제외, 보조어 집계표로 따로 제공 |
| 후보 | 그 밖의 모든 단위 | **관심대상 후보** |

지역명이 들어간 고유명칭(`전주비빔밥`, `군산시간여행축제`)은 지역명이 일부일 뿐이므로 지우지 않습니다(의뢰서 4.1).

**SNS(인스타·유튜브) 추가 규칙.** `전주여행`, `전주맛집`, `군산카페`, `감성카페`, `빵지순례`처럼 *어디에나 붙는 여행·맛집·카페 일반어*는 보조어로 뺍니다(`dict/SNS일반어.txt`와 일반어 결합 패턴). `전주맛집○○`처럼 *시군명+일반어 뒤에 상호·메뉴가 붙은 태그*는 앞부분을 떼고 뒤의 상호·메뉴만 후보로 남깁니다.

In [ ]:
REGION_WORDS = load_word_list("지역명.txt")
PARTY_WORDS = load_word_list("제외_정당명.txt")            # 규칙 1: 정당명은 분석에서 제외
SNS_WORDS = load_word_list("SNS일반어.txt")                # 규칙 4: SNS 여행·맛집·카페 일반어
_PARTY = re.compile("|".join(sorted(map(re.escape, PARTY_WORDS), key=len, reverse=True)))
_HEADS = sorted({normalize_unit(h) for h in CONFIG["REGION_COMPOUND_HEADS"]}, key=len, reverse=True)
_HEAD_RE = re.compile(r"^(전북특별자치도|전북자치도|전라북도|전북도|전북)?(" + "|".join(map(re.escape, _HEADS)) + ")")
_HEAD_GENERIC_TAIL = {"지역", "일대", "권역", "인근", "주변", "일원", "관련", "내", "쪽", "지구", "현장", "문제", "사업", "개발", "개발사업"}
STOP_WORDS = load_word_list("불용어.txt")
AUX_WORDS = load_word_list("보조어.txt")
WHITELIST = load_word_list("화이트리스트.txt")
USER_WORDS = {normalize_unit(w) for w in pd.read_csv(DICT_DIR / "사용자사전.csv")["단어"]}
SYN = pd.read_csv(DICT_DIR / "표현통합사전.csv")
SYN_MAP = {normalize_unit(a): str(b) for a, b in zip(SYN["원표현"], SYN["표준명칭"])}

_REGION_PREFIX = re.compile(r"^(" + "|".join(sorted(
    [r for r in REGION_WORDS if re.fullmatch(r"[가-힣]+", r)], key=len, reverse=True)) + r")")
_SNS_GENERIC = re.compile(
    r"(스타그램|그램|stagram|투어|추천|맞팔|선팔|팔로우|좋아요|소통|일상|데일리|daily|여행|여행지|여행코스|"
    r"가볼만한곳|가볼만한|가볼곳|핫플|핫플레이스|데이트|나들이|카페|맛집|먹방|브이로그|vlog|travel|trip|"
    r"food|cafe|tour|korea|photo|instagood|추억|기록|주말|휴가|당일치기)$")
# 규칙 4: '수식어* + 업종·여행 일반어' 꼴(감성카페, 대형카페, 오션뷰카페, 현지인맛집, 빵지순례, 카페투어 …)은 SNS에서 보조어
_SNS_MOD = (r"(감성|대형|신상|분위기|오션뷰|리버뷰|마운틴뷰|뷰|루프탑|한옥|애견동반|애견|반려견동반|반려견|댕댕이|아이랑|아이와|가족|커플|"
            r"혼밥|혼술|로컬|현지인|노포|숨은|인생|동네|가성비|무한리필|수제|정원|산속|시골|바다|바닷가|호수|숲속|키즈|노키즈|"
            r"가볼만한|당일|주말|데이트|드라이브|단풍|벚꽃|가을|봄|여름|겨울|국내|전국|전라도|호남|이색|예쁜|유명|핫한|조용한|"
            r"베이커리|디저트|브런치|빵|커피|술|밥|고기|회|한식|양식|중식|일식|분식|레트로|빈티지|모던|넓은|큰|작은|"
            r"신규|오픈|그랜드오픈|오픈런|웨이팅|줄서는|존맛|찐|jmt|대박|최고|추천|소개|후기|리뷰|방문)")
_SNS_HEAD = (r"(카페|맛집|식당|빵집|베이커리|술집|밥집|횟집|고깃집|브런치|디저트|여행|여행지|코스|투어|명소|핫플|핫플레이스|"
             r"데이트|데이트코스|나들이|산책|산책로|캠핑|글램핑|차박|숙소|펜션|숙박|호텔|드라이브|스냅|포토존|포토스팟|스팟|"
             r"야경|단풍|단풍명소|벚꽃|꽃구경|꽃놀이|빵지순례|카페투어|맛집투어|맛집탐방|카페탐방|먹스타그램|카페스타그램|"
             r"빵스타그램|여행스타그램|럽스타그램|일상스타그램|소품샵|편집샵|공방|원데이클래스|플리마켓|팝업|팝업스토어)")
_SNS_GENERIC_PLACE = re.compile(r"^" + _SNS_MOD + r"*" + _SNS_HEAD + r"(" + _SNS_MOD + r"|" + _SNS_HEAD + r"|스타그램|그램|탐방|순례|추천|투어)*$")
_REGION_GENERIC_PREFIX = None    # 시군명+일반어 접두(전주맛집○○ → ○○), 아래에서 지역명 목록으로 만듦
_TITLE_NAME = re.compile(r"(news|은행장|이사장|본부장|원장|사장|회장|청장|시장|군수|의장|위원장|교육감|총장|대표|교수)([가-힣]{3})")
_ADMIN_UNIT = re.compile(r"((전북|전라북도|전북특별자치도)?[가-힣]{2}(시|군))+[가-힣]{1,3}(읍|면)([가-힣]{1,3}리)?|[가-힣]{2}읍([가-힣]{1,3}리)?")   # '짜장면' 같은 음식명은 제외
_PARTICLES = {"이", "가", "은", "는", "을", "를", "의", "에", "서", "와", "과", "도", "로", "에서", "으로"}
_SUBREGION = re.compile(r"(완산구|덕진구|[가-힣]{1,3}(읍|면)|(전주시|군산시|익산시|정읍시|남원시|김제시)?[가-힣]{1,3}동|(시|군)(완산구|덕진구)?)")
_TITLE_TAIL = re.compile(r"(시장|군수|도지사|지사|의원|국회의원|교육감|위원장|의장|후보|예비후보|대표|총장|청장|본부장|이사장|회장)$")
_VERBAL_TAIL = {"추진", "개최", "실시", "진행", "마련", "확대", "강화", "선정", "참여", "운영", "지원", "협약",
                "체결", "발표", "확보", "건의", "촉구", "논의", "점검", "모집", "시행", "완료", "착수", "준공", "개막", "폐막"}

print(f"정당명 {len(PARTY_WORDS)} | SNS일반어 {len(SNS_WORDS)} | 지역명 {len(REGION_WORDS)} | 불용어 {len(STOP_WORDS)} | 보조어 {len(AUX_WORDS)} | 화이트리스트 {len(WHITELIST)} | 통합사전 {len(SYN_MAP)}")

In [ ]:
from functools import lru_cache
SYN_STD = {normalize_unit(v) for v in SYN_MAP.values()}
_UNIT_INFO_CACHE = {}


def unit_token_info(u: str):
    """정규화된 단위를 다시 형태소 분석해 (명사 토큰 수, 고유명사 포함 여부)를 돌려줍니다(결과 캐시).

    '2글자 단일 일반명사는 보조어'라는 규칙을 적용하려고 쓰며, 고유명사(NNP)로 분석되면 후보로 남깁니다.
    """
    if u not in _UNIT_INFO_CACHE:
        toks = [t for t in kiwi.tokenize(u) if _is_noun(t)]
        _UNIT_INFO_CACHE[u] = (len(toks), any(t.tag == "NNP" for t in toks))
    return _UNIT_INFO_CACHE[u]


_REGION_GENERIC_PREFIX = re.compile(
    r"^(" + "|".join(sorted([r for r in REGION_WORDS if re.fullmatch(r"[가-힣]+", r)], key=len, reverse=True)) + r")"
    r"(맛집|카페|빵집|여행|핫플|가볼만한곳|가볼곳|맛집추천|카페추천|데이트|나들이|브런치|디저트|술집|밥집|횟집|베이커리)+")


def strip_generic_prefix(u: str) -> str:
    """'시군명+일반어' 접두를 떼어 뒤의 상호·메뉴만 남깁니다(규칙 4, SNS 전용).

    예: '전주맛집베테랑' → '베테랑', '군산카페이성당' → '이성당'. 남는 부분이 2글자 미만이면 그대로 둡니다.
    시군명 없이 '카페○○'로 시작하는 상호(예: 카페라떼, 카페온)는 상호 자체일 수 있어 떼지 않습니다.
    """
    m = _REGION_GENERIC_PREFIX.match(u)
    if m and len(u) - m.end() >= 2 and u not in USER_WORDS and u not in SYN_MAP:
        return u[m.end():]
    return u


def pretty_name(u: str) -> str:
    """표시용 이름: 한글과 섞인 4자 이하 영문 약어를 대문자로 바꿉니다(새만금ai센터 → 새만금AI센터, kcc → KCC)."""
    return re.sub(r"[a-z]{1,4}(?![a-z])", lambda m: m.group(0).upper(), u) if re.search(r"[a-z]", u) and len(re.sub(r"[^a-z]", "", u)) <= 4 else u


def strip_generic_suffix(u: str) -> str:
    """해시태그 끝의 SNS 상투어(맛집, 여행, 스타그램, 데이트 …)를 반복해서 떼어 핵심 표현을 남깁니다.

    예: '전주한옥마을데이트' → '전주한옥마을', '군산맛집추천' → '군산'(→ 지역명)
    사용자사전·통합사전에 있는 표현은 떼지 않습니다(예: '군산시간여행축제').
    """
    prev = None
    while u and u != prev and u not in USER_WORDS and u not in SYN_MAP:
        prev = u
        u = _SNS_GENERIC.sub("", u)
    return u


@lru_cache(maxsize=None)
def judge_unit(u: str, from_tag: bool = False, sns: bool = False) -> tuple:
    """정규화된 단위 하나를 (판정, 표준명칭)으로 분류합니다. 판정은 정당명/지역명/불용어/보조어/후보 중 하나입니다.

    Parameters
    ----------
    u : normalize_unit()을 거친 문자열
    from_tag : 해시태그·태그에서 온 단위면 True(SNS 상투 접미어 제거 규칙 적용)
    sns : 인스타·유튜브 단위면 True(규칙 4: 여행·맛집·카페 일반어 제외, 시군명+일반어 접두 제거)
    """
    if not u:
        return "불용어", u
    if _PARTY.search(u):
        return "정당명", u                             # 규칙 1: 정당명이 들어간 단위는 분석에서 제외
    if sns:
        core = strip_generic_prefix(u)
        if core != u:
            u = core                                   # 전주맛집베테랑 → 베테랑
    if from_tag:
        core = strip_generic_suffix(u)
        if core != u:
            if not core or core in REGION_WORDS:
                return "보조어", u                     # 예: 전주맛집, 군산여행스타그램
            u = core
    u = SYN_MAP.get(u, u)                              # 표현 통합(표준명칭)
    key = normalize_unit(u)
    if _PARTY.search(key):
        return "정당명", u
    hm = _HEAD_RE.match(key)                           # 규칙 5: 새만금 + 뒤에 오는 말
    if hm:
        rest = key[hm.end():]
        if not rest:
            return "지역명", u                          # '새만금' 단독 = 지역명
        if len(rest) >= 2 and rest not in STOP_WORDS and rest not in _HEAD_GENERIC_TAIL and rest not in REGION_WORDS \
                and not _SNS_GENERIC.fullmatch(rest) and rest not in SNS_WORDS and not _SNS_GENERIC_PLACE.fullmatch(rest):
            std = key[hm.start(2):]                    # '전북새만금국제공항' → '새만금국제공항'
            return "후보", SYN_MAP.get(std, pretty_name(std))
        return "보조어", u                              # 새만금지역, 새만금여행 등
    if key in REGION_WORDS:
        return "지역명", u
    if key.startswith("person"):
        return "불용어", u                              # 유튜브 메타데이터 찌꺼기(person홍길동)
    tm = _TITLE_NAME.fullmatch(key)
    if tm:
        return judge_unit(tm.group(2), from_tag, sns)   # '이사장최정웅', 'NEWS이정민' → 이름만
    if _ADMIN_UNIT.fullmatch(key):
        return "지역명", u                              # 장수군장수읍, 전북순창군순창읍, 봉동읍, 김제시금산면
    if len(key) < 2 or key.isdigit() or key in STOP_WORDS or len(key) > CONFIG["MAX_UNIT_LEN"]:
        return "불용어", u
    if re.fullmatch(r"제?\d+[가-힣]{0,2}", key):                 # 제1, 1층, 2학기, 3인
        return "불용어", u
    if re.fullmatch(r"[0-9a-z]+", key):                         # 영문·숫자만: 화이트리스트·사용자사전에 있을 때만 후보
        return ("후보", u) if (key in WHITELIST or key in USER_WORDS or key in SYN_STD) else ("보조어", u)
    _rest = key                                                 # 지역명 접두어를 최대 3번까지 떼어 냄(전북+전주시+완산구)
    for _ in range(3):
        if not _REGION_PREFIX.match(_rest):
            break
        _nxt = _REGION_PREFIX.sub("", _rest, count=1).lstrip("시군")
        if not _nxt or _nxt == _rest:
            _rest = "" if not _nxt else _rest
            break
        _rest = _nxt
        if _rest in REGION_WORDS:
            break
    if _rest == "":
        return "지역명", u                                      # 지역명+시/군만으로 된 단위(전북시 등)
    if (_SUBREGION.fullmatch(key) and (not key.endswith(("동", "읍", "면")) or unit_token_info(key)[1])) or \
       (_rest != key and (_rest in REGION_WORDS or _SUBREGION.fullmatch(_rest))):
        return "지역명", u                                      # 하위 행정구역(완산구, 전주시완산구, 전북장수군)
    if key in AUX_WORDS:
        return "보조어", u
    if sns and (key in SNS_WORDS or _SNS_GENERIC_PLACE.fullmatch(key)):
        return "보조어", u                              # 규칙 4: 감성카페, 빵지순례, 소품샵 …
    m = _REGION_PREFIX.match(key)                     # 시군명 + 일반어 (전주맛집, 익산청년, 김제시민)
    if m and m.end() < len(key):
        rest = key[m.end():]
        if rest in REGION_WORDS or rest in _PARTICLES:
            return "지역명", u                                  # 전북전주시, 전라북도장수군, 순창군이, 익산시가
        if rest in AUX_WORDS or rest in STOP_WORDS or _SNS_GENERIC.fullmatch(rest) or rest in {"시", "군", "청", "도"}:
            return "보조어", u
        if sns and (rest in SNS_WORDS or _SNS_GENERIC_PLACE.fullmatch(rest)):
            return "보조어", u                          # 전주감성카페, 군산빵지순례
    if key in WHITELIST or key in USER_WORDS or key in SYN_STD:
        return "후보", (u if u in SYN_MAP.values() else pretty_name(u))
    if len(key) <= 2:
        n_tok, has_nnp = unit_token_info(key)
        if n_tok <= 1 and not has_nnp:
            return "보조어", u                             # 2글자 단일 일반명사(예산, 교육, 행사 …)
    return "후보", pretty_name(u)


for _u, _f, _s in [("전주맛집", True, True), ("전주한옥마을데이트", True, True), ("군산시간여행축제", False, False),
                   ("예산", False, False), ("전주시", False, False), ("새만금", False, False), ("새만금국제공항", False, False),
                   ("새만금AI센터", False, False), ("더불어민주당", False, False), ("감성카페", True, True),
                   ("전주맛집베테랑", True, True), ("익산청년", False, False), ("지평선축제", False, False)]:
    print(f"{_u:14s} → {judge_unit(normalize_unit(_u), _f, _s)}")

In [ ]:
def select_ngrams(units: pd.DataFrame, media: str) -> set:
    """어절을 잇는 복합명사 후보 중 '자주 함께 나오고 결합이 강한' 것만 고릅니다.

    기준(둘 다 충족):
      ① df(AB) ≥ NGRAM_MIN_DF[media]                 … AB가 등장한 콘텐츠 수
      ② df(AB) / min(df(A), df(B)) ≥ NGRAM_ASSOC      … 구성 단어 중 덜 흔한 쪽이 AB 안에서 쓰이는 비율
    그리고 마지막 구성어가 '추진·개최·지원' 같은 동작성 일반어이거나 '시장·군수·의원' 같은 직함이면
    (예: '우범기 전주시장' → 인물과 직함을 기계적으로 이은 것) 제외하고, 구성어 가운데 불용어가 있어도 제외합니다.

    규칙 5(새만금): 첫 구성어가 REGION_COMPOUND_HEADS(새만금)이면 결합 강도 ②를 보지 않고
    df(AB) ≥ REGION_COMPOUND_MIN_DF[media]만 확인합니다. '새만금'은 어디에나 붙는 지역명이라 결합 강도가 늘 낮게 나오지만,
    '새만금 국제공항', '새만금 이차전지'처럼 뒤에 오는 말과 묶어야 구체적인 대상이 되기 때문입니다.
    """
    df_unit, df_ng = Counter(), Counter()
    for runs, ngs in zip(units["runs"], units["ngrams"]):
        df_unit.update({normalize_unit(r) for r in runs})
        df_ng.update({tuple(normalize_unit(x) for x in ng) for ng in ngs})
    keep = set()
    min_df, assoc = CONFIG["NGRAM_MIN_DF"][media], CONFIG["NGRAM_ASSOC"]
    heads, hmin = set(_HEADS), CONFIG["REGION_COMPOUND_MIN_DF"][media]
    for ng, c in df_ng.items():
        if any(not x for x in ng):
            continue
        if ng[0] in heads:                                         # 새만금 + 뒤에 오는 말
            if c >= hmin and not any(x in heads for x in ng[1:]) and ng[-1] not in _VERBAL_TAIL \
                    and not _TITLE_TAIL.search(ng[-1]) and not any(x in STOP_WORDS for x in ng[1:]):
                keep.add(ng)
            continue
        if c < min_df:
            continue
        if ng[-1] in _VERBAL_TAIL or _TITLE_TAIL.search(ng[-1]) or any(x in STOP_WORDS for x in ng):
            continue
        if c / max(1, min(df_unit[x] for x in ng)) >= assoc:
            keep.add(ng)
    return keep


def build_content_targets(df: pd.DataFrame, units: pd.DataFrame, media: str):
    """콘텐츠별로 관심대상 후보(표준명칭) 집합을 만들어 '콘텐츠×관심대상' 긴 표(long table)로 반환합니다.

    한 콘텐츠 안에서 같은 대상이 제목·설명·태그·타임라인에 여러 번 나와도 1건만 남깁니다(의뢰서 4.3).
    보조어·지역명은 따로 모아 보조 집계용 표로 돌려줍니다.

    Returns
    -------
    ct  : DataFrame[content_id, target, src]      … 후보(src: 본문/복합/태그)
    aux : DataFrame[content_id, target, judge]     … 보조어·지역명
    mapping : Counter{(원표현, 표준명칭): 건수}      … 표현 통합 대응 기록
    """
    keep_ng = select_ngrams(units, media)
    tag_map = dict(zip(df["content_id"], df["tag_list"]))
    sns = media in ("인스타", "유튜브")
    rows, aux_rows, mapping = [], [], Counter()
    for cid, runs, ngs in zip(units["content_id"], units["runs"], units["ngrams"]):
        seen = {}
        cand = [(r, "본문") for r in runs]
        kept = [tuple(normalize_unit(x) for x in ng) for ng in ngs]
        kept = [ng for ng in kept if ng in keep_ng]
        # 새만금 복합어는 가장 긴 것만 남김(새만금 일자리 창출이 있으면 '새만금일자리'는 뺌)
        kept = [ng for ng in kept if not (ng[0] in _HEADS and any(len(o) > len(ng) and o[:len(ng)] == ng for o in kept))]
        cand += [("".join(ng), "복합") for ng in kept]
        cand += [(t, "태그") for t in tag_map.get(cid, [])]
        for raw, src in cand:
            u = normalize_unit(raw)
            judge, std = judge_unit(u, from_tag=(src == "태그"), sns=sns)
            if judge == "후보":
                if u != normalize_unit(std):
                    mapping[(u, std)] += 1
                if std not in seen:
                    seen[std] = (src, raw)
            elif judge in ("보조어", "지역명", "정당명"):
                aux_rows.append((cid, std, judge))
        rows += [(cid, t, s, r) for t, (s, r) in seen.items()]
    ct = pd.DataFrame(rows, columns=["content_id", "target", "src", "raw"])
    aux = pd.DataFrame(aux_rows, columns=["content_id", "target", "judge"]).drop_duplicates()
    return ct, aux, mapping


CT, AUX, MAPPING = {}, {}, Counter()
for m, d in zip(MEDIA, (news, ig, yt)):
    t0 = time.time()
    CT[m], AUX[m], mp = build_content_targets(d[d["valid"]], UNITS[m], m)
    MAPPING.update(mp)
    print(f"{m}: 후보 출현 {len(CT[m]):,}행, 고유 후보 {CT[m]['target'].nunique():,}개 ({time.time()-t0:.0f}초)")

In [ ]:
def attach_meta(ct: pd.DataFrame, df: pd.DataFrame) -> pd.DataFrame:
    """콘텐츠×관심대상 표에 날짜·연·월·시군·출처·빈도 산정 단위를 붙입니다."""
    meta_cols = ["content_id", "count_key", "rep_key", "date", "year", "month", "sigun_all", "source", "type", "수집층"]
    if "collect_mode" in df:
        meta_cols.append("collect_mode")
    return ct.merge(df[meta_cols], on="content_id", how="left")


for m, d in zip(MEDIA, (news, ig, yt)):
    CT[m] = attach_meta(CT[m], d)
    AUX[m] = attach_meta(AUX[m], d)


def collapse_source_boilerplate(ct: pd.DataFrame, df: pd.DataFrame, media: str):
    """같은 채널(유튜브)·작성자(인스타)가 거의 모든 콘텐츠에 반복해 붙이는 대상은 그 출처에서 *1건만* 셉니다.

    예: 휴대폰 수리점 계정이 200여 개 게시물마다 같은 해시태그 묶음('액정수리', '데이터복구' …)을 붙이거나,
    부동산 채널이 영상마다 같은 설명란 문구('급매물', '권리분석' …)를 넣으면 그 단어가 상위에 오릅니다.
    이런 반복 생산은 여러 이용자의 관심이 아니므로, 출처 콘텐츠가 SOURCE_BOILER_MIN건 이상이고
    대상이 그 출처 콘텐츠의 SOURCE_BOILER_SHARE(30건 이상이면 _BIG) 이상에 나오면 가장 이른 1건만 남깁니다.
    다른 출처의 언급은 그대로 셉니다. 뉴스(언론사 6곳)에는 적용하지 않습니다.

    Returns
    -------
    ct  : 반복분을 뺀 콘텐츠×관심대상 표
    log : DataFrame[매체, 출처, 출처콘텐츠수, 관심대상, 원래건수, 반영건수]
    """
    cols = ["매체", "출처", "출처콘텐츠수", "관심대상", "원래건수", "반영건수"]
    if media == "뉴스":
        return ct.assign(반복생산=False), pd.DataFrame(columns=cols)
    n_src = df[df["valid"]].groupby("source")["content_id"].nunique()
    pair = ct.groupby(["source", "target"])["content_id"].nunique()
    src_n = n_src.reindex(pair.index.get_level_values(0)).values
    thr = np.where(src_n >= 30, CONFIG["SOURCE_BOILER_SHARE_BIG"], CONFIG["SOURCE_BOILER_SHARE"])
    flag = pair[(src_n >= CONFIG["SOURCE_BOILER_MIN"]) & (pair.values / src_n >= thr) & (pair.values >= 2)]
    if flag.empty:
        return ct.assign(반복생산=False), pd.DataFrame(columns=cols)
    c = ct.copy()
    is_flag = pd.MultiIndex.from_frame(c[["source", "target"]]).isin(flag.index)
    # v2: 원자료 빈도를 숨기지 않도록 행은 그대로 두고, 반복군 키만 '출처 단위'로 바꿔 반복군 보정 집계에서 1건으로 셈
    c.loc[is_flag, "rep_key"] = "SRC|" + c.loc[is_flag, "source"].astype(str)
    c["반복생산"] = is_flag
    log = flag.rename("원래건수").reset_index()
    log.columns = ["출처", "관심대상", "원래건수"]
    log.insert(0, "매체", media)
    log.insert(2, "출처콘텐츠수", log["출처"].map(n_src))
    log["반영건수"] = 1
    return c, log.sort_values("원래건수", ascending=False)[cols]


CT["유튜브"] = CT["유튜브"][~CT["유튜브"]["target"].isin(YT_STAFF)].reset_index(drop=True)   # 본문에 남은 제작진 이름 제외

SRC_BOILER = []
for m, d in zip(MEDIA, (news, ig, yt)):
    before = len(CT[m])
    CT[m], lg = collapse_source_boilerplate(CT[m], d, m)
    SRC_BOILER.append(lg)
    if m != "뉴스":
        print(f"{m}: 반복 생산 표시 {len(lg):,}쌍(출처×대상), 해당 행 {int(CT[m]['반복생산'].sum()):,}(원집계에는 포함, 반복군 보정 집계에서 출처당 1건)")
SRC_BOILER = pd.concat(SRC_BOILER, ignore_index=True)

# 규칙 1: 정당명 제외 기록(매체별 제외된 표현과 콘텐츠 수)
PARTY_EXCLUDED = pd.concat([
    AUX[m][AUX[m]["judge"] == "정당명"].groupby("target")["count_key"].nunique().rename("제외 콘텐츠 수")
       .reset_index().assign(매체=m) for m in MEDIA], ignore_index=True)
PARTY_EXCLUDED = PARTY_EXCLUDED.rename(columns={"target": "제외 표현"})[["매체", "제외 표현", "제외 콘텐츠 수"]] \
    .sort_values(["매체", "제외 콘텐츠 수"], ascending=[True, False])
PARTY_EXCLUDED["제외 사유"] = "정당명(분석 제외)"
print("정당명이 나와 제외 처리된 콘텐츠 수:",
      {m: int(AUX[m].loc[AUX[m]["judge"] == "정당명", "count_key"].nunique()) for m in MEDIA})
display(SRC_BOILER.head(15))

def _area_rule_hit(target: str) -> str:
    """영역분류규칙.csv의 첫 번째로 맞는 영역(없으면 빈 문자열). 범용어 판정에서만 쓰는 간단한 버전입니다."""
    key = normalize_unit(target)
    for area, pat in _AREA_RULES_SIMPLE:
        if pat.search(key):
            return area
    return ""


_AREA_RULES_SIMPLE = [(r["영역"], re.compile(r["패턴"], re.I))
                      for _, r in pd.read_csv(DICT_DIR / "영역분류규칙.csv").sort_values("우선순위").iterrows()]


_SURNAMES = set("김이박최정강조윤장임한오서신권황안송류유홍전고문양손배백허남심노하곽성차주우구민진나지엄채원천방공현함변염여추도소석선설마길연위표명기반왕금옥육인맹제모탁국어은편용예경봉")
_TITLE_AFTER = (r"\s?(전북|전북도|전북특별자치도|[가-힣]{2}(시|군)|[가-힣]{2})?\s?(국회의원|의원|시장|군수|도지사|지사|교육감|후보|예비후보|"
                r"위원장|의장|대표|당선인|당선자|전\s?의원|전\s?시장|부지사|부시장|부군수|원내대표|최고위원|장관|총장|대통령|"
                r"전\s?대통령|여사|총리|청장|이사장|은행장|본부장|원장|사장|회장|교수|변호사|감독|선수)")


_PERSON_STATS = {}


def _person_title_counts():
    """뉴스 전체를 *한 번만* 훑어 '3~4글자 한글 + (지역) + 직함' 패턴의 앞 단어별 건수와 단어별 등장 기사 수를 셉니다."""
    if not _PERSON_STATS:
        pat = re.compile(r"(?<![가-힣])([가-힣]{2,3})" + _TITLE_AFTER)
        # v2: '서장 홍길동', '홍길동 씨'처럼 직함이 앞에 오거나 '씨'가 붙는 꼴도 인명 단서로 셈
        pat2 = re.compile(r"(?:서장|청장|원장|국장|과장|팀장|센터장|관장|회장|대표|위원장|이사장|교수|소장|단장|본부장|지사|권한대행)\s?([가-힣]{3})(?![가-힣])")
        pat3 = re.compile(r"(?<![가-힣])([가-힣]{3})\s?씨(?:는|가|의|를|에게)?(?![가-힣])")
        title_cnt = Counter()
        for t in news.loc[news["valid"], "text"]:
            title_cnt.update({m.group(1) for p_ in (pat, pat2, pat3) for m in p_.finditer(t)})
        _PERSON_STATS["title"] = title_cnt
        _PERSON_STATS["df"] = CT["뉴스"].groupby("target")["rep_key"].nunique()   # 기사군 기준(1차와 동일)
    return _PERSON_STATS["title"], _PERSON_STATS["df"]


def looks_like_person(word: str, min_ratio: float = 0.2) -> bool:
    """3~4글자 한글이고 성씨로 시작하는 단어가 뉴스에서 직함(의원·시장·군수·후보 등) 앞에 자주 오면 인명으로 봅니다.

    Kiwi는 사전에 없는 인명을 일반명사(NNG)로 분석하므로, 범용어 판정에서 인명이 빠지지 않도록 하는 보호 규칙입니다.
    판정 기준: (단어 + 직함 형태가 나온 기사 수) ÷ (단어가 나온 기사군 수) ≥ min_ratio, 그리고 직함 형태 5건 이상.
    """
    if not re.fullmatch(r"[가-힣]{2,3}", word) or word[0] not in _SURNAMES:
        return False
    key = normalize_unit(word)
    if key in WHITELIST or key in USER_WORDS or key in SYN_STD or _area_rule_hit(word):
        return False                                  # 특산품·지명·축제처럼 이미 정해진 대상(반딧불 등)은 인명이 아님
    title_cnt, df = _person_title_counts()
    n_title, n = title_cnt.get(word, 0), df.get(word, 0)
    return n_title >= 3 and n > 0 and n_title / n >= min_ratio


def detect_spread_generic(min_n=30, min_cover=10, min_entropy=0.8) -> pd.DataFrame:
    """14개 시군에 고르게 퍼진 *일반명사*를 범용어(보조어)로 판정합니다.

    '관광객', '지역사회', '마무리'처럼 어느 지역에서나 쓰이는 말은 단어 목록으로 다 막기 어렵습니다.
    그래서 세 매체를 합친 시군 분포로 판단합니다. 조건은 아래와 같습니다(모두 충족 시 범용어).
      ① 등장 콘텐츠 min_n건 이상
      ② min_cover개 이상 시군에서 등장('전북 공통' 제외)
      ③ 시군 분포의 정규화 엔트로피 ≥ min_entropy(1에 가까울수록 고르게 퍼짐)
      ④ 형태소 분석상 고유명사(NNP)가 없고, 화이트리스트·사용자사전·통합사전에 없으며,
         영역 규칙상 상품·장소·문화·기업 대상으로 분류되지 않음(예: '반도체', '가을축제'는 유지)
    '한옥마을'(전주 집중), '김관영'(고유명사)은 남고 '관광객', '지역경제'는 빠집니다.
    """
    allct = pd.concat([CT[m][["target", "content_id", "sigun_all"]] for m in MEDIA])
    e = allct.explode("sigun_all")
    e = e[e["sigun_all"].isin(SIGUN)]
    dist = e.groupby(["target", "sigun_all"])["content_id"].nunique().unstack(fill_value=0)
    n = allct.groupby("target")["content_id"].nunique()
    p = dist.div(dist.sum(axis=1), axis=0)
    ent = -(p * np.log(p.where(p > 0, 1))).sum(axis=1) / np.log(len(SIGUN))
    cover = (dist > 0).sum(axis=1)
    cand = pd.DataFrame({"n": n, "시군수": cover, "엔트로피": ent}).dropna()
    cand = cand[(cand["n"] >= min_n) & (cand["시군수"] >= min_cover) & (cand["엔트로피"] >= min_entropy)]
    keep_areas = {"상품·먹거리", "장소·공간", "문화·콘텐츠·스포츠", "기업·산업"}
    rows = []
    for t, r in cand.iterrows():
        key = normalize_unit(t)
        if key in WHITELIST or key in USER_WORDS or key in SYN_STD:
            continue
        if unit_token_info(key)[1]:                 # 고유명사 포함
            continue
        if _REGION_PREFIX.match(key):               # 지역명으로 시작하는 기관·대상(전북농협 등)
            continue
        if looks_like_person(t):                    # 형태소 분석기가 모르는 인명(이원택 등)
            continue
        if _area_rule_hit(t) in keep_areas:
            continue
        rows.append((t, int(r["n"]), int(r["시군수"]), round(r["엔트로피"], 3)))
    return pd.DataFrame(rows, columns=["범용어", "등장건수", "등장시군수", "정규화엔트로피"]).sort_values("등장건수", ascending=False)


SPREAD_GENERIC = detect_spread_generic()
_gen = set(SPREAD_GENERIC["범용어"])
for m in MEDIA:
    moved = CT[m][CT[m]["target"].isin(_gen)].drop(columns=["src", "raw", "반복생산"], errors="ignore").assign(judge="보조어")
    AUX[m] = pd.concat([AUX[m], moved], ignore_index=True)
    CT[m] = CT[m][~CT[m]["target"].isin(_gen)].reset_index(drop=True)
print(f"범용어로 판정해 보조어로 옮긴 대상: {len(_gen)}개")
print("예:", ", ".join(SPREAD_GENERIC["범용어"].head(40)))

# 표현 통합 대응표(원표현 → 표준명칭, 적용 건수): 산출물 ②에 포함
MAPPING_TABLE = pd.DataFrame([(a, b, c) for (a, b), c in MAPPING.items()],
                             columns=["원표현(정규화)", "표준명칭", "적용 건수"]).sort_values("적용 건수", ascending=False)
MAPPING_TABLE.head(20)

## 7. 광고성 콘텐츠·비대상 표현 제외, 지역 연결 불명 식품, 6개 분야 분류

**7.0 광고·홍보성 해시태그 나열 콘텐츠(2차 검토 반영).** 한 게시물의 해시태그에 `전주전자담배 익산전자담배 정읍전자담배 …`처럼 *지역명 + 같은 상품·서비스어*가 4개 이상 나열되면, 지역 관심이 아니라 검색 노출용 광고로 보고 **콘텐츠 전체를 분석에서 뺍니다**(02_전처리집계 S4b). 이때 나온 상품·서비스어(전자담배·미용가위 등)는 `dict/제외_광고홍보어.txt`의 수기 목록과 함께 광고·홍보성 표현으로 묶여, 다른 콘텐츠에 나와도 주순위에서 빠집니다. `맛집·여행·카페`처럼 일반어 결합(보조어)은 판정에 쓰지 않습니다.

In [ ]:
AD_WORDS = load_word_list("제외_광고홍보어.txt")
AD_EXEMPT = load_word_list("광고판정_예외어.txt")   # 지역명과 함께 나열돼도 광고로 보지 않는 먹거리·관광 표현(군산짬뽕·전주비빔밥 등)
_EXTRA_PLACE = ["혁신도시", "전북혁신도시", "전주혁신도시", "전북대", "전주대", "원광대", "군산대", "우석대", "에코시티", "신시가지",
                "객사", "하가지구", "만성지구", "이서", "아중", "덕진", "완산", "효자", "송천", "서신", "중화산", "평화", "삼천", "인후", "우아",
                "광주", "서울", "대전", "부산", "대구", "인천", "세종", "청주", "천안", "여수", "순천", "목포", "전남", "충남", "경기"]
_PLACE_PREFIX = sorted({w for w in REGION_WORDS if re.search("[가-힣]", w)} | set(SIGUN_BASE) | set(SIGUN) | set(_EXTRA_PLACE),
                       key=len, reverse=True)
_PREFIX_RE = re.compile(r"^(?:" + "|".join(map(re.escape, _PLACE_PREFIX)) + r"|[가-힣]{1,3}동)(?=[가-힣a-z0-9]{2,}$)")
_GENERIC_STEM = SNS_WORDS | AUX_WORDS | STOP_WORDS | AD_EXEMPT


def ad_stuffing(media: str, d: pd.DataFrame, min_var: int):
    """해시태그에 '지역명+같은 표현'이 min_var개 이상 나열된 콘텐츠와 그 표현(stem)을 찾습니다(SNS만)."""
    if media == "뉴스":
        return set(), Counter()
    tags = d.set_index("content_id")["tags"].fillna("").map(lambda s: normalize_unit(s.replace("#", " ")))
    c = pd.concat([CT[media][["content_id", "target"]], AUX[media][["content_id", "target"]]]).drop_duplicates()
    c = c[c["content_id"].isin(tags.index)]
    stem = c["target"].map(lambda t: _PREFIX_RE.sub("", normalize_unit(t)))
    c = c.assign(stem=stem)
    c = c[(c["stem"] != c["target"].map(normalize_unit)) & (c["stem"].str.len() >= 2) & ~c["stem"].isin(_GENERIC_STEM)]
    in_tag = [normalize_unit(t) in tags.get(cid, "") for cid, t in zip(c["content_id"], c["target"])]
    c = c[in_tag]
    g = c.groupby(["content_id", "stem"])["target"].nunique()
    g = g[g >= min_var].reset_index()
    return set(g["content_id"]), Counter(g["stem"])


AD_LOG = []
AD_STEMS = Counter()
for m, d in zip(MEDIA, (news, ig, yt)):
    ids, stems = ad_stuffing(m, d[d["valid"]], CONFIG["AD_STUFF_MIN_VARIANTS"])
    AD_STEMS.update(stems)
    d["광고나열"] = d["content_id"].isin(ids)
    d.loc[d["광고나열"], "valid"] = False
    n_ct = int(CT[m]["content_id"].isin(ids).sum())
    CT[m] = CT[m][~CT[m]["content_id"].isin(ids)].reset_index(drop=True)
    AUX[m] = AUX[m][~AUX[m]["content_id"].isin(ids)].reset_index(drop=True)
    AD_LOG.append({"매체": m, "제외콘텐츠수": len(ids), "제외대상행수": n_ct, "나열표현": ", ".join(f"{s}({n})" for s, n in stems.most_common(15))})
    print(f"{m}: 광고·홍보성 해시태그 나열 콘텐츠 {len(ids):,}건 분석 제외")
AD_MANUAL = set(AD_WORDS)
_INST_LIKE = re.compile(r"(청|교육청|학교|대학|센터|시청|군청|도청|의회|협회|공단|공사|재단|위원회)$")
AD_AUTO = {s for s, n in AD_STEMS.items() if n >= CONFIG["AD_STEM_AUTO_MIN"] and not _INST_LIKE.search(s) and s not in AD_EXEMPT}
AD_WORDS = AD_MANUAL | AD_AUTO
print("광고·홍보성 표현(수기):", sorted(AD_MANUAL), "\n광고·홍보성 표현(자동):", sorted(AD_AUTO))
AD_LOG = pd.DataFrame(AD_LOG)
display(AD_LOG)

### 7.1~7.3 행위자·비대상 표현, 식품, 분야

**7.1 행위자 단독 표현(과업지시서 6.2, 6.3).** 다음은 핵심 관심대상 순위에서 빼고 `보조` 표(05_제외보조어사전, 원장)에 남깁니다. 콘텐츠 자체는 지우지 않으므로 같은 콘텐츠의 정책·장소·행사 대상은 그대로 집계됩니다.

| 구분 | 기준 | 처리 |
|---|---|---|
| 지정인명 | `dict/제외_지정인명.txt`(이재명·윤석열·김관영·이원택) | 주순위 제외(발주자 지정) |
| 인명(자동 판정) | 뉴스에서 직함과 함께 자주 쓰이는 2~3글자 이름, 수기 분류표의 인물 | 주순위 제외, **승인 필요**로 표시 |
| 주소·타지역 | '대전역·서울' 같은 도 밖 지명 단독, '군산시구영3길', '하나로11길'처럼 도로명 주소, '고창군공음면학원농장길'처럼 시군+읍면이 이어진 표현, 원문에서 뒤에 번지가 붙는 '○○길·○○로'(예: 학원농장길 150), '인천광역시' 같은 도 밖 광역시 | 주순위 제외(업체·장소 안내 문구). 장소 자체(학원농장)는 유지 |
| 선거구명 | '전주병·익산갑'처럼 시군+갑·을·병·정, '○○선거구' | 주순위 제외(국회의원·지방의원 선거구 단위. '전주 병·의원' 같은 오추출도 함께 걸러짐) |
| 상호·지점명 | '전주한옥마을점', '전주혁신점', '○○본점'처럼 지점을 뜻하는 '점'으로 끝나는 표현(백화점·서점·반점 등 일반명사는 제외) | 주순위 제외 |
| 광고·홍보성 | 7.0의 광고·홍보성 표현과 그 지역명 결합형(전주전자담배 등) | 주순위 제외 |
| 기관·직함 단독 | '전북대학교', '전주시의회', '도지사', '고용노동부', '○○센터', '○○협의체'처럼 기관·직함·단체로 *끝나는* 표현(수기 분류표에서 다른 분야로 확정한 것은 제외) | 주순위 제외. '전북대학교축제'처럼 뒤에 구체 대상이 붙으면 유지 |

**7.2 지역 연결이 필요한 식품(6.3).** '치즈·사과·한우'처럼 어디서나 쓰는 식품명은 단독으로 로컬 대상이라 단정하지 않습니다. 같은 콘텐츠에 '임실 치즈', '장수 사과'처럼 *시군명+식품명*이 명시되면 `임실치즈`, `장수사과`로 연결하고, 아니면 '식품 주제(보조·검토)'로 둡니다. 민원·사과문의 '사과'도 이 규칙으로 걸러집니다.

**7.3 6개 분야.** `dict/영역분류규칙.csv`(우선순위 정규식)와 `dict/영역분류_수기.csv`(수기 확정, 우선)로 주분야 1개와 보조분야를 줍니다. 6번째 분야는 단독 인물·기관 순위가 아니라 *지역의 구체 활동·서비스·사회적 쟁점*이며, 정치·선거 이슈는 하위유형으로 따로 표시합니다.

In [ ]:
DESIGNATED_PERSONS = load_word_list("제외_지정인명.txt")
LINK_FOODS = load_word_list("지역연결필요식품.txt")
AREAS = ["상품·먹거리", "장소·공간", "문화·콘텐츠·스포츠", "기업·산업", "정책·공공사업", "인물·기관·사회 현상"]
AREA_RULES = pd.read_csv(DICT_DIR / "영역분류규칙.csv").sort_values("우선순위")
AREA_RULES["re"] = AREA_RULES["패턴"].map(lambda p: re.compile(p, re.I))
MANUAL = pd.read_csv(DICT_DIR / "영역분류_수기.csv").dropna(subset=["관심대상"])
MANUAL_MAP = {normalize_unit(r["관심대상"]): (r["주분류"], r["보조분류"] if isinstance(r["보조분류"], str) else "",
                                            str(r["메모"]) if isinstance(r["메모"], str) else "") for _, r in MANUAL.iterrows()}
_MANUAL_PERSON = re.compile(r"(인물|군수|시장|총장|은행장|가수|방송인|도지사|의원)")
_ACTOR_INST = re.compile(
    r"(도지사|부지사|지사|군수|(전주|군산|익산|정읍|남원|김제)시장|부시장|부군수|교육감|국회의원|의원|도의원|시의원|군의원|의장|위원장|총장|"
    r"청장|대통령|장관|총리|교육청|교육지원청|경찰서|경찰청|경찰|해경|소방서|소방본부|법원|지법|지검|검찰청|검찰|공사|공단|재단|진흥원|"
    r"연구원|의료원|보건소|체육회|위원회|협회|연합회|농협|병원|대학교|대학|교대|개발청|기상청|시청|군청|도청|의회|도의회|시의회|군의회|"
    r"상공회의소|사무소|학교|고등학교|중학교|초등학교|지원청|지청|본청|교육원|관리원|지원단|사업소|"
    r"[가-힣]{2,}청|중앙회|운동연합|연합|센터|본부|홍보단|협의체|협의회|청년회|부녀회|봉사단|봉사회|동호회|추진단|자문단|서포터즈|기자단|노동조합|노조|노인회|적십자사|노총|소방|"
    r"(노동|교통|해양수산|자원|안전|복지|가족|벤처기업|통신|교육|국방|외교|법무|환경|통일|재정|관광|식품|산림|농림축산식품)부)$")


_ADDRESS = re.compile(r"(\d+(번)?길|\d+(번)?로|로\d+|길\d+|(광역시|특별시|특별자치시)|번지)$")
_OUTSIDE = re.compile(r"^(서울|부산|대구|인천|광주|대전|울산|세종|수원|청주|천안|여수|순천|목포|제주|강릉|경주|구미|포항|창원|김해|산청|함양|거창|하동|진주|통영|광양|구례|곡성|담양|논산|대천)(역|시|광역시|터미널|공항|여행|맛집)?$")
_ADMIN_SEQ = re.compile(r"(시|군)[가-힣]{1,4}(읍|면|동)|^[가-힣]{1,4}(읍|면)[가-힣]{2,}(길|로)$")
_SIGUN_ALT = "|".join(sorted(set(SIGUN_BASE) | {"전북", "도", "광역"}, key=len, reverse=True))
_DISTRICT = re.compile(rf"^({_SIGUN_ALT})(시|군)?[갑을병정]$|선거구(민)?$")
_KEEP_JEOM = ("백화점", "서점", "반점", "편의점", "잡화점", "상점", "매점", "음식점", "제과점", "주점", "문구점", "철물점", "거점", "정점",
              "접점", "시점", "관점", "장점", "단점", "강점", "약점", "쟁점", "초점", "문제점", "출발점", "종점", "기점", "원점", "분기점",
              "교차점", "만점", "득점", "실점", "허점", "오점", "공통점", "차이점", "개선점", "요점", "중점", "논점", "전환점", "시작점",
              "반환점", "연결점", "맛점", "빵점", "합점", "구멍가게점", "화점", "꽃집점")
ADDRESS_LIKE = set()          # 7.1 아래에서 원문의 '○○길 150' 꼴로 판정해 채움


def is_branch(key: str) -> bool:
    """'전주한옥마을점'·'전주혁신점'·'○○본점'처럼 지점을 뜻하는 표현인지."""
    return len(key) >= 3 and key.endswith("점") and not key.endswith(_KEEP_JEOM)


def is_ad(key: str) -> bool:
    """광고·홍보성 표현 또는 '지역명+광고 표현'."""
    return key in AD_WORDS or _PREFIX_RE.sub("", key) in AD_WORDS or any(key.endswith(w) for w in AD_MANUAL if len(w) >= 3)


def actor_type(target: str) -> str:
    """주순위에서 뺄 표현이면 그 유형을, 아니면 빈 문자열을 돌려줍니다.

    유형: 주소·타지역 / 선거구명 / 상호·지점명 / 광고·홍보성 / 지정인명 / 인명 / 기관·직함.
    수기 분류표에서 인물·기관 분야가 아닌 분야로 확정한 표현(예: 전북은행 → 기업·산업)은 행위자로 보지 않습니다.
    """
    key = normalize_unit(target)
    if _ADDRESS.search(str(target).lower()) or _ADDRESS.search(key) or _ADMIN_SEQ.search(key) or key in ADDRESS_LIKE or _OUTSIDE.search(key):
        return "주소·타지역"
    if _DISTRICT.search(key):
        return "선거구명"
    if is_ad(key):
        return "광고·홍보성"
    if is_branch(key) and key not in MANUAL_MAP:
        return "상호·지점명"
    if key in DESIGNATED_PERSONS:
        return "지정인명"
    if key in MANUAL_MAP:
        a, _, memo = MANUAL_MAP[key]
        if a != "인물·기관·사회 현상":
            return ""
        if _MANUAL_PERSON.search(memo):
            return "인명"
        if "기관" in memo:
            return "기관·직함"
        return ""
    if re.fullmatch(r"(서장|청장|군수|시장|원장|국장|의장|회장|대표)[가-힣]{2,3}", key) or re.search(r"(권한대행|시장직|군수직|의원직|지사직)$", key):
        return "기관·직함"
    if _ACTOR_INST.search(key):
        return "기관·직함"
    if looks_like_person(target):
        return "인명"
    return ""


def link_foods(media: str, df: pd.DataFrame):
    """지역 연결이 필요한 식품명을 '시군명+식품명' 표준 대상으로 바꾸거나 보조(식품 주제)로 옮깁니다.

    Returns
    -------
    ct_new : 처리 후 콘텐츠×관심대상 표
    moved  : 보조로 옮긴 행(judge='식품주제(지역연결불명)')
    log    : DataFrame[매체, 원표현, 처리, 연결대상, 건수]
    """
    c = CT[media]
    is_food = c["target"].map(lambda t: normalize_unit(t) in LINK_FOODS)
    if not is_food.any():
        return c, c.iloc[0:0].assign(judge=""), pd.DataFrame()
    text = df.set_index("content_id")["text"]
    raw_text = (df["title"].fillna("") + " " + df["body"].fillna("") + " " + df["tags"].fillna("")).set_axis(df["content_id"])
    bases = sorted(set(SIGUN_BASE) | {"장수군", "완주군"}, key=len, reverse=True)
    f = c[is_food].copy()
    new_t = []
    for cid, t in zip(f["content_id"], f["target"]):
        food = normalize_unit(t)
        s = f"{text.get(cid, '')} {raw_text.get(cid, '')}"
        m = re.search(r"(" + "|".join(bases) + r")(시|군)?\s?#?" + re.escape(food), s)
        if m:
            std = m.group(1).replace("군", "") if m.group(1) in ("장수군", "완주군") else m.group(1)
            std = SYN_MAP.get(normalize_unit(std + food), std + food)
            new_t.append(std)
        else:
            new_t.append(None)
    f["new"] = new_t
    linked = f[f["new"].notna()].assign(target=lambda x: x["new"]).drop(columns="new")
    moved = f[f["new"].isna()].drop(columns=["new", "src", "raw", "반복생산"], errors="ignore").assign(judge="식품주제(지역연결불명)")
    rest = c[~is_food]
    out = pd.concat([rest, linked], ignore_index=True).drop_duplicates(["content_id", "target"])
    log = pd.concat([
        f[f["new"].notna()].groupby(["target", "new"]).size().rename("건수").reset_index()
          .rename(columns={"target": "원표현", "new": "연결대상"}).assign(처리="시군명+식품명으로 연결"),
        f[f["new"].isna()].groupby("target").size().rename("건수").reset_index()
          .rename(columns={"target": "원표현"}).assign(처리="식품 주제(보조·검토)", 연결대상=""),
    ], ignore_index=True)
    log.insert(0, "매체", media)
    return out, moved, log


def address_like_targets(min_n: int = 3, share: float = 0.5) -> set:
    """원문에서 뒤에 번지가 붙는 '○○길·○○로'(예: '학원농장길 150')를 주소로 판정합니다.

    '변산마실길 1코스'처럼 코스·구간 번호는 번지로 보지 않습니다. 대상이 나온 콘텐츠의 절반 이상에서 번지가 붙으면 주소.
    """
    raw = {}
    for m, d in zip(MEDIA, (news, ig, yt)):
        v = d[d["valid"]]
        raw[m] = (v["title"].fillna("") + " " + v["body"].fillna("") + " " + v["tags"].fillna("")).set_axis(v["content_id"]).to_dict()
    hit, tot = Counter(), Counter()
    for m in MEDIA:
        c = CT[m][CT[m]["target"].map(lambda t: bool(re.fullmatch(r"[가-힣0-9]{2,}(길|로)", normalize_unit(t))))]
        for t, g in c.groupby("target")["content_id"]:
            pat = re.compile(r"\s?".join(map(re.escape, normalize_unit(t))) + r"\s?\d+(-\d+)?(?!\s?(코스|구간|km|㎞|번째|회|년|월|일|시|분|명|개|곳|종|길))")
            for cid in g.values[:80]:
                tot[t] += 1
                hit[t] += bool(pat.search(raw[m].get(cid, "")))
    return {normalize_unit(t) for t in tot if tot[t] >= min_n and hit[t] / tot[t] >= share}


ADDRESS_LIKE = address_like_targets()
print(f"번지가 붙는 길·로 표현 {len(ADDRESS_LIKE):,}개 → 주소로 처리. 예:", sorted(ADDRESS_LIKE)[:15])

ACTOR_LOG, FOOD_LOG = [], []
for m, d in zip(MEDIA, (news, ig, yt)):
    CT[m], moved, lg = link_foods(m, d[d["valid"]])
    AUX[m] = pd.concat([AUX[m], moved], ignore_index=True)
    FOOD_LOG.append(lg)
    tg = CT[m]["target"].unique()
    at = {t: actor_type(t) for t in tg}
    at = {t: v for t, v in at.items() if v}
    is_actor = CT[m]["target"].isin(at)
    mv = CT[m][is_actor].drop(columns=["src", "raw", "반복생산"], errors="ignore")
    mv = mv.assign(judge=mv["target"].map(at))
    AUX[m] = pd.concat([AUX[m], mv], ignore_index=True)
    ACTOR_LOG.append(mv.groupby(["target", "judge"])["content_id"].nunique().rename("콘텐츠수").reset_index().assign(매체=m))
    CT[m] = CT[m][~is_actor].reset_index(drop=True)
    print(f"{m}: 행위자 단독 표현 {len(at):,}개({int(is_actor.sum()):,}행) 보조로 이동, 남은 후보 {CT[m]['target'].nunique():,}개")
ACTOR_LOG = pd.concat(ACTOR_LOG, ignore_index=True).sort_values("콘텐츠수", ascending=False)
FOOD_LOG = pd.concat(FOOD_LOG, ignore_index=True).sort_values("건수", ascending=False)
display(ACTOR_LOG.groupby(["매체", "judge"])["target"].nunique().unstack(fill_value=0))
display(ACTOR_LOG.head(20))
display(FOOD_LOG.head(20))

In [ ]:
_POLITICS = re.compile(r"(선거|후보|공천|경선|투표|탄핵|정치|국회|당선|출마|여론조사|선거구|전당대회|타운홀|정당|계엄|대선|총선|지선|국정감사|국감|도정질문|시정질문)")


@lru_cache(maxsize=None)
def classify_area(target: str) -> tuple:
    """관심대상 하나를 (주분야, 보조분야, 분류근거, 검토필요, 하위유형)으로 분류합니다.

    수기 분류표가 우선이고, 없으면 규칙 순서대로 맞춰 봅니다. 6번째 분야(인물·기관·사회 현상)는
    정치·선거 이슈와 그 밖의 사회 현상을 하위유형으로 구분합니다.
    """
    key = normalize_unit(target)
    if key in MANUAL_MAP:
        a, b, memo = MANUAL_MAP[key]
        sub = ("정치·선거" if _POLITICS.search(key) else "사회 현상") if a == AREAS[5] else ""
        return a, b, "수기 분류: " + memo, a not in AREAS, sub
    hits = []
    for _, r in AREA_RULES.iterrows():
        mm = r["re"].search(key)
        if mm and r["영역"] not in [h[0] for h in hits]:
            hits.append((r["영역"], mm.group(0)))
    if not hits:
        return "미분류(검토필요)", "", "규칙 불일치", True, ""
    main, sub = hits[0], (hits[1] if len(hits) > 1 else ("", ""))
    why = f"'{main[1]}' → {main[0]}" + (f" / '{sub[1]}' → {sub[0]}" if sub[0] else "")
    sub_t = ("정치·선거" if _POLITICS.search(key) else "사회 현상") if main[0] == AREAS[5] else ""
    return main[0], sub[0], why, len(hits) > 1, sub_t


t0 = time.time()
_n_any = pd.concat([CT[m].groupby("target")["content_id"].nunique() for m in MEDIA], axis=1).max(axis=1)
_pool = _n_any[_n_any >= CONFIG["CLASSIFY_MIN_N"]].index
AREA_TABLE = pd.DataFrame([(t,) + classify_area(t) for t in _pool],
                          columns=["target", "주분야", "보조분야", "분류근거", "검토필요", "하위유형"]).set_index("target")
AREA_MAP = AREA_TABLE["주분야"].to_dict()
print(f"분야 분류 대상 {len(AREA_TABLE):,}개 ({time.time()-t0:.0f}초)")
display(AREA_TABLE["주분야"].value_counts())

## 8. 콘텐츠 마스터·수집층·비교그룹 (과업지시서 4, 17장)

**수집 구조 확인 결과(원자료 `수집기준`·`유튜브검색기준`·`인스타태그범위` 시트).**

| 구분 | 2023.1.1~10.30 (보완 수집, 2026-11-05 수집) | 2023.10.31~2026.10.30 (기본 수집, 2026-10-31 수집) |
|---|---|---|
| 인스타 | 시군별 같은 해시태그 목록. 태그마다 '최근 게시물'과 '인기 게시물(릴스 탭)'을 쪽수 상한까지 받음(태그별로 한쪽만 받은 경우 있음) | 같은 해시태그 목록의 '최근 게시물'을 쪽수 상한까지 받음 |
| 유튜브 | 검색어×연도 상위 50개, **조회수 순** 정렬 | 검색어×연도 상위 50개, **유튜브 검색 기본 순서** |
| 유튜브 '추가' | 시범 수집 때 더 받은 영상(2023~2025년에만 있음, 2026년 없음). 2023 보완 수집 여부와는 *다른* 표시 | 같음 |

- 구간은 *작성일* 기준으로 정의되어 있으므로, 2023-01-01~10-30 작성 콘텐츠는 행 단위로 보완 수집분임을 확인할 수 있습니다(`행별출처확인수준` = 구간 정의로 확인).
- 비교그룹ID는 매체×수집구간×정렬방식×(유튜브)기본/추가로 만듭니다. 연도 비교(트랙 B)는 `IG-기본`, `YT-기본순-기본`, `NEWS`만 씁니다.
- 수집시각, 표집확률, 팔로워·구독자 수는 원자료에 없어 **미확보**로 둡니다.

In [ ]:
def in_period(s: pd.Series, start, end) -> pd.Series:
    """날짜 Series가 [start, end] 구간(양 끝 포함, 종료일은 그날 23:59까지)에 드는지 판정합니다."""
    return (s >= pd.Timestamp(start)) & (s < pd.Timestamp(end) + pd.Timedelta(days=1))


SEASON_OF = {m: s for s, ms in CONFIG["SEASONS"].items() for m in ms}
DF = {"뉴스": news, "인스타": ig, "유튜브": yt}


def season_cols(dates: pd.Series) -> pd.DataFrame:
    """날짜 → 계절, 계절연도(겨울은 12월을 다음 해 겨울로: 2024-12 → 2025겨울)."""
    m = dates.dt.month
    season = m.map(SEASON_OF)
    sy = dates.dt.year + (m == 12).astype(int)
    return pd.DataFrame({"계절": season, "계절연도": sy.astype("Int64").astype(str) + season.fillna("")}, index=dates.index)


def compare_group(d: pd.DataFrame) -> pd.Series:
    """비교그룹ID: 매체×수집구간×정렬방식×기본/추가."""
    m = d["media"].iat[0]
    if m == "뉴스":
        return pd.Series("NEWS", index=d.index)
    if m == "인스타":
        return pd.Series(np.where(d["수집구간"].str.contains("보완"), "IG-보완", "IG-기본"), index=d.index)
    sort = np.where(d["수집구간"].str.contains("보완"), "YT-조회수순", "YT-기본순")
    return pd.Series(sort, index=d.index) + "-" + d["collect_mode"].fillna("불명확")


SELECT_RULE = {
    "NEWS": ("포털 검색 결과(6개 지역 언론사, 시군·광역 검색어)", "검색 결과 표시 순(원자료 기준)"),
    "IG-보완": ("시군별 해시태그 결과: 최근 게시물+인기 게시물(릴스 탭), 태그별 쪽수 상한", "태그별 최근순/인기순 혼합"),
    "IG-기본": ("시군별 해시태그 결과: 최근 게시물, 태그별 쪽수 상한", "최근순"),
    "YT-조회수순-기본": ("검색어×연도 상위 50개", "조회수 순"),
    "YT-조회수순-추가": ("시범 수집 추가분(보완 구간)", "조회수 순(시범)"),
    "YT-기본순-기본": ("검색어×연도 상위 50개", "유튜브 검색 기본 순서"),
    "YT-기본순-추가": ("시범 수집 추가분(2023.10.31~2025)", "유튜브 검색 기본 순서(시범)"),
}

# 출처 유형(과업지시서 17.7): 유튜브는 채널명·본문 규칙, 인스타는 본문 광고·협찬 표기와 원자료 검토유형만 근거로 사용
PROD_RULES = pd.read_csv(DICT_DIR / "생산주체규칙.csv").sort_values("우선순위")


def source_type(d: pd.DataFrame) -> tuple:
    """(출처유형, 출처유형근거) Series 쌍. 계정명만으로 협찬을 단정하지 않고 표기 문구를 근거로 남깁니다."""
    m = d["media"].iat[0]
    if m == "뉴스":
        return pd.Series("언론(지역 언론사)", index=d.index), pd.Series("뉴스 원자료(언론사 열)", index=d.index)
    typ = pd.Series("", index=d.index)
    why = pd.Series("", index=d.index)
    for _, r in PROD_RULES.iterrows():
        if r["대상필드"] == "채널명" and m != "유튜브":
            continue
        field = d["source"].fillna("") if r["대상필드"] == "채널명" else (d["title"].fillna("") + " " + d["body"].fillna(""))
        hit = field.str.extract("(" + r["패턴"] + ")", flags=re.I)[0]
        new = (typ == "") & hit.notna()
        typ[new] = r["주체"]
        why[new] = r["대상필드"] + "에 '" + hit[new].astype(str) + "'"
    if m == "인스타":
        biz = (typ == "") & d["검토유형"].str.contains("업체 영업", na=False)
        typ[biz] = "기업·점포 홍보(검토)"
        why[biz] = "원자료 검토유형: " + d.loc[biz, "검토유형"]
        typ[typ == ""] = "불명확(작성자 비식별)"
        why[why == ""] = "광고·협찬 표기 없음, 작성자 해시만 제공"
    else:
        typ[typ == ""] = "일반 이용자·크리에이터(추정)"
        why[why == ""] = "기관·언론·업체 표기 없음"
    return typ, why


for m, d in DF.items():
    sc = season_cols(d["date"])
    d["계절"], d["계절연도"] = sc["계절"], sc["계절연도"]
    d["비교그룹ID"] = compare_group(d)
    d["출처유형"], d["출처유형근거"] = source_type(d)
    d["in_common"] = in_period(d["date"], CONFIG["COMMON_START"], CONFIG["COMMON_END"])
print(pd.concat({m: d["비교그룹ID"].value_counts() for m, d in DF.items()}, axis=1).fillna(0).astype(int))
print(pd.concat({m: d["출처유형"].value_counts() for m, d in DF.items() if m != "뉴스"}, axis=1).fillna(0).astype(int))

In [ ]:
def reaction_states(d: pd.DataFrame) -> pd.DataFrame:
    """반응 지표별 값과 상태(유효/실제0/숨김/미제공/적용불가)를 만듭니다(과업지시서 13장 결측 처리).

    - 인스타 좋아요숨김=예 → 좋아요 '숨김'(원자료의 자리표시값은 버림)
    - 인스타 사진·캐러셀의 조회수 → '적용불가'(릴스만 제공)
    - 뉴스의 모든 반응 지표 → '적용불가'
    - 값이 비어 있으면 '미제공', 0이면 '실제0'(유효 관측에 포함)
    """
    m = d["media"].iat[0]
    out = pd.DataFrame(index=d.index)
    for k, col in (("조회수", "views"), ("좋아요", "likes"), ("댓글수", "comments")):
        if m == "뉴스":
            out[k] = np.nan
            out[k + "상태"] = "적용불가"
            continue
        v = d[col]
        st = np.where(v.isna(), "미제공", np.where(v == 0, "실제0", "유효"))
        st = pd.Series(st, index=d.index)
        if m == "인스타" and k == "좋아요":
            st[d["likes_hidden"]] = "숨김"
        if m == "인스타" and k == "조회수":
            st[d["type"] != "릴스"] = "적용불가"
            v = v.where(d["type"] == "릴스")
        out[k] = v
        out[k + "상태"] = st
    return out


for m, d in DF.items():
    rs = reaction_states(d)
    for c in rs.columns:
        d[c] = rs[c]
REACTION_STATE = pd.concat({m: pd.concat({k: d[k + "상태"].value_counts() for k in ("조회수", "좋아요", "댓글수")}, axis=1)
                            for m, d in DF.items() if m != "뉴스"}).fillna(0).astype(int)
REACTION_STATE

### 8.1 상대 반응 점수(과업지시서 14장)

- 비교 집단: 플랫폼×콘텐츠형식×게시연도·분기. 집단 안 유효 관측이 `PCTL_MIN_GROUP`(30)개 미만이면 플랫폼×형식×게시연도로 한 단계 넓히고, 그래도 부족하면 산출불가로 둡니다.
- 백분위: 중간순위 백분위 = (평균순위 − 0.5) ÷ N × 100 (`pandas.Series.rank(method="average")`, 동점은 평균 순위).
- 대표 상대반응점수: 유튜브(일반·숏츠·라이브)와 인스타 릴스는 조회수 백분위와 좋아요 백분위가 *모두* 있을 때 평균, 사진·캐러셀은 좋아요 백분위. 한쪽만 있으면 `조회수전용`·`좋아요전용`으로 점수유형을 따로 둡니다. 댓글 백분위는 보조 지표입니다.
- 이 점수는 같은 플랫폼·형식·시기 안의 상대 위치일 뿐이며, 2023 보완분처럼 조회수 순으로 뽑힌 표본의 선택 편향을 바로잡지 못합니다.

In [ ]:
def mid_rank_pct(v: pd.Series) -> pd.Series:
    """유효값만으로 중간순위 백분위(0~100)를 계산합니다."""
    r = v.rank(method="average")
    return (r - 0.5) / v.notna().sum() * 100


def reaction_scores(d: pd.DataFrame) -> pd.DataFrame:
    """콘텐츠별 비교집단, 지표별 백분위, 대표반응점수와 점수유형을 계산합니다(SNS만)."""
    q = d["date"].dt.year.astype("Int64").astype(str) + "Q" + d["date"].dt.quarter.astype("Int64").astype(str)
    yr = d["date"].dt.year.astype("Int64").astype(str)
    g1 = d["media"] + "|" + d["type"].astype(str) + "|" + q
    g2 = d["media"] + "|" + d["type"].astype(str) + "|" + yr
    out = pd.DataFrame(index=d.index)
    expanded = pd.Series(False, index=d.index)
    gid = pd.Series("", index=d.index)
    gn = pd.Series(np.nan, index=d.index)
    for k in ("조회수", "좋아요", "댓글수"):
        v = d[k].where(d[k + "상태"].isin(["유효", "실제0"]))
        n1 = v.groupby(g1).transform("count")
        n2 = v.groupby(g2).transform("count")
        p1 = v.groupby(g1).transform(mid_rank_pct)
        p2 = v.groupby(g2).transform(mid_rank_pct)
        pct = np.where(n1 >= CONFIG["PCTL_MIN_GROUP"], p1, np.where(n2 >= CONFIG["PCTL_MIN_GROUP"], p2, np.nan))
        out[k + "백분위"] = np.where(v.notna(), pct, np.nan)
        if k == "좋아요":
            gid = pd.Series(np.where(n1 >= CONFIG["PCTL_MIN_GROUP"], g1, np.where(n2 >= CONFIG["PCTL_MIN_GROUP"], g2, "산출불가")), index=d.index)
            gn = pd.Series(np.where(n1 >= CONFIG["PCTL_MIN_GROUP"], n1, np.where(n2 >= CONFIG["PCTL_MIN_GROUP"], n2, np.nan)), index=d.index)
        expanded |= v.notna() & (n1 < CONFIG["PCTL_MIN_GROUP"]) & (n2 >= CONFIG["PCTL_MIN_GROUP"])
    vp, lp = out["조회수백분위"], out["좋아요백분위"]
    view_based = (d["media"] == "유튜브") | (d["type"] == "릴스")
    score = np.where(view_based & vp.notna() & lp.notna(), (vp + lp) / 2,
             np.where(view_based & vp.notna(), vp, np.where(lp.notna(), lp, np.nan)))
    typ = np.where(view_based & vp.notna() & lp.notna(), "조회수·좋아요평균",
           np.where(view_based & vp.notna(), "조회수전용",
           np.where(view_based & lp.notna(), "좋아요전용", np.where(lp.notna(), "좋아요기본", "산출불가"))))
    out["대표반응점수"] = score
    out["반응점수유형"] = typ
    out["반응비교집단ID"] = gid
    out["비교집단유효N"] = gn
    out["비교집단확장여부"] = np.where(expanded, "확장(게시연도)", "")
    # 작성자 평소 반응 대비(같은 형식 10건 이상인 출처만): 콘텐츠 점수 − 출처×형식 중앙값
    key = d["source"].astype(str) + "|" + d["type"].astype(str)
    cnt = out["대표반응점수"].groupby(key).transform("count")
    med = out["대표반응점수"].groupby(key).transform("median")
    out["기존출처평소반응비교"] = np.where(cnt >= CONFIG["AUTHOR_MIN_SAME_FORMAT"], out["대표반응점수"] - med, np.nan)
    return out


for m in ("인스타", "유튜브"):
    sc = reaction_scores(DF[m])
    for c in sc.columns:
        DF[m][c] = sc[c]
for c in ("조회수백분위", "좋아요백분위", "댓글수백분위", "대표반응점수", "비교집단유효N", "기존출처평소반응비교"):
    news[c] = np.nan
news["반응점수유형"], news["반응비교집단ID"], news["비교집단확장여부"] = "적용불가", "적용불가", ""
pd.concat({m: DF[m]["반응점수유형"].value_counts() for m in ("인스타", "유튜브")}, axis=1).fillna(0).astype(int)

In [ ]:
def content_master(m: str) -> pd.DataFrame:
    """03_콘텐츠마스터(과업지시서 부록 열 명세). 긴 본문은 원자료 시트·원본ID로 참조하고 여기에는 넣지 않습니다."""
    d = DF[m]
    sheet = {"뉴스": "(CSV)", "인스타": "인스타", "유튜브": "유튜브"}[m]
    rel_note = d["relevance"].map({"A": "본문·위치 지역 단서", "B": "수집태그 외 태그에 지역 단서", "C": "수집태그 외 지역 단서 없음"})
    decision = np.where(d["valid"], "포함", "제외")
    why = np.where(d["date"].isna(), "날짜 결측",
          np.where(~d["valid"], "지역 관련성 " + d["relevance"] + "(" + rel_note.fillna("") + ")", "지역 관련성 " + d["relevance"]))
    grp = compare_group(d)
    sel = grp.map(lambda g: SELECT_RULE.get(g, ("불명확", "불명확")))
    tl = d["timeline"].ne("") if "timeline" in d else pd.Series(False, index=d.index)
    title = d["title"].fillna("")
    out = pd.DataFrame({
        "원본ID": d["원본ID"], "콘텐츠ID": d["content_id"], "매체": m,
        "입력파일": RAW_NEWS.name if m == "뉴스" else RAW_SNS.name, "시트": sheet,
        "논리레코드번호": d["논리레코드번호"] if m == "뉴스" else np.nan,
        "원본필드해시": d["원본필드해시"] if m == "뉴스" else "",
        "원본게시물영상ID": "" if m == "뉴스" else d["content_id"],
        "링크_제공값": d["url"].fillna(""),
        "날짜": d["date"].dt.strftime("%Y-%m-%d"),
        "날짜출처": {"뉴스": "검색표시날짜(실제 발행일 확인 불가)", "인스타": "작성일(KST)", "유튜브": "게시일(KST)"}[m],
        "연도": d["year"], "월": d["month"], "계절연도": d["계절연도"], "계절": d["계절"],
        "형식": d["type"], "라이브여부": np.where(d["type"] == "라이브", "예", "아니오") if m == "유튜브" else "",
        "길이초": d["length_sec"] if m == "유튜브" else np.nan,
        "출처ID": d["source"].astype(str), "출처표시명": d["source"].astype(str) if m != "인스타" else "(작성자 해시)",
        "출처유형": d["출처유형"], "출처유형근거": d["출처유형근거"],
        "원본시군": d["원본시군"], "원본관련시군": d["원본관련시군"],
        "수집후보시군": d["sigun_all"].map(lambda L: ",".join(L)),
        "수집층": d["수집층"],
        "제목": title.str[:200], "제목절단여부": np.where(title.str.len() > 200, "절단(원본ID로 원문 참조)", ""),
        "본문미리보기설명_원장참조": d["원본ID"],
        "타임라인연결여부": np.where(tl, "예", "아니오") if m == "유튜브" else "",
        "중복군ID": d["rep_key"].astype(str),
        "분석포함결정": decision, "판정근거": why,
        "기존판정원장참조": ("통합재판정=" + d["review_flag"].astype(str)) if m == "뉴스" else d["검토유형"].fillna(""),
        "조회수": d["조회수"], "조회수상태": d["조회수상태"], "좋아요": d["좋아요"], "좋아요상태": d["좋아요상태"],
        "댓글수": d["댓글수"], "댓글수상태": d["댓글수상태"],
        "수집시각": "미확보(구간별 수집일만 확인)" if m != "뉴스" else "미확보",
        "반응비교집단ID": d["반응비교집단ID"], "비교집단유효N": d["비교집단유효N"],
        "조회수백분위": d["조회수백분위"].round(2), "좋아요백분위": d["좋아요백분위"].round(2), "댓글수백분위": d["댓글수백분위"].round(2),
        "대표반응점수": d["대표반응점수"].round(2), "반응점수유형": d["반응점수유형"], "비교집단확장여부": d["비교집단확장여부"],
        "기존출처평소반응비교": d["기존출처평소반응비교"].round(2),
        "검토상태": np.where(d["검토등급"].isin(["B", "C"]), "검토(원자료 검토등급 " + d["검토등급"] + ")", "자동 판정"),
        "수집회차ID": np.where(d["수집구간"].str.contains("보완"), "SNS-R1(2026-11-05 수집)",
                       np.where(d["수집구간"].str.contains("기본"), "SNS-R2(2026-10-31 수집)", "NEWS-R1")),
        "수집구간": d["수집구간"],
        "보완대상기간여부": np.where(d["수집구간"].str.contains("보완"), "예", "아니오") if m != "뉴스" else "해당없음",
        "실제보완수집여부": np.where(d["수집구간"].str.contains("보완"), "보완", "기본") if m != "뉴스" else "해당없음",
        "선택방식": sel.map(lambda x: x[0]), "정렬기준": sel.map(lambda x: x[1]),
        "검색태그세트ID": {"뉴스": "NEWSQ-v1(시군·광역 검색어)", "인스타": "IGTAG-v1(시군별 동일 태그 목록)", "유튜브": "YTQ-v1(시군×7개 검색어+전북 3개)"}[m],
        "행별출처확인수준": "구간 정의(작성일 기준)로 확인" if m != "뉴스" else "단일 수집",
        "비교그룹ID": grp,
    })
    return out


MASTER = {m: content_master(m) for m in MEDIA}
print({m: len(v) for m, v in MASTER.items()})
MASTER["유튜브"].head(3).T.head(40)

In [ ]:
def collection_layer_table() -> pd.DataFrame:
    """27_수집층비교가능성: 매체×연도×월×시군×형식×수집층 단위 수록량과 비교 가능 판정."""
    rows = []
    for m in MEDIA:
        d = DF[m]
        x = d.assign(시군=d["원본시군"].replace("", "불명확"), 비교그룹=compare_group(d))
        g = x.groupby(["year", "month", "시군", "type", "수집층", "비교그룹"], dropna=False).agg(
            원자료수=("content_id", "size"), 유효콘텐츠수=("valid", "sum")).reset_index()
        g["매체"] = m
        rows.append(g)
    t = pd.concat(rows, ignore_index=True)

    def judge(r):
        grp = r["비교그룹"]
        if grp == "NEWS":
            return "가능", "단일 수집 방식(언론사 6곳). 2023 보완 수집 해당 없음", "NEWS"
        if grp in ("IG-보완", "YT-조회수순-기본", "YT-조회수순-추가"):
            return "불가", "2023.1.1~10.30 보완 수집: 선택 방식(인기·조회수 순)이 기본 구간과 다름 → 트랙 A(발굴)·C(민감도)만 사용", "IG-기본" if grp.startswith("IG") else "YT-기본순-기본"
        if grp == "YT-기본순-추가":
            return "불가", "시범 수집 추가분: 2026년에 없음 → 연도 비교에서 제외", "YT-기본순-기본"
        if grp == "IG-기본":
            return "조건부", "같은 태그 목록·최근 게시물 방식이나 2026년 수록량이 2024~2025년의 4배 이상(태그별 쪽수 상한·최근 편중) → 비중으로만 비교", "IG-기본"
        return "조건부", "검색어×연도 상위 50개 구조(연도별 수록량 상한) → 비중으로만 비교", "YT-기본순-기본"
    j = t.apply(judge, axis=1, result_type="expand")
    t["비교가능판정"], t["판정사유"], t["비교대상층"] = j[0], j[1], j[2]
    t["동기간시작"] = t["year"].map(lambda y: f"{int(y)}-{CONFIG['YEAR_WINDOW'][0]}" if pd.notna(y) else "")
    t["동기간종료"] = t["year"].map(lambda y: f"{int(y)}-{CONFIG['YEAR_WINDOW'][1]}" if pd.notna(y) else "")
    t["수집회차ID"] = np.where(t["수집층"].str.contains("보완"), "SNS-R1(2026-11-05 수집)",
                         np.where(t["매체"] == "뉴스", "NEWS-R1", "SNS-R2(2026-10-31 수집)"))
    t["수집구간"] = np.where(t["수집층"].str.contains("보완"), "2023.1.1~10.30", np.where(t["매체"] == "뉴스", "전체", "2023.10.31~2026.10.30"))
    t["검색태그세트ID"] = t["매체"].map({"뉴스": "NEWSQ-v1", "인스타": "IGTAG-v1", "유튜브": "YTQ-v1"})
    t["태그구성변경"] = np.where(t["매체"] == "인스타", "두 구간 태그 목록 동일(수집기준 시트), 태그별 받은 쪽수는 다름", "없음(같은 검색어 목록)")
    t["선택방식"] = t["비교그룹"].map(lambda g: SELECT_RULE.get(g, ("", ""))[0])
    t["정렬기준"] = t["비교그룹"].map(lambda g: SELECT_RULE.get(g, ("", ""))[1])
    t["실제보완수집여부"] = np.where(t["수집층"].str.contains("보완"), "보완", np.where(t["매체"] == "뉴스", "해당없음", "기본"))
    t["행별출처확인수준"] = np.where(t["매체"] == "뉴스", "단일 수집", "구간 정의(작성일 기준)로 확인")
    t["근거파일시트"] = np.where(t["매체"] == "뉴스", RAW_NEWS.name, RAW_SNS.name + " : 수집기준/유튜브검색기준/인스타태그범위")
    t["확인주체"] = "분석 코드(자동)"
    t["해석제약"] = np.where(t["비교가능판정"] == "가능", "", "수집 비중 정규화만으로 선택 편향이 제거되지 않음")
    t = t.rename(columns={"year": "연도", "month": "월", "type": "형식", "비교그룹": "비교그룹ID"})
    t.insert(0, "점검ID", [f"L{i:06d}" for i in range(1, len(t) + 1)])
    cols = ["점검ID", "매체", "연도", "월", "시군", "형식", "수집회차ID", "수집층", "수집구간", "검색태그세트ID", "태그구성변경", "선택방식",
            "정렬기준", "원자료수", "유효콘텐츠수", "실제보완수집여부", "행별출처확인수준", "비교그룹ID", "비교대상층", "동기간시작", "동기간종료",
            "비교가능판정", "판정사유", "근거파일시트", "확인주체", "해석제약"]
    return t[cols]


LAYER27 = collection_layer_table()
LAYER27.groupby(["매체", "비교그룹ID", "비교가능판정"])["원자료수"].sum()

## 9. 근거 원장과 시군 귀속 (과업지시서 8~10장)

**9.1 키워드 근거.** 콘텐츠×관심대상마다 *원본 필드의 실제 부분 문자열*을 인용으로 남깁니다. 뉴스는 원본 `기사제목`·`미리보기`, 인스타는 `본문`·`해시태그`·`위치`, 유튜브는 `제목`·`설명`·`태그`·`타임라인(설명란)`·`타임라인(댓글)`에서 원어를 찾습니다(공백·#·가운뎃점 차이는 허용). 찾지 못하면 `원문일치검사=미발견`으로 표시하고 집계에서 빼지 않되 검토 목록에 올립니다. 문자열 출현 횟수(TF)는 보조 지표로 함께 셉니다. LLM은 쓰지 않았습니다.

**9.2 시군 귀속.** 수집 당시 시군(검색어·수집태그)은 *후보*로만 보존하고, 다음 순서로 최종 귀속합니다.

| 순위 | 근거 | 귀속확실성 | 연결등급 |
|---|---|---|---|
| 1 | 대상 이름에 시군명·대표 지명이 들어 있음(전주비빔밥 → 전주시, 마이산 → 진안군) | 높음 | A |
| 2 | 대상이 등장한 전체 콘텐츠(세 매체)에서 본문 지명의 70% 이상이 한 시군(5건 이상) → 대상 소재 시군(추정) | 중(분포 추정) | 본문에도 같은 시군이 있으면 A/B, 없으면 B |
| 3 | 이 콘텐츠 본문·제목·태그·위치·타임라인에 나온 시군 | 1개면 중, 여러 개면 낮음(다중귀속) | 인용 앞뒤 40자 안에 지명이 있으면 A, 같은 콘텐츠의 다른 곳이면 B, 여러 시군 중 근접 지명이 없으면 C(동반 출현) |
| 4 | 시군명 없이 '전북·전라북도'만 나옴 | 전북전체 | B |
| 5 | 지역 단서가 수집태그·검색어뿐 | 지역미확정 | C |

장수(오래 삶)·완주(끝까지 달림) 같은 동음이의어는 '장수군·장계·장수사과', '완주군·삼례'처럼 그 시군을 가리키는 말만 지명으로 인정합니다(원자료 수집 기준과 동일). 2차 검토에서 '무주상보시(불교 용어)·무주택', '지평선(일반어)', '청보리(타 지역 재배)', '선유도공원(서울)', '춘향전(고전 작품)', '군산복합체'도 지명에서 뺐습니다. 또 대상 바로 앞에 '전북·전라북도'가 붙어 있고(예: '전북 한지') 콘텐츠의 시군 지명이 대상과 떨어져 있으면, 그 시군이 아니라 **전북전체**로 귀속합니다(본문에 장인 거주지로 '임실'이 나온 '전북 한지' 기사를 임실군으로 잡던 오류 수정). 시군별 핵심 순위는 A/B 연결만으로 만들고, C는 별도 열로 제공합니다. 다중귀속 콘텐츠는 각 시군에 1번씩 들어가므로 *시군별 값을 더해 도 전체 합계로 쓰지 않습니다*.

In [ ]:
SIGUN_CUES = {
    "전주시": r"전주(?!곡)|jeonju|한옥마을|남부시장|덕진공원|전동성당|경기전|오목대|풍남문|에코시티|자만벽화",
    "군산시": r"군산(?!복합체)|gunsan|고군산|선유도(?!공원|역)|은파호수|경암동|철길마을|신시도|비응항|이성당|월명공원|초원사진관",
    "익산시": r"익산|iksan|미륵사|왕궁리|서동공원|원광대|보석박물관",
    "정읍시": r"정읍|jeongeup|내장산|내장사|쌍화차거리|구절초(축제|꽃축제|테마공원)",
    "남원시": r"남원|namwon|광한루|춘향(?!전)|바래봉|뱀사골|요천",
    "김제시": r"김제|gimje|지평선\s?(축제|쌀|산단|산업단지|농협|학교|중|고)|벽골제|금산사|망해사|심포항",
    "완주군": r"완주군|wanju|완주\s?(로컬|와일드|산업단지|수소|테크노|군청|군수|읍|면)|삼례|봉동읍|이서면|고산면|소양면|운주면|대둔산|오성한옥마을|위봉",
    "진안군": r"진안|jinan|마이산|용담호|운일암",
    "무주군": r"무주(?!택|상보시|상|공처)|muju|덕유산|구천동|적상산|태권도원|반디랜드",
    "장수군": r"장수군|장계면?|장수\s?(사과|한우|누리|트레일|읍|군청|군수|레드푸드|쿨밸리)|장안산|논개(사당|생가|활공장)|번암|천천면|산서면",
    "임실군": r"임실|imsil|오수면|오수의견|사선대",
    "순창군": r"순창|sunchang|강천산|채계산",
    "고창군": r"고창|gochang|선운사|청보리\s?(밭|축제)|고인돌박물관|구시포|상하농원|학원농장",
    "부안군": r"부안|buan|변산|채석강|내소사|격포|곰소",
}
_PROV_CUE = r"전북|전라북도|jeonbuk|jeollabuk"
REGION_RE = re.compile("|".join(f"(?P<g{i}>{p})" for i, p in enumerate(SIGUN_CUES.values())) + f"|(?P<prov>{_PROV_CUE})", re.I)
_GROUP_SIGUN = {f"g{i}": s for i, s in enumerate(SIGUN_CUES)}


def regions_in(text: str) -> tuple:
    """문자열 안의 (시군 집합, 전북 언급 여부)."""
    sg, prov = set(), False
    for mm in REGION_RE.finditer(text or ""):
        k = mm.lastgroup
        if k == "prov":
            prov = True
        else:
            sg.add(_GROUP_SIGUN[k])
    return sg, prov


# 원본 필드(인용은 이 문자열의 실제 부분 문자열이어야 함)
_tl = sns_raw["유튜브_타임라인"].copy()
_tl["출처"] = _tl["출처"].fillna("")
_tl["장소명·텍스트"] = _tl["장소명·텍스트"].fillna("").astype(str)
TL_ITEMS = {k: list(zip(g["장소명·텍스트"], g["시간"].fillna(""), g["출처"])) for k, g in _tl.groupby("영상ID")}
RAW_FIELDS = {
    "뉴스": [("기사제목", news_raw["기사제목"]), ("미리보기", news_raw["미리보기"])],
    "인스타": [("본문", sns_raw["인스타"]["본문"]), ("해시태그", sns_raw["인스타"]["해시태그"]), ("위치", sns_raw["인스타"]["위치"])],
    "유튜브": [("제목", sns_raw["유튜브"]["제목"]), ("설명", sns_raw["유튜브"]["설명"]), ("태그", sns_raw["유튜브"]["태그"]),
              ("타임라인(설명란)", sns_raw["유튜브"]["영상ID"].map(lambda v: " | ".join(t for t, _, s in TL_ITEMS.get(v, []) if s != "댓글"))),
              ("타임라인(댓글)", sns_raw["유튜브"]["영상ID"].map(lambda v: " | ".join(t for t, _, s in TL_ITEMS.get(v, []) if s == "댓글")))],
}
FIELD_SEP = "\x1e"


def content_text_index(m: str) -> dict:
    """콘텐츠ID → (이어 붙인 원문, [(필드명, 시작, 끝)], 시군 집합, 전북 언급)."""
    d = DF[m]
    names = [n for n, _ in RAW_FIELDS[m]]
    cols = [s.fillna("").astype(str).values for _, s in RAW_FIELDS[m]]
    out = {}
    for i, cid in enumerate(d["content_id"].values):
        if not d["valid"].iat[i]:
            continue
        parts, offs, pos = [], [], 0
        for n, c in zip(names, cols):
            t = c[i]
            parts.append(t)
            offs.append((n, pos, pos + len(t)))
            pos += len(t) + 1
        full = FIELD_SEP.join(parts)
        sg, prov = regions_in(full)
        out[cid] = (full, offs, sg, prov)
    return out


t0 = time.time()
TEXT_IDX = {m: content_text_index(m) for m in MEDIA}
print(f"원문 색인 {time.time()-t0:.0f}초:", {m: len(v) for m, v in TEXT_IDX.items()})
for m in MEDIA:
    has = np.mean([bool(v[2]) for v in TEXT_IDX[m].values()])
    prov = np.mean([(not v[2]) and v[3] for v in TEXT_IDX[m].values()])
    print(f"  {m}: 본문에 시군 지명 있음 {has:.1%}, 전북만 언급 {prov:.1%}")

In [ ]:
_SEP_CLASS = r"[\s#·\-_.,'’/]*"
_PROV_PREFIX = re.compile(r"(전북|전라북도|전북특별자치도|전북도)\s?#?$")


def evidence_pass(m: str) -> pd.DataFrame:
    """콘텐츠×관심대상마다 원문 인용·근거 필드·위치·TF·근접 지명을 찾습니다.

    원어(raw)별로 정규식을 한 번만 만들고, 그 원어가 나온 콘텐츠들에서 검색합니다(메모리 절약).
    인용은 일치 위치 앞뒤 25자(필드 경계 안)이며, 타임라인은 일치한 항목 문자열 전체를 인용합니다.
    """
    c = CT[m][["content_id", "target", "raw", "src"]].reset_index(drop=True)
    idx = TEXT_IDX[m]
    n = len(c)
    field = np.full(n, "", dtype=object); quote = np.full(n, "", dtype=object); spos = np.full(n, -1)
    tf = np.zeros(n, dtype=np.int32); near = np.full(n, "", dtype=object); tl_time = np.full(n, "", dtype=object)
    tl_seg = np.zeros(n, dtype=np.int16)
    provp = np.zeros(n, dtype=bool)
    keys = c["raw"].map(lambda r: re.sub(r"[^0-9a-z가-힣]", "", unicodedata.normalize("NFKC", str(r)).lower()))
    for key, rows in c.groupby(keys).indices.items():
        if not key:
            continue
        pat = re.compile(_SEP_CLASS.join(map(re.escape, key)), re.I)
        for i in rows:
            info = idx.get(c.at[i, "content_id"])
            if info is None:
                continue
            full, offs, _, _ = info
            hits = list(pat.finditer(full))
            if not hits:
                continue
            h = hits[0]
            tf[i] = len(hits)
            for fname, a, b in offs:
                if a <= h.start() < b:
                    field[i] = fname
                    spos[i] = h.start() - a
                    if fname.startswith("타임라인"):
                        items = [it for it in TL_ITEMS.get(c.at[i, "content_id"], []) if (it[2] == "댓글") == (fname == "타임라인(댓글)")]
                        segs = [it for it in items if pat.search(it[0])]
                        tl_seg[i] = len(segs)
                        if segs:
                            quote[i], tl_time[i] = segs[0][0][:120], str(segs[0][1])
                    else:
                        qa, qb = max(a, h.start() - 25), min(b, h.end() + 25)
                        quote[i] = full[qa:qb]
                    wa, wb = max(a, h.start() - 40), min(b, h.end() + 40)
                    sg, _ = regions_in(full[wa:wb])
                    near[i] = ",".join(sorted(sg))
                    provp[i] = bool(_PROV_PREFIX.search(full[max(a, h.start() - 10):h.start()]))
                    break
    c["근거필드"], c["원문인용"], c["필드내위치"], c["문자열출현횟수"] = field, quote, spos, tf
    c["근접시군"], c["시간표시"], c["타임라인구간수"] = near, tl_time, tl_seg
    c["광역접두"] = provp
    c["원문일치검사"] = np.where(c["근거필드"] != "", "일치", "미발견")
    return c


EVID = {}
for m in MEDIA:
    t0 = time.time()
    EVID[m] = evidence_pass(m)
    print(f"{m}: {len(EVID[m]):,}행, 원문 일치 {(EVID[m]['원문일치검사'] == '일치').mean():.2%} ({time.time()-t0:.0f}초)")
display(EVID["유튜브"].groupby("근거필드").size())

In [ ]:
def target_home_sigun() -> pd.DataFrame:
    """대상별 소재 시군: ① 대상 이름의 지명, ② 세 매체 본문 지명 분포(70% 이상·5건 이상)."""
    rows = []
    for m in MEDIA:
        idx = TEXT_IDX[m]
        e = EVID[m][["content_id", "target"]]
        sg = e["content_id"].map(lambda c: idx[c][2] if c in idx else set())
        x = pd.DataFrame({"target": e["target"], "sg": sg})
        x = x[x["sg"].map(len) > 0].explode("sg")
        rows.append(x)
    x = pd.concat(rows)
    dist = x.groupby(["target", "sg"]).size().rename("n").reset_index()
    tot = x.drop_duplicates().groupby("target").size()
    top = dist.sort_values("n", ascending=False).drop_duplicates("target").set_index("target")
    all_t = set().union(*[set(CT[m]["target"]) for m in MEDIA])
    out = []
    for t in all_t:
        sg_name, _ = regions_in(t)
        out.append((t, ",".join(sorted(sg_name)), "대상명에 지명 포함" if sg_name else "", "높음" if sg_name else ""))
    home = pd.DataFrame(out, columns=["target", "대상소재시군", "소재근거", "소재확실성"]).set_index("target")
    # 분포 추정(이름에 지명이 없는 대상만)
    share = (top["n"] / dist.groupby("target")["n"].sum()).rename("share")
    cnt = dist.groupby("target")["n"].sum()
    cand = pd.DataFrame({"sg": top["sg"], "share": share, "cnt": cnt}).dropna()
    cand = cand[(cand["share"] >= 0.7) & (cand["cnt"] >= 5)]
    fill = home.index.isin(cand.index) & (home["대상소재시군"] == "")
    home.loc[fill, "대상소재시군"] = cand.loc[home.index[fill], "sg"].values
    home.loc[fill, "소재근거"] = "본문 지명 분포(" + (cand.loc[home.index[fill], "share"] * 100).round(0).astype(int).astype(str).values + "%, " \
                               + cand.loc[home.index[fill], "cnt"].astype(int).astype(str).values + "건)"
    home.loc[fill, "소재확실성"] = "중(분포 추정)"
    return home


t0 = time.time()
HOME = target_home_sigun()
print(f"대상 소재 시군 판정 {time.time()-t0:.0f}초:", HOME["소재확실성"].replace("", "미확정").value_counts().to_dict())

In [ ]:
def attribute_regions(m: str) -> pd.DataFrame:
    """콘텐츠×관심대상×최종귀속시군 관계 원장(시군 귀속·연결등급·확실성)."""
    e = EVID[m]
    idx = TEXT_IDX[m]
    home_sg = e["target"].map(HOME["대상소재시군"]).fillna("")
    home_why = e["target"].map(HOME["소재확실성"]).fillna("")
    out_rows = []
    cids, tgts, nears = e["content_id"].values, e["target"].values, e["근접시군"].values
    provp = e["광역접두"].values
    for i in range(len(e)):
        info = idx.get(cids[i])
        csg, prov = (info[2], info[3]) if info else (set(), False)
        near = set(nears[i].split(",")) - {""}
        hs = home_sg.iat[i]
        if hs:
            certainty = "높음" if home_why.iat[i] == "높음" else "중(분포 추정)"
            for s in hs.split(","):
                grade = "A" if (home_why.iat[i] == "높음" or s in near) else "B"
                out_rows.append((i, s, grade, certainty, "대상 소재 시군", len(hs.split(",")) > 1))
            continue
        if csg and provp[i] and not (csg & near):
            out_rows.append((i, "전북전체", "B", "전북전체(대상 앞 '전북' 명시)", "대상 앞 광역 지명", False))
            continue
        if csg:
            multi = len(csg) > 1
            for s in sorted(csg):
                if not multi:
                    grade, cert = ("A" if s in near else "B"), "중"
                else:
                    grade, cert = ("A" if s in near else "C"), "낮음(다중)"
                out_rows.append((i, s, grade, cert, "콘텐츠 본문 지명", multi))
            continue
        if prov:
            out_rows.append((i, "전북전체", "B", "전북전체(시군 특정 불가)", "광역 지명만 언급", False))
        else:
            out_rows.append((i, "지역미확정", "C", "지역미확정", "수집태그·검색어만", False))
    r = pd.DataFrame(out_rows, columns=["row", "최종귀속시군", "연결등급", "귀속확실성", "귀속근거", "다중귀속여부"])
    base = e.reset_index(drop=True)
    r = r.join(base[["content_id", "target", "근거필드", "원문인용", "필드내위치", "근접시군"]], on="row")
    r["대상소재시군"] = r["target"].map(HOME["대상소재시군"]).fillna("")
    r["콘텐츠근거시군"] = r["content_id"].map(lambda c: ",".join(sorted(idx[c][2])) if c in idx else "")
    return r.drop(columns="row")


REL = {}
for m in MEDIA:
    t0 = time.time()
    REL[m] = attribute_regions(m)
    vc = REL[m]["연결등급"].value_counts(normalize=True).round(3).to_dict()
    print(f"{m}: 관계 {len(REL[m]):,}행, 등급 {vc} ({time.time()-t0:.0f}초)")
pd.concat({m: REL[m]["최종귀속시군"].value_counts() for m in MEDIA}, axis=1).fillna(0).astype(int)

In [ ]:
# 콘텐츠 단위 지역 집합(분모용): 본문 지명 ∪ 그 콘텐츠 대상들의 A/B 귀속 시군. 지역 단서가 없으면 '지역미확정'
CONTENT_REGION = {}
for m in MEDIA:
    idx = TEXT_IDX[m]
    ab = REL[m][REL[m]["연결등급"].isin(["A", "B"])].groupby("content_id")["최종귀속시군"].agg(set)
    reg = {}
    for cid, (full, offs, sg, prov) in idx.items():
        s = set(sg) | ab.get(cid, set())
        if not s:
            s = {"전북전체"} if prov else {"지역미확정"}
        reg[cid] = s
    CONTENT_REGION[m] = pd.Series(reg)
REGIONS = SIGUN + ["전북전체", "지역미확정"]
pd.concat({m: CONTENT_REGION[m].explode().value_counts().reindex(REGIONS) for m in MEDIA}, axis=1)

In [ ]:
# 관심 맥락(경험·메뉴·정책·쟁점): 같은 콘텐츠의 보조어(맛집·카페·축제·예산·유치 …)를 맥락 표지로 사용
CTX = {}
for m in MEDIA:
    a = AUX[m][AUX[m]["judge"] == "보조어"]
    freq = a["target"].value_counts()
    top_ctx = set(freq.head(300).index)
    a = a[a["target"].isin(top_ctx)]
    a = a.assign(_r=a["target"].map(freq)).sort_values("_r", ascending=False)
    CTX[m] = a.groupby("content_id")["target"].agg(lambda s: list(dict.fromkeys(s))[:3])


def target_context(m: str, targets, k: int = 3) -> pd.Series:
    """대상별로 함께 나온 보조어 상위 k개(관심맥락)."""
    c = CT[m][CT[m]["target"].isin(set(targets))][["content_id", "target"]]
    c = c.assign(ctx=c["content_id"].map(CTX[m])).dropna(subset=["ctx"]).explode("ctx")
    return c.groupby("target")["ctx"].agg(lambda s: ", ".join(s.value_counts().head(k).index))

## 10. 공통 집계 엔진 (과업지시서 12, 18장)

모든 집계표는 같은 함수 `aggregate()`로 만듭니다. 집계 층 g(매체·지역·분야·기간·수집층)마다
- **N_g(유효콘텐츠분모)** = 그 층의 유효 고유 콘텐츠 수, **n_kg(등장콘텐츠수)** = 대상 k가 등장한 고유 콘텐츠 수
- **등장비중** = n/N, **천건당언급** = n/N×1000, N=0이면 '산출불가'
- **분야내콘텐츠분모** = 그 층에서 같은 주분야 대상이 하나 이상 나온 콘텐츠 수, 분야내등장비중 = n/분야내분모
- **반복군보정등장수** = 뉴스 기사군 수 / 인스타 동일 본문 묶음·출처 반복 생산을 1건으로 본 수 / 유튜브 영상 수(반복 생산 표시분은 출처당 1건)
- **고유출처수**(언론사·작성자·채널), **최대출처점유율** = 가장 많이 올린 출처의 콘텐츠 수 ÷ n
- **반응**: 대상 포함 콘텐츠의 조회수·좋아요·댓글 합계·중앙값·최댓값·유효N·결측수(적용불가 제외), 상위 1개 콘텐츠 기여율, 대표반응점수 중앙값·유효N·주된 점수유형

한 콘텐츠의 반응은 그 콘텐츠에 나온 모든 대상에 연결되므로, 대상별 반응 합계를 더해 전체 조회수나 고유 이용자 수라고 하지 않습니다(**대상 포함 콘텐츠의 연관 반응량**).

In [ ]:
import hashlib
TID = {}


def target_id(t: str) -> str:
    """표준관심대상 → 대상ID(이름 해시, 실행마다 같음)."""
    if t not in TID:
        TID[t] = "T" + hashlib.md5(t.encode("utf-8")).hexdigest()[:10]
    return TID[t]


# 콘텐츠 단위 표(분모·반응 계산용)
CONT = {}
for m in MEDIA:
    d = DF[m][DF[m]["valid"]].copy()
    d["region"] = d["content_id"].map(CONTENT_REGION[m])
    keep = ["content_id", "rep_key", "date", "year", "month", "계절", "계절연도", "type", "source", "비교그룹ID", "출처유형",
            "조회수", "조회수상태", "좋아요", "좋아요상태", "댓글수", "댓글수상태", "대표반응점수", "반응점수유형", "region", "수집구간"]
    CONT[m] = d[keep].reset_index(drop=True)

# 콘텐츠×대상 원장에 분야·TF·근거필드를 붙임(EVID는 CT와 같은 행 순서)
for m in MEDIA:
    assert len(EVID[m]) == len(CT[m]) and (EVID[m]["content_id"].values == CT[m]["content_id"].values).all()
    CT[m]["tf"] = EVID[m]["문자열출현횟수"].values
    CT[m]["근거필드"] = EVID[m]["근거필드"].values
    CT[m]["area"] = CT[m]["target"].map(AREA_MAP).fillna("미분류(검토필요)")
    CT[m]["비교그룹ID"] = CT[m]["content_id"].map(CONT[m].set_index("content_id")["비교그룹ID"])

REACT_COLS = ["조회수", "조회수상태", "좋아요", "좋아요상태", "댓글수", "댓글수상태", "대표반응점수", "반응점수유형"]


def aggregate(m: str, rows: pd.DataFrame, cont: pd.DataFrame, keys: list) -> pd.DataFrame:
    """공통 빈도·반응 지표를 계산합니다.

    Parameters
    ----------
    rows : 콘텐츠×대상 행(content_id, target, rep_key, source, month, area, tf + keys 열)
    cont : 같은 층의 콘텐츠 표(content_id + keys 열) — 분모 계산용
    keys : 층을 나누는 열 이름 목록(예: ['region'], ['period'], [])

    Returns
    -------
    DataFrame: keys + target + 지표 열(한 행 = 한 층의 한 대상)
    """
    gk = keys + ["target"]
    r = rows.drop_duplicates(keys + ["target", "content_id"])
    g = r.groupby(gk, sort=False)
    out = g.agg(등장콘텐츠수=("content_id", "size"), 반복군보정등장수=("rep_key", "nunique"),
                고유출처수=("source", "nunique"), 관측월수_대상=("month", "nunique"),
                문자열출현횟수_보조=("tf", "sum"), 주분야=("area", "first"))
    top_src = r.groupby(gk + ["source"], sort=False).size().groupby(level=list(range(len(gk)))).max()
    out["최대출처점유율"] = (top_src.reindex(out.index) / out["등장콘텐츠수"]).round(4)
    out["출처당평균등장수"] = (out["등장콘텐츠수"] / out["고유출처수"]).round(3)
    # 분모
    if keys:
        den = cont.groupby(keys)["content_id"].nunique()
        out["유효콘텐츠분모"] = den.reindex(out.index.droplevel("target")).values
    else:
        out["유효콘텐츠분모"] = cont["content_id"].nunique()
    out["등장비중"] = np.where(out["유효콘텐츠분모"] > 0, out["등장콘텐츠수"] / out["유효콘텐츠분모"], np.nan)
    out["천건당언급"] = (out["등장비중"] * 1000).round(3)
    # 분야내 분모: 같은 층에서 그 주분야 대상이 하나라도 나온 콘텐츠 수
    ar = r.groupby(keys + ["area"])["content_id"].nunique() if keys else r.groupby(["area"])["content_id"].nunique()
    akey = list(zip(*[out.index.get_level_values(k) for k in keys], out["주분야"])) if keys else out["주분야"]
    out["분야내콘텐츠분모"] = ar.reindex(akey).values
    out["분야내등장비중"] = (out["등장콘텐츠수"] / out["분야내콘텐츠분모"]).round(6)
    if m == "뉴스":
        out["기사군수"] = out["반복군보정등장수"]
        for k in ("조회수", "좋아요", "댓글수"):
            for s in ("합계", "중앙값", "최댓값"):
                out[f"관련{k}{s}"] = np.nan
            out[f"{k}유효N"] = np.nan
            out[f"{k}결측수"] = np.nan
        out["조회수상위1콘텐츠기여율"] = out["좋아요상위1콘텐츠기여율"] = np.nan
        out["대표반응점수중앙값"] = out["반응점수유효N"] = np.nan
        out["반응점수유형"] = "적용불가"
    else:
        out["기사군수"] = np.nan
        rr = r[gk + ["content_id"]].merge(cont[["content_id"] + REACT_COLS].drop_duplicates("content_id"), on="content_id", how="left")
        for k in ("조회수", "좋아요", "댓글수"):
            v = rr[k].where(rr[k + "상태"].isin(["유효", "실제0"]))
            miss = rr[k + "상태"].isin(["숨김", "미제공"])
            gg = v.groupby([rr[c] for c in gk], sort=False)
            out[f"관련{k}합계"] = gg.sum(min_count=1).reindex(out.index)
            out[f"관련{k}중앙값"] = gg.median().reindex(out.index)
            out[f"관련{k}최댓값"] = gg.max().reindex(out.index)
            out[f"{k}유효N"] = gg.count().reindex(out.index)
            out[f"{k}결측수"] = miss.groupby([rr[c] for c in gk], sort=False).sum().reindex(out.index)
        out["조회수상위1콘텐츠기여율"] = (out["관련조회수최댓값"] / out["관련조회수합계"]).where(out["관련조회수합계"] > 0).round(4)
        out["좋아요상위1콘텐츠기여율"] = (out["관련좋아요최댓값"] / out["관련좋아요합계"]).where(out["관련좋아요합계"] > 0).round(4)
        sc = rr["대표반응점수"].groupby([rr[c] for c in gk], sort=False)
        out["대표반응점수중앙값"] = sc.median().reindex(out.index).round(2)
        out["반응점수유효N"] = sc.count().reindex(out.index)
        tt = rr[rr["반응점수유형"] != "산출불가"].groupby(gk + ["반응점수유형"], sort=False).size()
        if len(tt):
            dom = tt.reset_index().sort_values(0, ascending=False).drop_duplicates(gk).set_index(gk)["반응점수유형"]
            out["반응점수유형"] = dom.reindex(out.index).fillna("산출불가")
        else:
            out["반응점수유형"] = "산출불가"
    # 대표 콘텐츠: SNS는 대표반응점수 최고, 뉴스는 기사군이 가장 큰 최신 기사
    cu = cont.drop_duplicates("content_id").set_index("content_id")
    if m == "뉴스":
        sz = r["rep_key"].map(r["rep_key"].value_counts())
        rep = r.assign(_s=sz, _d=r["content_id"].map(cu["date"])).sort_values(["_s", "_d"], ascending=False)
    else:
        rep = r.assign(_s=r["content_id"].map(cu["대표반응점수"])).sort_values("_s", ascending=False)
    out["대표콘텐츠ID"] = rep.drop_duplicates(gk).set_index(gk)["content_id"].reindex(out.index)
    return out.reset_index()


def support_flags(m: str, df: pd.DataFrame) -> pd.DataFrame:
    """최소지지충족(콘텐츠 5개·출처 3개, 뉴스는 언론사 2개)과 단일출처의존 표시."""
    min_src = CONFIG["NEWS_MIN_PRESS"] if m == "뉴스" else CONFIG["MIN_SUPPORT_SOURCES"]
    ok = (df["등장콘텐츠수"] >= CONFIG["MIN_SUPPORT_CONTENTS"]) & (df["고유출처수"] >= min_src)
    df["최소지지충족"] = np.where(ok, "충족", np.where(df["등장콘텐츠수"] >= CONFIG["MIN_SUPPORT_CONTENTS"], "출처 부족", "콘텐츠 부족"))
    df["단일출처의존"] = df["최대출처점유율"] >= CONFIG["TOP_SOURCE_MAX_SHARE"]
    return df


def rank_within(df: pd.DataFrame, keys: list) -> pd.DataFrame:
    """층 안 빈도순위(등장콘텐츠수, 동점은 출처 수), 반복군보정순위, 상대반응순위(지지 충족 대상만)."""
    df = df.sort_values(keys + ["등장콘텐츠수", "고유출처수"], ascending=[True] * len(keys) + [False, False])
    grp = df.groupby(keys, sort=False) if keys else None
    df["빈도순위"] = (grp.cumcount() + 1) if keys else np.arange(1, len(df) + 1)
    df["반복군보정순위"] = df.groupby(keys)["반복군보정등장수"].rank(ascending=False, method="min") if keys else df["반복군보정등장수"].rank(ascending=False, method="min")
    sup = df["최소지지충족"] == "충족"
    rk = df[sup].groupby(keys)["대표반응점수중앙값"].rank(ascending=False, method="min") if keys else df.loc[sup, "대표반응점수중앙값"].rank(ascending=False, method="min")
    df["반응순위"] = rk.reindex(df.index)
    return df


print("집계 엔진 준비 완료")

### 10.1 기간·트랙 정의

| 트랙 | 사용 자료 | 용도 |
|---|---|---|
| **A** | 2023-01-01~2026-10-26 유효 자료 전체(2023 보완분 포함) | 관심대상 발굴: 매체별·시군별·분야별 다빈도, 고반응, 숨은 대상, 매체 비교. 결과 명칭은 '수집자료 내 관측 관심대상' |
| **B** | 2024~2026 비교 가능한 층: 뉴스 전체, 인스타 기본 구간, 유튜브 기본순·기본 수집 | 주 추세: HOT, RISING(2026 대 2025 동기간 기본, 2024 대비·2024~2025 연도별 비중 동일가중 평균 보조), STEADY, SEASONAL |
| **C** | A 자료 + 2023~2025 동일가중 평균 기준 | 2023 포함 민감도: 판정이 바뀌면 '2023 보완수집 민감' |

**연도별 결과(트랙 Y, 2차 검토 반영)**: 매체별·분야별·시군별·SNS 반응을 연도마다 따로 낼 때는 각 연도 1/1~10/26 동기간에, 2024~2026은 트랙 B와 같은 비교 가능한 수집층을 씁니다. 2023년 SNS는 보완 수집층(인스타 인기·릴스 탭 포함, 유튜브 조회수순 상위)밖에 없어 그대로 집계하되 '참고(수집 방식 다름)'로 표시합니다. 연도 간 비교는 등장콘텐츠수가 아니라 등장비중으로 합니다.

연도 비교는 각 연도 1월 1일~10월 26일 동기간(2026 관측 종료일 기준)을 기본으로, 완전한 달(1~9월)을 민감도로 씁니다. 2023~2025 연간 전체는 별도 기간유형으로만 제공합니다. 계절은 계절연도(겨울 = 전년 12월~2월)로 정의하고 관측월·완전성·비교가능 여부를 함께 적습니다.

In [ ]:
def build_periods() -> pd.DataFrame:
    """기간 정의표: 기간ID, 기간유형, 시작, 종료, 계절연도, 관측월수, 기간완전성, 매체별 비교가능 여부."""
    cs, ce = pd.Timestamp(CONFIG["COMMON_START"]), pd.Timestamp(CONFIG["COMMON_END"])
    supp_end = pd.Timestamp(CONFIG["SUPP_SEGMENT_END"])
    rows = [("A_전체", "공통기간 전체", cs, ce, "", "전체")]
    for y in (2023, 2024, 2025, 2026):
        rows.append((f"Y{y}", "연도 동기간(1/1~10/26)", pd.Timestamp(f"{y}-{CONFIG['YEAR_WINDOW'][0]}"), pd.Timestamp(f"{y}-{CONFIG['YEAR_WINDOW'][1]}"), "", "동기간"))
        rows.append((f"Y{y}_9", "연도 민감도(1~9월)", pd.Timestamp(f"{y}-{CONFIG['YEAR_WINDOW_SENS'][0]}"), pd.Timestamp(f"{y}-{CONFIG['YEAR_WINDOW_SENS'][1]}"), "", "동기간"))
    for y in (2023, 2024, 2025):
        rows.append((f"F{y}", "연간 전체(부분연도 2026과 비교 금지)", pd.Timestamp(f"{y}-01-01"), pd.Timestamp(f"{y}-12-31"), "", "연간"))
    for sy in (2023, 2024, 2025, 2026):
        for s, ms in CONFIG["SEASONS"].items():
            if s == "겨울":
                a, b = pd.Timestamp(f"{sy-1}-12-01"), pd.Timestamp(f"{sy}-03-01") - pd.Timedelta(days=1)
            else:
                a = pd.Timestamp(f"{sy}-{ms[0]:02d}-01")
                b = (pd.Timestamp(f"{sy}-{ms[-1]:02d}-01") + pd.offsets.MonthEnd(0))
            rows.append((f"S{sy}{s}", "계절", a, b, f"{sy}{s}", "계절"))
    p = pd.DataFrame(rows, columns=["기간ID", "기간유형", "기간시작", "기간종료", "계절연도", "분류"])
    # 관측: 공통기간과 겹치는 부분
    p["관측시작"] = p["기간시작"].clip(lower=cs)
    p["관측종료"] = p["기간종료"].clip(upper=ce)
    p["관측월수"] = ((p["관측종료"].dt.year - p["관측시작"].dt.year) * 12 + p["관측종료"].dt.month - p["관측시작"].dt.month + 1).clip(lower=0)
    full = (p["기간시작"] >= cs) & (p["기간종료"] <= ce)
    p["기간완전성"] = np.where(full, "완전", np.where(p["관측종료"] < p["관측시작"], "미관측", "불완전"))
    reason = []
    for _, r in p.iterrows():
        if r["기간완전성"] == "완전":
            reason.append("")
        elif r["기간시작"] < cs:
            reason.append("2022년 12월 미수록(공통기간 이전)")
        else:
            reason.append("2026-10-27 이후 미수록(10월 부분·11월 미관측)")
    p["불완전기간사유"] = reason
    # SNS 비교가능: 보완 구간(2023-01-01~10-30)과 겹치면 비교불가(트랙 A·C만)
    sns_mixed = p["기간시작"] <= supp_end
    p["SNS_비교가능여부"] = np.where(p["분류"] == "전체", "해당없음(발굴용)",
                            np.where(sns_mixed, "불가(2023 보완 수집 구간 포함)", np.where(p["기간완전성"] == "완전", "가능(조건부: 비중 비교)", "불완전")))
    p["뉴스_비교가능여부"] = np.where(p["분류"] == "전체", "해당없음(발굴용)", np.where(p["기간완전성"] == "완전", "가능", "불완전"))
    p.loc[p["분류"] == "동기간", ["SNS_비교가능여부", "뉴스_비교가능여부"]] = p.loc[p["분류"] == "동기간", ["SNS_비교가능여부", "뉴스_비교가능여부"]].replace("불완전", "가능(동기간)")
    p.loc[(p["분류"] == "동기간") & ~sns_mixed, "SNS_비교가능여부"] = "가능(조건부: 비중 비교)"
    return p


PERIODS = build_periods()
B_GROUPS = {"뉴스": {"NEWS"}, "인스타": {"IG-기본"}, "유튜브": {"YT-기본순-기본"}}


def layer_mask(m: str, df: pd.DataFrame, track: str) -> pd.Series:
    """트랙별 자료 범위. B는 비교 가능한 수집층만."""
    if track == "B":
        return df["비교그룹ID"].isin(B_GROUPS[m])
    if track == "Y":   # 연도별 결과: 2024~2026은 비교 가능한 층(B), 2023은 그해 유일한 자료(보완 수집 포함)
        return df["비교그룹ID"].isin(B_GROUPS[m]) | (df["date"] < pd.Timestamp("2024-01-01"))
    return pd.Series(True, index=df.index)


def period_rows(m: str, pid: str, track: str = "A"):
    """기간·트랙에 맞는 (콘텐츠×대상 행, 콘텐츠 표)."""
    p = PERIODS.set_index("기간ID").loc[pid]
    c = CT[m]
    c = c[in_period(c["date"], p["관측시작"], p["관측종료"]) & layer_mask(m, c, track)]
    k = CONT[m]
    k = k[in_period(k["date"], p["관측시작"], p["관측종료"]) & layer_mask(m, k, track)]
    return c, k


PERIODS[["기간ID", "기간유형", "기간시작", "기간종료", "관측월수", "기간완전성", "SNS_비교가능여부"]].head(12)

## 11. 매체별·시군별·분야별·연도계절·월별 집계 (09~14번 시트)

- **09_매체별키워드전체**: 트랙 A(공통기간 전체)의 *모든* 관심대상. Excel에는 등장 3건 이상만 싣고 전체는 CSV에 둡니다.
- **10_시군별빈도**: 14개 시군 + 전북전체 + 지역미확정, A/B 연결만. 기간은 전체·연도 동기간·계절. 지역고유성비 = 그 시군 내 등장비중 ÷ 다른 지역 등장비중.
- **11_시군별SNS반응**: 절대 반응(관련 조회수·좋아요 합계)과 상대 반응(대표반응점수 중앙값), 사분면(빈도×반응, 지지조건 충족 대상의 중앙값 기준; 평균·상위사분위 기준 민감도 병기), 숨은 관심대상(빈도 낮음·반응 높음·지지 충족).
- **12_분야별빈도반응**: 주분야 기준 + 보조분야 기준 행을 따로 둡니다.
- **13_연도계절키워드**, **14_월별추이**: 같은 집계 키와 분모. 유튜브는 트랙 A(전체 수집층)와 B(기본순·기본)를 모두 제공합니다.

In [ ]:
def report_list(df: pd.DataFrame) -> pd.DataFrame:
    """보고용 상위 목록 조건: 최소지지 충족 & 단일출처의존 아님 & 6개 분야로 분류됨."""
    return df[(df["최소지지충족"] == "충족") & ~df["단일출처의존"] & df["주분야"].isin(AREAS)]


def finish(m: str, df: pd.DataFrame, keys: list, meta: dict) -> pd.DataFrame:
    """지지조건·순위·공통 집계키·해석 열을 붙입니다."""
    df = support_flags(m, df)
    df = rank_within(df, keys)
    df["대상ID"] = df["target"].map(target_id)
    df["표준관심대상"] = df["target"]
    df["매체"] = m
    df["분석버전"] = ANALYSIS_VERSION
    for k, v in meta.items():
        df[k] = v
    df["보조분야"] = df["target"].map(AREA_TABLE["보조분야"]).fillna("")
    df["하위유형"] = df["target"].map(AREA_TABLE["하위유형"]).fillna("")
    df["검토상태"] = np.where(df["target"].map(AREA_TABLE["검토필요"]).fillna(True), "검토(분야 규칙 불일치 또는 복수 분야)", "자동 판정")
    df["근거원장파일"] = f"07_키워드근거_{m}.csv"
    return df


def track_meta(m: str, track: str, period: pd.Series = None) -> dict:
    """분석트랙·수집층·비교가능판정·해석제약 등 공통 메타 열."""
    sns = m != "뉴스"
    if track == "A":
        meta = {"분석트랙": "A", "수집층": "전체 수집층", "보완수집포함여부": "포함" if sns else "해당없음",
                "비교그룹ID": ",".join(sorted(CONT[m]["비교그룹ID"].unique())), "선택방식": "수집층 혼합",
                "비교가능판정": "발굴용(연도 비교 불가)", "해석제약": "수집자료 내 관측 관심대상. 모집단 선호·연도별 성장의 확정 결과 아님"}
    elif track == "Y":
        y2023 = period is not None and period["관측시작"].year == 2023
        if not sns:
            meta = {"분석트랙": "Y", "수집층": "NEWS", "보완수집포함여부": "해당없음", "비교그룹ID": "NEWS",
                    "선택방식": SELECT_RULE["NEWS"][0], "비교가능판정": "가능", "해석제약": ""}
        elif y2023:
            g = sorted(CONT[m].loc[CONT[m]["date"] < pd.Timestamp("2024-01-01"), "비교그룹ID"].unique())
            meta = {"분석트랙": "Y", "수집층": ",".join(g), "보완수집포함여부": "포함(2023 보완 수집층)", "비교그룹ID": ",".join(g),
                    "선택방식": "; ".join(SELECT_RULE[x][0] for x in g if x in SELECT_RULE),
                    "비교가능판정": "참고(수집 방식 다름)", "해석제약": "2023 SNS는 인기·조회수순 표집이 섞여 2024~2026과 같은 기준이 아님. 비중도 직접 비교 금지"}
        else:
            meta = {"분석트랙": "Y", "수집층": ",".join(sorted(B_GROUPS[m])), "보완수집포함여부": "제외",
                    "비교그룹ID": ",".join(sorted(B_GROUPS[m])), "선택방식": "; ".join(SELECT_RULE[g][0] for g in sorted(B_GROUPS[m])),
                    "비교가능판정": "조건부", "해석제약": "비중으로만 비교(연도별 수록량·표집 구조 차이)"}
    else:
        meta = {"분석트랙": "B", "수집층": ",".join(sorted(B_GROUPS[m])), "보완수집포함여부": "제외" if sns else "해당없음",
                "비교그룹ID": ",".join(sorted(B_GROUPS[m])), "선택방식": "; ".join(SELECT_RULE[g][0] for g in sorted(B_GROUPS[m])),
                "비교가능판정": "가능" if m == "뉴스" else "조건부", "해석제약": "" if m == "뉴스" else "비중으로만 비교(연도별 수록량·표집 구조 차이)"}
    meta["행별출처확인수준"] = "단일 수집" if not sns else "구간 정의(작성일 기준)로 확인"
    meta["콘텐츠형식"] = "전체"
    if period is not None:
        meta.update({"기간유형": period["기간유형"], "기간시작": period["관측시작"].strftime("%Y-%m-%d"),
                     "기간종료": period["관측종료"].strftime("%Y-%m-%d"), "계절연도": period["계절연도"],
                     "관측월수": int(period["관측월수"]), "기간완전성": period["기간완전성"],
                     "비교가능여부": period["SNS_비교가능여부"] if sns else period["뉴스_비교가능여부"]})
    return meta


PER = PERIODS.set_index("기간ID")
T09 = {}
for m in MEDIA:
    t0 = time.time()
    c, k = period_rows(m, "A_전체", "A")
    df = aggregate(m, c, k, [])
    meta = track_meta(m, "A", PER.loc["A_전체"])
    meta.update({"집계수준": "매체 전체", "지역": "전체", "분모정의": "같은 매체·기간 유효 콘텐츠 수"})
    T09[m] = finish(m, df, [], meta)
    ctx = target_context(m, T09[m].loc[T09[m]["등장콘텐츠수"] >= 3, "target"])
    T09[m]["관심맥락"] = T09[m]["target"].map(ctx).fillna("")
    print(f"09 {m}: {len(T09[m]):,}개 대상 ({time.time()-t0:.0f}초)")
T09["인스타"][["target", "등장콘텐츠수", "반복군보정등장수", "고유출처수", "최대출처점유율", "주분야", "관심맥락", "대표반응점수중앙값"]].head(15)

In [ ]:
_RL = {}


def region_table(m: str, pid: str, track: str = "A", min_n: int = 3) -> pd.DataFrame:
    """시군(지역)별 빈도·반응 표. A/B 연결만 핵심 집계에 쓰고 C 동반 출현 수는 별도 열로 붙입니다."""
    p = PER.loc[pid]
    if m not in _RL:
        meta_ct = CT[m][["content_id", "target", "rep_key", "source", "month", "date", "area", "tf", "비교그룹ID"]]
        _RL[m] = REL[m][["content_id", "target", "최종귀속시군", "연결등급", "다중귀속여부"]].merge(meta_ct, on=["content_id", "target"], how="left") \
            .rename(columns={"최종귀속시군": "region"})
    r = _RL[m]
    r = r[in_period(r["date"], p["관측시작"], p["관측종료"]) & layer_mask(m, r, track)]
    core = r[r["연결등급"].isin(["A", "B"])]
    cont = CONT[m][in_period(CONT[m]["date"], p["관측시작"], p["관측종료"]) & layer_mask(m, CONT[m], track)]
    cont = cont[["content_id", "region"] + REACT_COLS + ["date"]].explode("region")
    df = aggregate(m, core, cont, ["region"])
    cnt_c = r[r["연결등급"] == "C"].groupby(["region", "target"])["content_id"].nunique().rename("C동반출현수")
    multi = core[core["다중귀속여부"]].groupby(["region", "target"])["content_id"].nunique().rename("다중귀속콘텐츠수")
    df = df.join(cnt_c, on=["region", "target"]).join(multi, on=["region", "target"])
    df[["C동반출현수", "다중귀속콘텐츠수"]] = df[["C동반출현수", "다중귀속콘텐츠수"]].fillna(0).astype(int)
    df["다중귀속여부"] = np.where(df["다중귀속콘텐츠수"] > 0, "일부 다중귀속", "없음")
    # 지역고유성비 = 해당 지역 비중 / 다른 지역 비중(다른 지역 = 14개 시군 중 나머지)
    tot_n = df[df["region"].isin(SIGUN)].groupby("target")["등장콘텐츠수"].sum()
    den_all = cont[cont["region"].isin(SIGUN)].groupby("region")["content_id"].nunique()
    other_n = df["target"].map(tot_n) - df["등장콘텐츠수"]
    other_den = den_all.sum() - df["region"].map(den_all)
    other_share = other_n / other_den
    df["지역고유성분모"] = other_den
    df["지역고유성비"] = np.where(df["region"].isin(SIGUN) & (other_share > 0), df["등장비중"] / other_share, np.nan)
    df["지역고유성판정"] = np.where(~df["region"].isin(SIGUN), "해당없음", np.where(other_n.fillna(0) == 0, "다른 지역 0건(신규/비교불가)", ""))
    df = df[df["등장콘텐츠수"] >= min_n]
    meta = track_meta(m, track, p)
    meta.update({"집계수준": "시군", "분모정의": "같은 매체·지역·기간 유효 콘텐츠 수(본문 지명 또는 대상 귀속)"})
    df = finish(m, df, ["region"], meta)
    df["지역"] = df.pop("region")
    df["시군내빈도순위"] = df["빈도순위"]
    df["관심맥락"] = df["target"].map(T09[m].set_index("target")["관심맥락"]).fillna("")
    return df


t0 = time.time()
T10_parts = []
for m in MEDIA:
    pids = ["A_전체"] + [f"Y{y}" for y in (2023, 2024, 2025, 2026)] + [p for p in PER.index if p.startswith("S") and PER.at[p, "기간완전성"] != "미관측"]
    for pid in pids:
        T10_parts.append(region_table(m, pid, "Y" if pid.startswith("Y") else "A"))
T10 = pd.concat(T10_parts, ignore_index=True)
print(f"10_시군별빈도 {len(T10):,}행 ({time.time()-t0:.0f}초)")
T10[(T10["매체"] == "인스타") & (T10["기간유형"] == "공통기간 전체") & (T10["지역"] == "부안군")][["시군내빈도순위", "표준관심대상", "등장콘텐츠수", "등장비중", "고유출처수", "지역고유성비"]].head(10)

In [ ]:
def quadrant(df: pd.DataFrame, keys: list, basis: str = "median") -> pd.DataFrame:
    """빈도×반응 사분면. 지지조건 충족 대상들의 기준선(중앙값/평균/상위 사분위)으로 나눕니다."""
    sup = df[(df["최소지지충족"] == "충족") & df["대표반응점수중앙값"].notna()]
    fn = {"median": "median", "mean": "mean", "q3": lambda s: s.quantile(0.75)}[basis]
    base_f = sup.groupby(keys)["등장콘텐츠수"].agg(fn)
    base_r = sup.groupby(keys)["대표반응점수중앙값"].agg(fn)
    n = sup.groupby(keys).size()
    idx = pd.MultiIndex.from_frame(df[keys]) if len(keys) > 1 else df[keys[0]]
    bf, br, nn = base_f.reindex(idx).values, base_r.reindex(idx).values, n.reindex(idx).values
    hi_f, hi_r = df["등장콘텐츠수"].values > bf, df["대표반응점수중앙값"].values > br
    q = np.where(hi_f & hi_r, "언급많음·반응높음", np.where(hi_f, "언급많음·반응낮음", np.where(hi_r, "언급적음·반응높음", "언급적음·반응낮음")))
    ok = (df["최소지지충족"] == "충족").values & df["대표반응점수중앙값"].notna().values
    return pd.DataFrame({"사분면": np.where(ok, q, "지지부족"), "사분면빈도기준선": bf, "사분면반응기준선": br, "사분면후보수": nn}, index=df.index)


HIDDEN_REVIEW = pd.read_csv(DICT_DIR / "숨은관심대상_검토.csv") if (DICT_DIR / "숨은관심대상_검토.csv").exists() \
    else pd.DataFrame(columns=["매체", "표준관심대상", "판정", "사유"])
_HR = {(r["매체"], normalize_unit(r["표준관심대상"])): (r["판정"], r["사유"]) for _, r in HIDDEN_REVIEW.iterrows()}


_CSETS = {m: CT[m].groupby("content_id")["target"].agg(frozenset) for m in ("인스타", "유튜브")}
_TIDX = {m: CT[m].groupby("target").indices for m in ("인스타", "유튜브")}


@lru_cache(maxsize=None)
def independent_texts(m: str, t: str, start, end, thr: float = 0.5, cap: int = 300) -> int:
    """대상 t가 나온 콘텐츠 중 '같은 안내문을 옮겨 올린' 묶음을 하나로 본 독립 문구 수.

    두 콘텐츠의 관심대상 집합 자카드 유사도가 thr 이상이면 같은 문구로 보고 연결 성분 수를 셉니다
    (축제 프로그램 안내문을 여러 계정이 그대로 올린 경우 방지).
    """
    rows = CT[m].iloc[_TIDX[m].get(t, [])]
    rows = rows[in_period(rows["date"], start, end)]
    ids = list(dict.fromkeys(rows["content_id"]))[:cap]
    sets = [_CSETS[m][c] for c in ids]
    parent = list(range(len(ids)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            a, b = sets[i], sets[j]
            if len(a & b) / max(len(a | b), 1) >= thr:
                parent[find(i)] = find(j)
    return len({find(i) for i in range(len(ids))})


def sns_reaction_table(m: str, pid: str = "A_전체") -> pd.DataFrame:
    """11_시군별SNS반응: 지역별(+매체 전체) 절대·상대 반응, 사분면, 숨은 관심대상과 의존 구분.

    pid = 'A_전체'(2023~2026 전체, 트랙 A) 또는 'Y2023'~'Y2026'(연도 동기간, 트랙 Y).
    숨은 관심대상은 ① 언급적음·반응높음 사분면 ② 지지조건 충족 ③ 단일출처의존 아님을 '후보'로 두고,
    ④ 반응이 소수 게시물·특정 계정에 몰리지 않고 ⑤ 원문 검토(dict/숨은관심대상_검토.csv)에서 제외되지 않은 것만 '예'로 확정합니다.
    """
    if pid == "A_전체":
        reg = T10[(T10["매체"] == m) & (T10["기간유형"] == "공통기간 전체")].copy()
        allm = T09[m][T09[m]["등장콘텐츠수"] >= 3].copy()
    else:
        per = PER.loc[pid]
        st = per["관측시작"].strftime("%Y-%m-%d")
        reg = T10[(T10["매체"] == m) & (T10["기간유형"] == per["기간유형"]) & (T10["기간시작"] == st)].copy()
        allm = T13[(T13["매체"] == m) & (T13["분석트랙"] == "Y") & (T13["비교기간ID"] == pid)].copy()
    allm["지역"] = "매체 전체"
    allm["시군내빈도순위"] = allm["빈도순위"]
    df = pd.concat([allm, reg], ignore_index=True)
    sup = df["최소지지충족"] == "충족"
    df["시군내절대반응순위"] = df[sup].groupby("지역")["관련좋아요합계" if m == "인스타" else "관련조회수합계"].rank(ascending=False, method="min").reindex(df.index)
    df["시군내상대반응순위"] = df[sup].groupby("지역")["대표반응점수중앙값"].rank(ascending=False, method="min").reindex(df.index)
    q = quadrant(df, ["지역"], "median")
    df = pd.concat([df, q], axis=1)
    for b, name in (("mean", "평균"), ("q3", "상위사분위")):
        df[f"사분면_{name}기준"] = quadrant(df, ["지역"], b)["사분면"]
    top1 = df["좋아요상위1콘텐츠기여율" if m == "인스타" else "조회수상위1콘텐츠기여율"]
    few_posts = (top1 >= CONFIG["EVENT_TOP1_SHARE"]) | (df["반응점수유효N"].fillna(0) < CONFIG["HIDDEN_MIN_SCORED"])
    one_acct = (df["최대출처점유율"] >= CONFIG["HIDDEN_MAX_SOURCE_SHARE"]) | (df["고유출처수"] < CONFIG["HIDDEN_MIN_SOURCES"])
    df["반응의존구분"] = np.where(few_posts & one_acct, "소수 게시물·특정 계정 의존", np.where(few_posts, "소수 게시물 의존",
                         np.where(one_acct, "특정 계정 의존", "여러 게시물·계정 분산")))
    df["숨은관심대상후보"] = np.where((df["사분면"] == "언급적음·반응높음") & ~df["단일출처의존"].astype(bool), "예", "")
    rv = [_HR.get((m, normalize_unit(t)), ("", "")) for t in df["표준관심대상"]]
    df["원문검토판정"] = [a for a, _ in rv]
    df["원문검토사유"] = [b for _, b in rv]
    cand = df["숨은관심대상후보"] == "예"
    p_ = PER.loc[pid]
    df["독립문구수"] = np.nan
    df.loc[cand, "독립문구수"] = [independent_texts(m, t, p_["관측시작"], p_["관측종료"]) for t in df.loc[cand, "표준관심대상"]]
    df["대상소재시군"] = df["표준관심대상"].map(HOME["대상소재시군"]).fillna("")
    df["숨은관심대상여부"] = np.select(
        [~cand, df["원문검토판정"] == "제외", df["대상소재시군"] == "", df["독립문구수"] < CONFIG["HIDDEN_MIN_INDEP_TEXTS"],
         df["반응의존구분"] != "여러 게시물·계정 분산"],
        ["", "제외(원문 검토)", "보류(소재 시군 불명: 일반어·타 지역 가능)", "보류(같은 안내문 반복 게시)", "보류(" + df["반응의존구분"] + ")"], "예")
    df["원문검토상태"] = np.where(df["원문검토판정"] != "", "검토 완료", np.where(cand, "미검토(자동 판정)", ""))
    df["소수근거여부"] = np.where(df["최소지지충족"] != "충족", np.where(top1 >= CONFIG["EVENT_TOP1_SHARE"], "단일흥행 후보", "소수근거"), "")
    df["집계수준"] = np.where(df["지역"] == "매체 전체", "매체 전체", "시군")
    df["비교기간ID"] = pid
    df["분석기간"] = "2023~2026 전체" if pid == "A_전체" else f"{pid[1:]}년(1/1~10/26)"
    return df


T11 = pd.concat([sns_reaction_table(m, "A_전체") for m in ("인스타", "유튜브")], ignore_index=True)   # 연도별은 13번 집계 뒤에 추가
_t11 = T11
print("11_시군별SNS반응:", len(T11), "행 / 전체기간 숨은 관심대상 후보:", int((_t11["숨은관심대상후보"] == "예").sum()),
      "확정:", int((_t11["숨은관심대상여부"] == "예").sum()))
_t11[(_t11["지역"] == "매체 전체") & (_t11["숨은관심대상후보"] == "예")].sort_values("대표반응점수중앙값", ascending=False)[["매체", "표준관심대상", "등장콘텐츠수", "고유출처수", "대표반응점수중앙값", "반응의존구분", "숨은관심대상여부"]].head(15)

In [ ]:
def area_table(m: str) -> pd.DataFrame:
    """12_분야별빈도반응: 주분야 행 + 보조분야 행. 분류가능콘텐츠비율 = 분류된 대상이 1개 이상인 콘텐츠 비율."""
    base = T09[m][T09[m]["등장콘텐츠수"] >= 3].copy()
    c, k = period_rows(m, "A_전체", "A")
    classified = c[c["area"].isin(AREAS)]["content_id"].nunique() / max(k["content_id"].nunique(), 1)
    main = base[base["주분야"].isin(AREAS)].copy()
    main["주분야복수분야구분"] = "주분야"
    main["분야"] = main["주분야"]
    sub = base[base["보조분야"].isin(AREAS)].copy()
    sub["주분야복수분야구분"] = "보조분야"
    sub["분야"] = sub["보조분야"]
    out = pd.concat([main, sub], ignore_index=True)
    out["분류가능콘텐츠비율"] = round(classified, 4)
    out["집계수준"] = "분야"
    out["분야내순위"] = out.sort_values("등장콘텐츠수", ascending=False).groupby(["분야", "주분야복수분야구분"]).cumcount().reindex(out.index) + 1
    return out


T12 = pd.concat([area_table(m) for m in MEDIA], ignore_index=True)
print("12_분야별빈도반응:", len(T12), "행")
print(T12.drop_duplicates("매체").set_index("매체")["분류가능콘텐츠비율"])

In [ ]:
def period_table(m: str, pids: list, track: str, min_n: int = 3) -> pd.DataFrame:
    """13_연도계절키워드: 기간별 빈도·반응. 기간마다 별도로 집계해 같은 분모 정의를 씁니다."""
    parts = []
    for pid in pids:
        p = PER.loc[pid]
        if p["기간완전성"] == "미관측":
            continue
        c, k = period_rows(m, pid, track)
        if k.empty:
            continue
        df = aggregate(m, c, k, [])
        df = df[df["등장콘텐츠수"] >= min_n]
        meta = track_meta(m, track, p)
        meta.update({"집계수준": "매체 전체", "지역": "전체", "분모정의": "같은 매체·기간·수집층 유효 콘텐츠 수", "비교기간ID": pid,
                     "대응동기간": (f"{p['관측시작']:%m-%d}~{p['관측종료']:%m-%d}" if p["기간유형"].startswith("연도") else ""),
                     "불완전기간사유": p["불완전기간사유"]})
        if track == "A" and m != "뉴스" and str(p["SNS_비교가능여부"]).startswith("불가"):
            meta["비교가능판정"] = "불가(2023 보완 수집 포함)"
        parts.append(finish(m, df, [], meta))
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


t0 = time.time()
_p13 = [p for p in PER.index if p != "A_전체"]
_pY = [f"Y{y}" for y in (2023, 2024, 2025, 2026)]
T13 = pd.concat([period_table("뉴스", _p13, "A"), period_table("인스타", _p13, "A"),
                 period_table("유튜브", _p13, "A"), period_table("유튜브", _p13, "B")]
                + [period_table(m, _pY, "Y") for m in MEDIA], ignore_index=True)
for m in MEDIA:
    ctx = T09[m].set_index("target")["관심맥락"]
    T13.loc[T13["매체"] == m, "관심맥락"] = T13.loc[T13["매체"] == m, "표준관심대상"].map(ctx).fillna("")
# 연도별 SNS 반응(트랙 Y)을 11번 표에 추가
T11 = pd.concat([T11] + [sns_reaction_table(m, pid) for m in ("인스타", "유튜브") for pid in _pY], ignore_index=True)
print(f"13_연도계절키워드 {len(T13):,}행 / 11 연도별 추가 후 {len(T11):,}행")
display(pd.crosstab([T11["매체"], T11["분석기간"]], T11.loc[T11["집계수준"] == "매체 전체", "숨은관심대상여부"].replace("", "-")))
print(f"13_연도계절키워드 {len(T13):,}행 ({time.time()-t0:.0f}초)")

In [ ]:
def monthly_table(m: str, track: str, min_total: int = 20) -> pd.DataFrame:
    """14_월별추이: 대상별 월간 등장콘텐츠수·비중(그 달 유효 콘텐츠 분모). 0관측과 자료없음(분모 0)을 구분합니다."""
    c = CT[m][in_period(CT[m]["date"], CONFIG["COMMON_START"], CONFIG["COMMON_END"]) & layer_mask(m, CT[m], track)]
    k = CONT[m][in_period(CONT[m]["date"], CONFIG["COMMON_START"], CONFIG["COMMON_END"]) & layer_mask(m, CONT[m], track)]
    tot = c.groupby("target")["content_id"].nunique()
    tg = tot[tot >= min_total].index
    num = c[c["target"].isin(tg)].groupby(["target", "month"])["content_id"].nunique()
    rep = c[c["target"].isin(tg)].groupby(["target", "month"])["rep_key"].nunique()
    den = k.groupby("month")["content_id"].nunique()
    months = pd.period_range(CONFIG["COMMON_START"], CONFIG["COMMON_END"], freq="M").astype(str)
    full = pd.MultiIndex.from_product([tg, months], names=["target", "month"])
    out = pd.DataFrame({"등장콘텐츠수": num.reindex(full).fillna(0).astype(int), "반복군보정등장수": rep.reindex(full).fillna(0).astype(int)}).reset_index()
    out["유효콘텐츠분모"] = out["month"].map(den).fillna(0).astype(int)
    out["등장비중"] = np.where(out["유효콘텐츠분모"] > 0, out["등장콘텐츠수"] / out["유효콘텐츠분모"].where(out["유효콘텐츠분모"] > 0), np.nan)
    out["천건당언급"] = (out["등장비중"] * 1000).round(3)
    out["관측상태"] = np.where(out["유효콘텐츠분모"] == 0, "자료없음", np.where(out["등장콘텐츠수"] == 0, "0관측", "관측"))
    out["누적관측월"] = out.groupby("target").cumcount() + 1
    out["대상ID"] = out["target"].map(target_id)
    out["표준관심대상"] = out["target"]
    out["주분야"] = out["target"].map(AREA_MAP).fillna("미분류(검토필요)")
    out["매체"] = m
    out["분석버전"] = ANALYSIS_VERSION
    meta = track_meta(m, track)
    for kk, v in meta.items():
        out[kk] = v
    out["월"] = out.pop("month")
    out["기간유형"], out["기간시작"], out["기간종료"] = "월", out["월"] + "-01", out["월"]
    out["집계수준"], out["지역"], out["분모정의"] = "매체 전체", "전체", "그 달 유효 콘텐츠 수"
    seg = np.where(out["월"] <= "2023-10", "보완 구간", "기본 구간")
    out["동일표집층비교여부"] = np.where(m == "뉴스", "예", np.where((track == "B") | (seg == "기본 구간"), "예(기본 구간)", "아니오(2023 보완 구간)"))
    return out.drop(columns="target")


T14 = pd.concat([monthly_table("뉴스", "A"), monthly_table("인스타", "A"), monthly_table("유튜브", "A"), monthly_table("유튜브", "B")], ignore_index=True)
print(f"14_월별추이 {len(T14):,}행, 대상 {T14['대상ID'].nunique():,}개")

## 12. 관심 유형: HOT·RISING·STEADY·SEASONAL·EVENT/VIRAL (과업지시서 16장, 초기 기준)

| 유형 | 기준(초기값, `CONFIG`) | 트랙 |
|---|---|---|
| HOT | 2026 동기간(1/1~10/26) 등장 건수 상위 30 중 최소 건수(뉴스 50·인스타 30·유튜브 15)와 출처 다양성(언론사 3·작성자 10·채널 5) 충족. SNS는 고반응 여부를 별도 열로 | B |
| RISING | 2026 대 2025 동기간 비중비 ≥ 1.5, 천건당언급 증가 ≥ 1, 현재 고유 콘텐츠 ≥ 5, 관측월 ≥ 2, SNS 출처 ≥ 3(뉴스 언론사 ≥ 2, 1곳이면 단일 언론사 후보). 2024 비중과 2024~2025 동일가중 평균 비중을 함께 제시 | B |
| 신규 관측 | 2024·2025 동기간 모두 0건이고 현재 지지조건 충족. 증가율은 표기하지 않음 | B |
| STEADY | 비교 가능한 3개 연도(뉴스 2023~2026, SNS 2024~2026 동기간)에 매년 3건 이상, 전체 20건 이상, 출처 3개 이상. 연도별 비중 변동계수 ≤ 0.5이면 '지속 관심형', 아니면 후보 | B |
| SEASONAL | 완전 관측·동일 표집 계절 중 같은 계절이 2개 연도 이상에서 계절 지수(그 계절 비중 ÷ 전체 완전 계절 비중) ≥ 1.5이고 각 3건 이상 | B |
| EVENT/VIRAL | 보고용 조건 충족 대상만 점검. 단일콘텐츠형: 반응량 상위 1개 콘텐츠 기여율 ≥ 50%. 사건형: 한 연도 등장의 50% 이상이 한 달에 집중(그 달 수록량으로 나눈 *월별 비중* 기준, 연 10건 이상). 같은 달 집중이 2개 연도 이상 반복되면 '판정유보(연례 행사 가능)' | A |

통과 대상이 없으면 '없음'으로 보고하며 기준을 완화하지 않습니다. 유형은 중복 부여할 수 있습니다. RISING은 게시물 등장 관심의 변화이며, 한 번 수집한 누적 조회수로 과거 반응의 증가를 복원하지 않습니다.

In [ ]:
def year_stats(m: str, track: str, years, win: str = "") -> dict:
    """연도별 동기간(또는 1~9월) 집계 {연도: aggregate 결과(target 인덱스)}."""
    out = {}
    for y in years:
        c, k = period_rows(m, f"Y{y}{win}", track)
        if k.empty:
            continue
        out[y] = aggregate(m, c, k, []).set_index("target")
    return out


YS = {}
for m in MEDIA:
    t0 = time.time()
    YS[(m, "B", "")] = year_stats(m, "B", (2023, 2024, 2025, 2026) if m == "뉴스" else (2024, 2025, 2026))
    YS[(m, "A", "")] = year_stats(m, "A", (2023,))          # 트랙 C 기준용 2023(보완분 포함)
    YS[(m, "B", "_9")] = year_stats(m, "B", (2023, 2024, 2025, 2026) if m == "뉴스" else (2024, 2025, 2026), "_9")
    YS[(m, "A", "_9")] = year_stats(m, "A", (2023,), "_9")
    print(f"{m}: 연도 집계 {time.time()-t0:.0f}초")

In [ ]:
def share_of(ys: dict, y: int, col: str = "등장비중") -> pd.Series:
    return ys[y][col] if y in ys else pd.Series(dtype=float)


def rising_table(m: str, win: str = "") -> pd.DataFrame:
    """RISING·신규 관측 판정표(트랙 B 기본 + 트랙 C 민감도 판정 병기)."""
    yb = YS[(m, "B", win)]
    ya = YS[(m, "A", win)]
    cur = yb[2026]
    base = yb[2025]
    t = cur.index
    den_cur = cur["유효콘텐츠분모"].iloc[0]
    den_base = base["유효콘텐츠분모"].iloc[0]
    s25 = base["등장비중"].reindex(t).fillna(0)
    s24 = yb[2024]["등장비중"].reindex(t).fillna(0)
    n25 = base["등장콘텐츠수"].reindex(t).fillna(0)
    n24 = yb[2024]["등장콘텐츠수"].reindex(t).fillna(0)
    avg_b = (s24 + s25) / 2
    y23 = yb[2023] if 2023 in yb else ya.get(2023)
    s23 = y23["등장비중"].reindex(t).fillna(0) if y23 is not None else pd.Series(0, index=t)
    avg_c = (s23 + s24 + s25) / 3
    sc = cur["등장비중"]
    min_src = CONFIG["NEWS_MIN_PRESS"] if m == "뉴스" else CONFIG["MIN_SUPPORT_SOURCES"]
    sup = (cur["등장콘텐츠수"] >= CONFIG["MIN_SUPPORT_CONTENTS"]) & (cur["관측월수_대상"] >= CONFIG["RISING_MIN_MONTHS"])
    src_ok = cur["고유출처수"] >= min_src
    ratio = np.where(s25 > 0, sc / s25.where(s25 > 0), np.nan)
    ratio_avg = np.where(avg_b > 0, sc / avg_b.where(avg_b > 0), np.nan)
    ratio_c = np.where(avg_c > 0, sc / avg_c.where(avg_c > 0), np.nan)
    d1000 = (sc - s25) * 1000
    d1000_c = (sc - avg_c) * 1000
    new = (n24 == 0) & (n25 == 0)
    # 기본 판정: 2025 대비. 2025=0이고 2024>0이면 보조 기준(2024~2025 평균)으로 판정
    basis = np.where(s25 > 0, "2025 동기간", np.where(avg_b > 0, "2024~2025 동일가중 평균(2025=0)", "기준 0(신규)"))
    r_use = np.where(s25 > 0, ratio, ratio_avg)
    d_use = np.where(s25 > 0, d1000, (sc - avg_b) * 1000)
    pass_b = sup & src_ok & ~new & (r_use >= CONFIG["RISING_MIN_RATIO"]) & (d_use >= CONFIG["RISING_MIN_PER1000_DELTA"])
    pass_c = sup & src_ok & (avg_c > 0) & (ratio_c >= CONFIG["RISING_MIN_RATIO"]) & (d1000_c >= CONFIG["RISING_MIN_PER1000_DELTA"])
    new_ok = sup & src_ok & new
    single_press = (m == "뉴스") & sup & ~src_ok & (cur["고유출처수"] == 1) & (r_use >= CONFIG["RISING_MIN_RATIO"])
    out = pd.DataFrame({
        "target": t, "비교기준연도": "2025(보조: 2024, 2024~2025 평균)", "기준기간시작": f"2025-{CONFIG['YEAR_WINDOW'][0]}",
        "기준기간종료": f"2025-{(CONFIG['YEAR_WINDOW_SENS'] if win else CONFIG['YEAR_WINDOW'])[1]}",
        "기준등장수": n25.values, "기준분모": den_base, "기준등장비중": s25.values,
        "현재등장수": cur["등장콘텐츠수"].values, "현재분모": den_cur, "현재등장비중": sc.values,
        "비중비": np.round(ratio, 3), "비중차": (sc - s25).values, "천건당언급증가": np.round(d1000, 3), "건수차": (cur["등장콘텐츠수"] - n25).values,
        "2024등장수": n24.values, "2024등장비중": s24.values, "2023등장비중_C": s23.values,
        "과거동일가중평균비중": avg_b.values, "평균대비비중비": np.round(ratio_avg, 3),
        "C기준_2023포함평균비중": avg_c.values, "C기준_비중비": np.round(ratio_c, 3),
        "판정기준": basis, "신규관측여부": np.where(new, "예", ""),
        "RISING_B": pass_b, "신규관측": new_ok, "RISING_C": pass_c, "단일언론사후보": single_press,
        "현재관측월수": cur["관측월수_대상"].values, "현재고유출처수": cur["고유출처수"].values,
        "최대출처점유율": cur["최대출처점유율"].values,
    }).set_index("target")
    out["과거연도별비중_원장참조"] = "13_연도계절키워드(비교기간ID=Y2023~Y2026)"
    return out


def tag_sampled_targets() -> set:
    """수집태그·검색어와 같은 표현(표집 변경 영향 가능 대상): 인스타 태그 범위 시트의 태그, 유튜브 검색어의 뒷말."""
    tags = {normalize_unit(str(t).lstrip("#")) for t in sns_raw["인스타태그범위"]["태그"].dropna()}
    return {t for t in tags if t}


SAMPLED = tag_sampled_targets()
RIS = {}
for m in MEDIA:
    r = rising_table(m)
    r9 = rising_table(m, "_9")
    r["RISING_1~9월"] = r9["RISING_B"].reindex(r.index).fillna(False)
    r["신규_1~9월"] = r9["신규관측"].reindex(r.index).fillna(False)
    RIS[m] = r
    print(f"{m}: RISING(B) {int(r['RISING_B'].sum())}개, 신규 관측 {int(r['신규관측'].sum())}개, RISING(C) {int(r['RISING_C'].sum())}개")

In [ ]:
def type_common(m: str, df: pd.DataFrame, pid: str, track: str) -> pd.DataFrame:
    """유형 시트 공통 열을 붙입니다(현재 기간 집계 지표 + 공통 집계키)."""
    c, k = period_rows(m, pid, track)
    agg = aggregate(m, c[c["target"].isin(df.index)], k, []).set_index("target")
    agg = agg.drop(columns=[x for x in agg.columns if x in df.columns])
    x = df.join(agg, how="left").reset_index().rename(columns={"index": "target"})
    meta = track_meta(m, track, PER.loc[pid])
    meta.update({"집계수준": "매체 전체", "지역": "전체", "분모정의": "같은 매체·기간·수집층 유효 콘텐츠 수"})
    return finish(m, x, [], meta)


def hot_table(m: str) -> pd.DataFrame:
    cur = YS[(m, "B", "")][2026]
    ok = (cur["등장콘텐츠수"] >= CONFIG["HOT_MIN_COUNT"][m]) & (cur["고유출처수"] >= CONFIG["HOT_MIN_DIVERSITY"][m])
    cand = cur[ok].sort_values(["등장콘텐츠수", "고유출처수"], ascending=False)
    top = cand.head(CONFIG["HOT_TOP_K"])
    df = pd.DataFrame(index=cur.sort_values("등장콘텐츠수", ascending=False).head(CONFIG["HOT_TOP_K"] * 3).index.union(top.index))
    df["HOT"] = df.index.isin(top.index)
    df["최신기간"] = f"2026-{CONFIG['YEAR_WINDOW'][0]}~2026-{CONFIG['YEAR_WINDOW'][1]}"
    df["HOT선정기준"] = f"등장 상위 {CONFIG['HOT_TOP_K']} & 건수 ≥ {CONFIG['HOT_MIN_COUNT'][m]} & 출처 ≥ {CONFIG['HOT_MIN_DIVERSITY'][m]}"
    why = []
    for t in df.index:
        r = cur.loc[t]
        miss = []
        if r["등장콘텐츠수"] < CONFIG["HOT_MIN_COUNT"][m]:
            miss.append("건수 미달")
        if r["고유출처수"] < CONFIG["HOT_MIN_DIVERSITY"][m]:
            miss.append("출처 다양성 미달")
        if not miss and t not in top.index:
            miss.append("상위 순위 밖")
        why.append(", ".join(miss))
    df["미충족사유"] = why
    out = type_common(m, df, "Y2026", "B")
    out["선정근거"] = np.where(out["HOT"], "2026 동기간 등장 " + out["등장콘텐츠수"].astype(str) + "건, 출처 " + out["고유출처수"].astype(str) + "곳", "")
    if m != "뉴스":
        med = out.loc[out["최소지지충족"] == "충족", "대표반응점수중앙값"].median()
        out["고반응여부"] = np.where(out["대표반응점수중앙값"] > med, "고반응(점수 중앙값 이상)", "보통 이하")
    else:
        out["고반응여부"] = "적용불가"
    return out.sort_values(["HOT", "등장콘텐츠수"], ascending=False)


def rising_sheet(m: str) -> pd.DataFrame:
    r = RIS[m]
    keep = r[r["RISING_B"] | r["신규관측"] | r["RISING_C"] | r["단일언론사후보"]].copy()
    keep["표집변경가능성"] = np.where(keep.index.map(lambda t: normalize_unit(t) in SAMPLED), "수집태그와 같은 표현(표집 영향 가능)",
                               "인스타 2026 수록량 급증(비중 비교)" if m == "인스타" else "")
    sens = []
    for t, x in keep.iterrows():
        if not (x["RISING_B"] or x["신규관측"]):
            sens.append("비교불가(트랙 B 미충족)")
            continue
        fails = int(not x["RISING_C"] and not x["신규관측"]) + int(not (x["RISING_1~9월"] or x["신규_1~9월"]))
        sens.append("안정" if fails == 0 else ("조건부" if fails == 1 else "불안정"))
    keep["민감도안정성"] = sens
    keep["2023보완수집민감"] = np.where(keep["RISING_B"] & ~keep["RISING_C"], "2023 보완수집 민감(C에서 미충족)",
                                 np.where(~keep["RISING_B"] & keep["RISING_C"] & ~keep["신규관측"], "C에서만 충족", ""))
    keep["유형"] = np.where(keep["RISING_B"], "RISING", np.where(keep["신규관측"], "신규 관측", np.where(keep["단일언론사후보"], "단일 언론사 후보", "C 민감도 후보")))
    keep["지지조건"] = f"현재 ≥{CONFIG['MIN_SUPPORT_CONTENTS']}건·관측월 ≥{CONFIG['RISING_MIN_MONTHS']}·출처 ≥{CONFIG['NEWS_MIN_PRESS'] if m == '뉴스' else CONFIG['MIN_SUPPORT_SOURCES']}"
    keep["선정근거"] = np.where(keep["유형"] == "신규 관측", "2024·2025 동기간 0건 → 2026 " + keep["현재등장수"].astype(int).astype(str) + "건",
                          "비중 " + (keep["기준등장비중"] * 1000).round(2).astype(str) + "‰ → " + (keep["현재등장비중"] * 1000).round(2).astype(str) + "‰ (×" + keep["비중비"].round(2).astype(str) + ")")
    keep["미충족사유"] = np.where(keep["RISING_B"] | keep["신규관측"], "", np.where(keep["단일언론사후보"], "언론사 1곳", "트랙 B 기준 미충족(C에서만 충족)"))
    out = type_common(m, keep, "Y2026", "B")
    out = out.sort_values(["유형", "천건당언급증가"], ascending=[True, False])
    out["유형내순위"] = out.groupby("유형").cumcount() + 1
    return out


HOT = {m: hot_table(m) for m in MEDIA}
RISE = {m: rising_sheet(m) for m in MEDIA}
for m in MEDIA:
    print(f"── {m} HOT:", ", ".join(HOT[m].loc[HOT[m]["HOT"], "표준관심대상"].head(12)))
    print(f"   RISING:", ", ".join(RISE[m].loc[RISE[m]["유형"] == "RISING", "표준관심대상"].head(12)))
    print(f"   신규:", ", ".join(RISE[m].loc[RISE[m]["유형"] == "신규 관측", "표준관심대상"].head(8)))

In [ ]:
def steady_table(m: str) -> pd.DataFrame:
    yb = YS[(m, "B", "")]
    years = sorted(yb)
    n = pd.concat({y: yb[y]["등장콘텐츠수"] for y in years}, axis=1).fillna(0)
    s = pd.concat({y: yb[y]["등장비중"] for y in years}, axis=1).fillna(0)
    src = pd.concat({y: yb[y]["고유출처수"] for y in years}, axis=1).fillna(0)
    ok_years = (n >= CONFIG["STEADY_MIN_PER_YEAR"]).sum(axis=1)
    total = n.sum(axis=1)
    cand = n[(ok_years >= CONFIG["STEADY_MIN_YEARS"]) & (total >= CONFIG["STEADY_MIN_TOTAL"]) & (src.max(axis=1) >= CONFIG["MIN_SUPPORT_SOURCES"])].index
    cv = (s.std(axis=1, ddof=0) / s.mean(axis=1)).reindex(cand)
    c, _ = period_rows(m, "A_전체", "B")
    c = c[c["target"].isin(cand)]
    mspread = c.groupby("target")["month"].nunique() / c["month"].nunique()
    topshare = c.groupby(["target", "source"]).size().groupby(level=0).max() / c.groupby("target").size()
    df = pd.DataFrame(index=cand)
    df["충족연도수"] = ok_years.reindex(cand)
    df["연도별등장수_원장참조"] = n.reindex(cand).astype(int).astype(str).agg("/".join, axis=1) + f" ({'/'.join(map(str, years))})"
    df["연도별비중_원장참조"] = (s.reindex(cand) * 1000).round(2).astype(str).agg("/".join, axis=1) + "‰"
    df["전체등장수"] = total.reindex(cand).astype(int)
    df["비중변동지표"] = cv.round(3)
    df["연도내월분산"] = mspread.reindex(cand).round(3)
    df["상투반복점검"] = np.where(topshare.reindex(cand) >= CONFIG["TOP_SOURCE_MAX_SHARE"], "특정 계정 의존(최대출처점유율 ≥ 0.5)", "")
    df["STEADY"] = (df["비중변동지표"] <= CONFIG["STEADY_MAX_CV"]) & (df["상투반복점검"] == "")
    df["선정근거"] = np.where(df["STEADY"], "변동계수 " + df["비중변동지표"].astype(str) + ", " + df["충족연도수"].astype(str) + "개 연도 충족", "")
    df["미충족사유"] = np.where(df["STEADY"], "", np.where(df["상투반복점검"] != "", "특정 계정 의존", "연도별 비중 변동 큼(후보)"))
    out = type_common(m, df, "A_전체", "B")
    out["기간시작"] = f"{years[0]}-01-01"
    return out.sort_values(["STEADY", "전체등장수"], ascending=False)


STEADY = {m: steady_table(m) for m in MEDIA}
for m in MEDIA:
    print(m, "STEADY:", int(STEADY[m]["STEADY"].sum()), "/ 후보", len(STEADY[m]), "→", ", ".join(STEADY[m].loc[STEADY[m]["STEADY"], "표준관심대상"].head(10)))

In [ ]:
def seasonal_table(m: str) -> pd.DataFrame:
    """완전 관측·동일 표집 계절만 비교. 계절 지수 = 그 계절 비중 ÷ 대상의 전체 완전 계절 비중."""
    pids = [p for p in PER.index if p.startswith("S") and PER.at[p, "기간완전성"] == "완전"
            and (m == "뉴스" or str(PER.at[p, "SNS_비교가능여부"]).startswith("가능"))]
    stats = {}
    for pid in pids:
        c, k = period_rows(m, pid, "B")
        if k.empty:
            continue
        g = c.groupby("target").agg(n=("content_id", "nunique"), src=("source", "nunique"))
        g["den"] = k["content_id"].nunique()
        stats[pid] = g
    if not stats:
        return pd.DataFrame()
    n = pd.concat({p: v["n"] for p, v in stats.items()}, axis=1).fillna(0)
    den = pd.Series({p: v["den"].iloc[0] for p, v in stats.items()})
    share = n / den
    pooled = n.sum(axis=1) / den.sum()
    idx = share.div(pooled, axis=0)
    season = pd.Series({p: PER.at[p, "계절연도"][-2:] for p in stats})
    rows = []
    ok_t = set(report_list(T09[m])["target"])
    for t in n.index[(n.sum(axis=1) >= 6) & n.index.isin(ok_t)]:
        best = {}
        for p in stats:
            if n.at[t, p] >= CONFIG["SEASONAL_MIN_PER_SEASON"] and idx.at[t, p] >= CONFIG["SEASONAL_MIN_INDEX"]:
                best.setdefault(season[p], []).append(p)
        if not best:
            continue
        s_best, ps = max(best.items(), key=lambda kv: len(kv[1]))
        rows.append((t, s_best, len(ps), ",".join(ps)))
    df = pd.DataFrame(rows, columns=["target", "피크계절", "반복확인완전연도수", "확인계절연도"]).set_index("target")
    if df.empty:
        return df
    df["SEASONAL"] = df["반복확인완전연도수"] >= CONFIG["SEASONAL_MIN_YEARS"]
    df = df[df["반복확인완전연도수"] >= 1]
    df["계절별분모_원장참조"] = "; ".join(f"{PER.at[p, '계절연도']}={int(den[p])}" for p in stats)
    df["계절별비중_원장참조"] = [("; ".join(f"{PER.at[p, '계절연도']}={share.at[t, p]*1000:.2f}‰" for p in stats)) for t in df.index]
    df["기간완전성점검"] = "완전 관측 계절만 사용" + ("" if m == "뉴스" else "(SNS: 2023 보완 구간과 겹치는 계절 제외)")
    df["행사일정근거"] = np.where(df.index.map(lambda t: AREA_MAP.get(t) == "문화·콘텐츠·스포츠"), "축제·행사형 가능(개최 시기 확인 필요)", "")
    df["선정근거"] = np.where(df["SEASONAL"], df["피크계절"] + " 지수 ≥ " + str(CONFIG["SEASONAL_MIN_INDEX"]) + " (" + df["확인계절연도"] + ")", "")
    df["미충족사유"] = np.where(df["SEASONAL"], "", "반복 확인 1개 연도뿐")
    out = type_common(m, df, "A_전체", "B")
    return out.sort_values(["SEASONAL", "등장콘텐츠수"], ascending=False)


SEASONAL = {m: seasonal_table(m) for m in MEDIA}
for m in MEDIA:
    s = SEASONAL[m]
    print(m, "SEASONAL:", int(s["SEASONAL"].sum()) if len(s) else 0, "→",
          ", ".join(s.loc[s["SEASONAL"], "표준관심대상"].head(10) + "(" + s.loc[s["SEASONAL"], "피크계절"].head(10) + ")") if len(s) else "")

In [ ]:
def event_table(m: str) -> pd.DataFrame:
    """EVENT/VIRAL 후보: 단일콘텐츠형(SNS 반응 상위 1개 기여율), 사건형(연간 한 달 집중), 연례 반복은 판정유보."""
    base = report_list(T09[m]).set_index("target")     # 최소지지 충족·분야 분류 대상만 점검
    c, _ = period_rows(m, "A_전체", "A")
    c = c[c["target"].isin(base.index)].drop_duplicates(["target", "content_id"])
    c = c.assign(y=c["date"].dt.year, mo=c["date"].dt.month)
    # 월별 수록량 차이(특히 인스타 2026년 9~10월 편중)를 지우기 위해 '그 달 등장비중'으로 집중률을 계산
    k = CONT[m][in_period(CONT[m]["date"], CONFIG["COMMON_START"], CONFIG["COMMON_END"])]
    den_ym = k.groupby([k["date"].dt.year.rename("y"), k["date"].dt.month.rename("mo")])["content_id"].nunique()
    by_y = c.groupby(["target", "y"]).size()
    by_ym = c.groupby(["target", "y", "mo"]).size()
    sh = by_ym / den_ym.reindex(pd.MultiIndex.from_arrays([by_ym.index.get_level_values(1), by_ym.index.get_level_values(2)])).values
    peak = sh.groupby(level=[0, 1]).max()
    peak_mo = sh.groupby(level=[0, 1]).idxmax().map(lambda x: x[2])
    conc = (peak / sh.groupby(level=[0, 1]).sum()).rename("집중률").to_frame()
    conc["n_y"] = by_y
    conc["peak_mo"] = peak_mo
    conc = conc[(conc["n_y"] >= 2 * CONFIG["MIN_SUPPORT_CONTENTS"]) & (conc["집중률"] >= CONFIG["EVENT_MONTH_SHARE"])]
    rows = {}
    for (t, y), r in conc.iterrows():
        rows.setdefault(t, []).append((int(y), int(r["peak_mo"]), float(r["집중률"])))
    out = []
    for t, lst in rows.items():
        mos = [mo for _, mo, _ in lst]
        annual = max(Counter(mos).values()) >= 2
        y, mo, sh = max(lst, key=lambda x: x[2])
        out.append((t, "판정유보(연례 행사 가능)" if annual else "사건형", f"{y}-{mo:02d}", round(sh, 3), "예" if annual else "아니오"))
    ev = pd.DataFrame(out, columns=["target", "세부유형", "피크월", "피크월콘텐츠기여율", "연례반복여부"]).set_index("target")
    if m != "뉴스":
        top1 = base[["조회수상위1콘텐츠기여율", "좋아요상위1콘텐츠기여율"]].max(axis=1)
        vir = base[top1 >= CONFIG["EVENT_TOP1_SHARE"]].index
        for t in vir:
            if t in ev.index:
                ev.at[t, "세부유형"] = ev.at[t, "세부유형"] + "+단일콘텐츠형"
            else:
                ev.loc[t] = ["단일콘텐츠형", "", np.nan, "아니오"]
        ev["상위1콘텐츠기여율"] = top1.reindex(ev.index).round(3)
    if ev.empty:
        return ev
    rep = T09[m].set_index("target")["대표콘텐츠ID"]
    titles = DF[m].set_index("content_id")
    ttl = (titles["title"].where(titles["title"] != "", titles["body"])).fillna("").str.replace(r"\s+", " ", regex=True).str[:80]
    score = CONT[m].set_index("content_id")["대표반응점수"]
    ym = c["date"].dt.strftime("%Y-%m")
    ctx = []
    for t in ev.index:
        pm = ev.at[t, "피크월"]
        if isinstance(pm, str) and pm:
            ids = c.loc[(c["target"] == t) & (ym == pm), "content_id"]
            best = ids.iloc[0] if m == "뉴스" else score.reindex(ids).idxmax() if score.reindex(ids).notna().any() else ids.iloc[0]
        else:
            best = rep.get(t)
        ctx.append(f"[{pm or '전체'}] " + ttl.get(best, ""))
    ev["사건맥락근거"] = ctx
    ev["관련사건행사"] = "검토 필요(사건맥락근거의 대표 콘텐츠 참조)"
    ev["단일출처점검"] = np.where(base["최대출처점유율"].reindex(ev.index) >= CONFIG["TOP_SOURCE_MAX_SHARE"], "단일 출처 의존", "")
    ev["EVENT_VIRAL"] = ev["세부유형"].str.contains("사건형|단일콘텐츠형") & ~ev["세부유형"].str.startswith("판정유보")
    ev["선정근거"] = np.where(ev["피크월"] != "", "피크월 " + ev["피크월"].astype(str) + " 집중률 " + ev["피크월콘텐츠기여율"].astype(str), "") \
                    + np.where(ev.get("상위1콘텐츠기여율", pd.Series(np.nan, index=ev.index)).fillna(0) >= CONFIG["EVENT_TOP1_SHARE"], " / 상위 1개 콘텐츠 반응 기여율 ≥ 50%", "")
    ev["미충족사유"] = np.where(ev["EVENT_VIRAL"], "", "연례 반복(같은 달 집중 2개 연도 이상)")
    out = type_common(m, ev, "A_전체", "A")
    return out.sort_values(["EVENT_VIRAL", "등장콘텐츠수"], ascending=False)


EVENT = {m: event_table(m) for m in MEDIA}
for m in MEDIA:
    e = EVENT[m]
    print(m, "EVENT/VIRAL:", int(e["EVENT_VIRAL"].sum()), "/ 판정유보", int((~e["EVENT_VIRAL"]).sum()))

## 13. 매체 비교와 대표 콘텐츠 (20, 21번 시트)

- **매체 비교**: 세 매체 상위(보고용 기준 충족) 대상의 합집합에 대해 매체별 등장 수·분모·비중과 맥락(함께 나온 보조어)을 나란히 둡니다. 원시 건수를 더한 통합 순위는 만들지 않습니다. 방송사 채널 영상과 같은 방송사 뉴스는 독립된 확산 증거로 보지 않도록 유튜브 출처 중 언론·방송 비중을 함께 적습니다.
- **대표 콘텐츠**: 분야×매체마다 보고용 상위 대상의 대표 콘텐츠를 뽑고, 시군 균형을 위해 시군별 상위 대상의 대표 콘텐츠를 더해 20~30건을 만듭니다. SNS는 원 링크, 뉴스는 원본 레코드ID로 연결하고 근거 원문 인용을 붙입니다.

In [ ]:
def cross_media() -> pd.DataFrame:
    tops = {m: report_list(T09[m]).head(CONFIG["CROSS_TOP_K"]) for m in MEDIA}
    union = sorted(set().union(*[set(v["target"]) for v in tops.values()]))
    rows = []
    X = {m: T09[m].set_index("target") for m in MEDIA}
    yt_type = CONT["유튜브"].set_index("content_id")["출처유형"]
    yt_ids = CT["유튜브"][CT["유튜브"]["target"].isin(union)].groupby("target")["content_id"].agg(lambda s: list(set(s)))
    for t in union:
        r = {"대상ID": target_id(t), "표준관심대상": t, "분야": AREA_MAP.get(t, "미분류(검토필요)"),
             "지역": HOME["대상소재시군"].get(t, "") or "다수·미확정",
             "공통기간시작": CONFIG["COMMON_START"], "공통기간종료": CONFIG["COMMON_END"]}
        present = 0
        for m in MEDIA:
            x = X[m]
            if t in x.index:
                r[f"{m}등장수"], r[f"{m}분모"], r[f"{m}비중"] = int(x.at[t, "등장콘텐츠수"]), int(x.at[t, "유효콘텐츠분모"]), x.at[t, "등장비중"]
                r[f"{m}맥락"] = x.at[t, "관심맥락"]
                r[f"{m}순위"] = int(x.at[t, "빈도순위"])
                present += int(t in set(tops[m]["target"]))
            else:
                r[f"{m}등장수"], r[f"{m}분모"], r[f"{m}비중"], r[f"{m}맥락"], r[f"{m}순위"] = 0, int(T09[m]["유효콘텐츠분모"].iloc[0]), 0.0, "", np.nan
        r["공통특화구분"] = {3: "3개 매체 공통", 2: "2개 매체 공통", 1: "단일 매체 특화"}[max(present, 1)]
        ids = yt_ids.get(t, [])
        share = yt_type.reindex(ids).eq("언론·방송 보도").mean() if len(ids) else np.nan
        r["출처독립성"] = f"유튜브 등장 영상 중 언론·방송 채널 {share:.0%}" if pd.notna(share) else "유튜브 미등장"
        r["표집비교가능성"] = "매체별 수집 방식이 달라 원시 건수 비교 불가, 비중만 참고"
        r["매체별순위_원장참조"] = "/".join("-" if pd.isna(r[f"{m}순위"]) else str(int(r[f"{m}순위"])) for m in MEDIA) + " (뉴스/인스타/유튜브)"
        r["근거ID"] = "07_키워드근거_*.csv(대상ID)"
        rows.append(r)
    out = pd.DataFrame(rows)
    for k, v in track_meta("뉴스", "A").items():
        if k in ("분석트랙", "해석제약"):
            out[k] = v
    out["보완수집포함여부"] = "SNS 포함"
    out["행별출처확인수준"] = "구간 정의(작성일 기준)로 확인"
    out["비교그룹ID"] = "매체별 전체 수집층"
    out["선택방식"] = "매체별 상이"
    out["비교가능판정"] = "조건부(비중·맥락 비교만)"
    return out.sort_values(["공통특화구분", "뉴스등장수"], ascending=[True, False])


T20 = cross_media()
print(T20["공통특화구분"].value_counts())
T20[T20["공통특화구분"] == "3개 매체 공통"][["표준관심대상", "분야", "뉴스등장수", "인스타등장수", "유튜브등장수", "출처독립성"]].head(15)

In [ ]:
def quote_of(m: str, cid: str, t: str) -> tuple:
    e = EVID[m]
    x = e[(e["content_id"] == cid) & (e["target"] == t)]
    if x.empty:
        return "", "", ""
    x = x.iloc[0]
    return x["근거필드"], x["원문인용"], x["src"]


def best_content(m: str, t: str, sg: str = "") -> tuple:
    """대표 콘텐츠: 연결등급 A(대상 이름에 지명 또는 인용 앞뒤 40자 안 지명)이고 단일 귀속인 콘텐츠 중
    SNS는 대표반응점수, 뉴스는 기사군 크기·최신 순으로 고릅니다. 없으면 B, 그래도 없으면 집계표의 대표콘텐츠."""
    r = REL[m][(REL[m]["target"] == t) & ~REL[m]["다중귀속여부"].astype(bool)]
    if sg:
        r = r[r["최종귀속시군"] == sg]
    for g in ("A", "B"):
        x = r[r["연결등급"] == g]
        if len(x):
            cu = CONT[m].set_index("content_id")
            ids = x["content_id"].unique()
            if m == "뉴스":
                rk = DF[m].set_index("content_id")["rep_key"]
                sz = rk.reindex(ids).map(rk.value_counts())
                order = pd.DataFrame({"sz": sz.values, "d": cu["date"].reindex(ids).values}, index=ids).sort_values(["sz", "d"], ascending=False)
            else:
                order = cu["대표반응점수"].reindex(ids).sort_values(ascending=False).to_frame()
            return order.index[0], g
    return T09[m].set_index("target").at[t, "대표콘텐츠ID"], ""


def representative_cases(n_target: int = 30) -> pd.DataFrame:
    picks = []
    for ar in AREAS:
        for m in MEDIA:
            top = report_list(T09[m])
            top = top[top["주분야"] == ar]
            if top.empty:
                picks.append((m, None, ar, "", "해당 분야 보고용 대상 없음"))
                continue
            picks.append((m, top.iloc[0]["target"], ar, "", f"{m} {ar} 등장 1위"))
    seen = {(m, t) for m, t, *_ in picks}
    for sg in SIGUN:
        best = None
        for m in ("인스타", "유튜브", "뉴스"):
            x = T10[(T10["매체"] == m) & (T10["지역"] == sg) & (T10["기간유형"] == "공통기간 전체")]
            x = report_list(x)
            x = x[~x.apply(lambda r: (m, r["target"]) in seen, axis=1)]
            if not x.empty:
                best = (m, x.iloc[0]["target"], x.iloc[0]["주분야"], sg, f"{sg} {m} 등장 1위(앞 선정과 중복 제외)")
                break
        if best:
            picks.append(best)
            seen.add((best[0], best[1]))
        if len(picks) >= n_target + 6:
            break
    rows = []
    for i, (m, t, ar, sg, why) in enumerate(picks, 1):
        if t is None:
            rows.append({"대표사례ID": f"R{i:02d}", "매체": m, "분야": ar, "대표선정사유": why, "주의사항": "공란 사유: 보고용 기준 충족 대상 없음"})
            continue
        x = T09[m].set_index("target").loc[t]
        cid, _g = best_content(m, t, sg)
        d = DF[m].set_index("content_id").loc[cid]
        fld, q, src = quote_of(m, cid, t)
        reg = REL[m][(REL[m]["content_id"] == cid) & (REL[m]["target"] == t)]
        rows.append({
            "대표사례ID": f"R{i:02d}", "매체": m, "콘텐츠ID": cid, "대상ID": target_id(t), "표준관심대상": t,
            "지역": sg or ",".join(reg["최종귀속시군"].unique()), "분야": ar, "대표콘텐츠선정": "연결등급 A·단일 귀속 중 반응(뉴스는 기사군) 최상위" if _g == "A" else ("연결등급 B 중 최상위(A 없음)" if _g else "집계표 대표 콘텐츠"),
            "게시검색표시일": d["date"].strftime("%Y-%m-%d"), "날짜출처": {"뉴스": "검색표시날짜", "인스타": "작성일(KST)", "유튜브": "게시일(KST)"}[m],
            "제목": (d["title"] or d["body"] or "")[:120].replace("\n", " "),
            "링크_제공시": d["url"] if m != "뉴스" else "", "뉴스원본레코드ID": d["원본ID"] if m == "뉴스" else "",
            "조회수": d["조회수"], "좋아요": d["좋아요"], "댓글수": d["댓글수"],
            "반응상태": f"조회수 {d['조회수상태']} / 좋아요 {d['좋아요상태']} / 댓글 {d['댓글수상태']}",
            "근거필드": fld, "원문인용": q, "관심맥락": x["관심맥락"], "대표선정사유": why,
            "연결등급": ",".join(reg["연결등급"].unique()),
            "주의사항": "대표 콘텐츠는 대상 포함 콘텐츠 중 하나이며 대상 전체의 평가·방문을 뜻하지 않음",
        })
    out = pd.DataFrame(rows)
    out["분석트랙"] = "A"
    out["보완수집포함여부"] = np.where(out["매체"] == "뉴스", "해당없음", "포함")
    out["행별출처확인수준"] = np.where(out["매체"] == "뉴스", "단일 수집", "구간 정의(작성일 기준)로 확인")
    grp_all = pd.concat([CONT[m].set_index("content_id")["비교그룹ID"] for m in MEDIA])
    out["비교그룹ID"] = out["콘텐츠ID"].map(grp_all).fillna("")
    out["선택방식"] = out["비교그룹ID"].map(lambda g: SELECT_RULE.get(g, ("", ""))[0])
    out["비교가능판정"] = "해당없음(사례)"
    out["해석제약"] = "단일 사례"
    return out


T21 = representative_cases()
print(len(T21), "건")
T21[["대표사례ID", "매체", "표준관심대상", "지역", "분야", "게시검색표시일", "원문인용"]].head(30)

## 14. 수집 편향 민감도 (과업지시서 17장, 28번 시트와 부록 시트)

**28_보완수집민감도.** 같은 대상에 대해 세 가지 범위를 나란히 비교합니다.
- **전체포함**: 공통기간 전체(2023 보완분 포함, 트랙 A)
- **2023제외**: 2024-01-01~2026-10-26
- **동일수집층**: 기본 구간·기본 수집만(인스타 IG-기본, 유튜브 YT-기본순-기본; 뉴스는 전체)

유형 열에는 각 범위에서의 RISING 판정(전체포함 = 트랙 C 기준 2023~2025 평균, 2023제외·동일수집층 = 트랙 B 기준)을 적습니다. 상위 목록 포함이 바뀌거나 RISING 판정이 바뀌면 '2023 보완수집 민감'으로 표시합니다.

**29_기타민감도(부록).** 1~9월 동기간, 반복군 기준, 출처 유형(일반 이용자·크리에이터 중심), 유튜브 타임라인 댓글 출처 제외, 합계 반응 vs 상위 1개 콘텐츠 제외 반응.

In [ ]:
def scope_stats(m: str, scope: str) -> pd.DataFrame:
    c, k = period_rows(m, "A_전체", "A")
    if scope == "2023제외":
        c, k = c[c["date"] >= "2024-01-01"], k[k["date"] >= "2024-01-01"]
    elif scope == "동일수집층":
        c, k = c[layer_mask(m, c, "B")], k[layer_mask(m, k, "B")]
    df = aggregate(m, c, k, [])
    df = support_flags(m, df)
    df = rank_within(df, [])
    return df.set_index("target")


def sensitivity_2023(m: str, top_n: int = 100) -> pd.DataFrame:
    S = {s: scope_stats(m, s) for s in ("전체포함", "2023제외", "동일수집층")}
    focus = list(report_list(S["전체포함"].reset_index()).head(top_n)["target"])
    focus += [t for t in RISE[m]["target"] if t not in focus]
    ris = RIS[m]
    rows = []
    for i, t in enumerate(focus, 1):
        r = {"민감도ID": f"{m[0]}S{i:05d}", "매체": m, "형식": "전체", "지역": "전체", "분야": AREA_MAP.get(t, "미분류(검토필요)"),
             "대상ID": target_id(t), "표준관심대상": t, "기간시작": CONFIG["COMMON_START"], "기간종료": CONFIG["COMMON_END"],
             "비교그룹ID": ",".join(sorted(B_GROUPS[m]))}
        for s, df in S.items():
            if t in df.index:
                x = df.loc[t]
                r[f"{s}_등장수"], r[f"{s}_분모"], r[f"{s}_비중"] = int(x["등장콘텐츠수"]), int(x["유효콘텐츠분모"]), x["등장비중"]
                r[f"{s}_빈도순위"] = int(x["빈도순위"])
                r[f"{s}_반응점수중앙값"], r[f"{s}_반응유효N"] = x["대표반응점수중앙값"], x["반응점수유효N"]
            else:
                r[f"{s}_등장수"], r[f"{s}_분모"], r[f"{s}_비중"] = 0, int(df["유효콘텐츠분모"].iloc[0]), 0.0
                r[f"{s}_빈도순위"] = np.nan
                r[f"{s}_반응점수중앙값"] = r[f"{s}_반응유효N"] = np.nan
        x = ris.loc[t] if t in ris.index else None
        r["전체포함_유형"] = ("RISING" if x is not None and x["RISING_C"] else "") if x is not None else "비교불가(2026 미등장)"
        b = ("RISING" if x["RISING_B"] else ("신규 관측" if x["신규관측"] else "")) if x is not None else "비교불가(2026 미등장)"
        r["2023제외_유형"] = b
        r["동일수집층_유형"] = b
        ranks = [r[f"{s}_빈도순위"] for s in S]
        r["순위변동"] = (np.nanmax(ranks) - np.nanmin(ranks)) if any(pd.notna(ranks)) else np.nan
        incl = [pd.notna(v) and v <= top_n for v in ranks]
        r["상위목록포함변경"] = "변경" if len(set(incl)) > 1 else "유지"
        r["유형변경"] = "변경" if len({r["전체포함_유형"], r["2023제외_유형"]}) > 1 else "유지"
        sens = r["상위목록포함변경"] == "변경" or r["유형변경"] == "변경"
        r["2023보완수집민감여부"] = "2023 보완수집 민감" if sens else ""
        r["비교불가사유"] = "" if x is not None else "2026 동기간 미등장"
        r["민감도안정성"] = "비교불가" if x is None and r["2023제외_등장수"] == 0 else ("조건부" if sens else "안정")
        r["해석제약"] = "수집 비중 정규화만으로 선택 편향이 제거되지 않음"
        r["근거집계ID"] = f"09_매체별키워드전체/{target_id(t)}"
        rows.append(r)
    return pd.DataFrame(rows)


t0 = time.time()
T28 = pd.concat([sensitivity_2023(m) for m in MEDIA], ignore_index=True)
print(f"28_보완수집민감도 {len(T28):,}행 ({time.time()-t0:.0f}초)")
T28.groupby("매체")[["상위목록포함변경", "유형변경", "민감도안정성"]].agg(lambda s: s.value_counts().to_dict())

In [ ]:
def other_sensitivity() -> pd.DataFrame:
    """부록 29_기타민감도: 기준 결과와 대안 결과의 상위 30 유지 여부·값 차이."""
    rows = []

    def compare(m, kind, base: pd.Series, alt: pd.Series, note: str, top: int = 30):
        bt = base.sort_values(ascending=False).head(top)
        at = alt.sort_values(ascending=False).head(top)
        brank = base.rank(ascending=False, method="min")
        arank = alt.rank(ascending=False, method="min")
        for t in bt.index.union(at.index):
            rows.append({"민감도유형": kind, "매체": m, "표준관심대상": t, "대상ID": target_id(t),
                         "기준값": base.get(t, 0), "비교값": alt.get(t, 0),
                         "기준순위": brank.get(t, np.nan), "비교순위": arank.get(t, np.nan),
                         "상위유지": "유지" if (t in bt.index and t in at.index) else ("기준에만" if t in bt.index else "비교에만"),
                         "비고": note})
    for m in MEDIA:
        rl = report_list(T09[m]).set_index("target")
        # 반복군 기준
        compare(m, "원 게시물 수 vs 반복군 기준", rl["등장콘텐츠수"], rl["반복군보정등장수"], "반복군 = 뉴스 기사군 / SNS 동일 본문·출처 반복 1건")
        # RISING 동기간 vs 1~9월
        r = RIS[m]
        b = r.loc[r["RISING_B"], "천건당언급증가"]
        a = r.loc[r["RISING_1~9월"], "천건당언급증가"] if "RISING_1~9월" in r else pd.Series(dtype=float)
        compare(m, "RISING: 1/1~10/26 vs 1~9월", b, a.reindex(a.index), "천건당언급 증가 기준 상위")
    for m in ("인스타", "유튜브"):
        c, k = period_rows(m, "A_전체", "A")
        general = k[~k["출처유형"].isin(["언론·방송 보도", "지자체·공공기관 공식 홍보", "기업·점포 홍보 및 협찬", "기업·점포 홍보(검토)"])]
        alt = aggregate(m, c[c["content_id"].isin(set(general["content_id"]))], general, []).set_index("target")
        base = report_list(T09[m]).set_index("target")["등장콘텐츠수"]
        compare(m, "모든 출처 vs 일반 이용자·크리에이터 중심", base, alt["등장콘텐츠수"].reindex(base.index.union(alt.index)).fillna(0),
                "공공·언론·업체·협찬 표기 콘텐츠 제외(인스타는 표기·검토유형 근거가 있는 것만 제외)")
        rl = report_list(T09[m]).set_index("target")
        col = "관련조회수합계" if m == "유튜브" else "관련좋아요합계"
        mx = "관련조회수최댓값" if m == "유튜브" else "관련좋아요최댓값"
        compare(m, "합계 반응 vs 상위 1개 콘텐츠 제외 반응", rl[col].fillna(0), (rl[col] - rl[mx]).fillna(0), f"{col} 기준")
    c = CT["유튜브"]
    desc_only = c[c["근거필드"] != "타임라인(댓글)"]
    base = c.groupby("target")["content_id"].nunique()
    alt = desc_only.groupby("target")["content_id"].nunique()
    focus = report_list(T09["유튜브"])["target"]
    compare("유튜브", "타임라인 전체 vs 설명란 출처만", base.reindex(focus).fillna(0), alt.reindex(focus).fillna(0), "근거가 댓글 타임라인뿐인 영상 제외")
    out = pd.DataFrame(rows)
    out.insert(0, "민감도ID", [f"O{i:05d}" for i in range(1, len(out) + 1)])
    return out


T29 = other_sensitivity()
T29.groupby(["민감도유형", "매체"])["상위유지"].agg(lambda s: f"유지 {int((s == '유지').sum())}/{int(s.isin(['유지', '기준에만']).sum())}")

## 15. 사전·검토 목록·검수·변경 이력 (04, 05, 22~26번 시트)

- **04_표준키워드사전**: 등장 3건 이상 대상의 표준명·원어 변형·대상유형·분야·지역 후보·상위 브랜드(지점 구분)·동일성 근거. 모두 '미승인(초기 기준)'입니다.
- **05_제외보조어사전**: 정당명·지역명·보조어·행위자(인명·기관)·지역 연결 불명 식품의 처리 기준과 등장 수. *콘텐츠 전체를 제외하지 않고 키워드만 주순위에서 뺐다*는 점을 열로 명시합니다.
- **22_검토오류목록**: 발주자 확인이 필요한 후보(미분류 상위, 자동 인명 판정, 동음이의 후보, 수집태그 대상의 RISING, 단일 출처 의존, 원문 미발견, 지역미확정 상위).
- **23_검수결과**: 자동 검산(원본 해시·중복 키·타임라인 중복·원문 인용 일치·결측 구분·행위자 단독 미포함)은 실측값으로, 독립 검수는 층화 표본과 판정 양식을 만들어 **미실시(발주자·별도 검토자 필요)** 로 둡니다. AI 검토를 사람 검수로 표시하지 않습니다.
- **26_변경이력**: 1차 결과(2026-11-04)의 매체별 상위 50개 키워드가 이번에 어떻게 처리됐는지(유지·맥락화·보조어 전환·제외)와 새로 상위에 든 대상.

In [ ]:
def target_kind(t: str, area: str) -> str:
    """대상유형(과업지시서 7장 예시 유형) — 이름 접미어와 분야로 정한 초기값."""
    k = normalize_unit(t)
    if area == "상품·먹거리":
        return "상호·브랜드" if re.search(r"(제과|제빵소|빵집|식당|옥$|집$|당$|상회|카페|베이커리)", k) else "메뉴·특산품"
    if area == "장소·공간":
        return "관광·생활 공간"
    if area == "문화·콘텐츠·스포츠":
        return "행사·축제·공연" if re.search(r"(축제|페스티벌|제$|대회|공연|박람회|엑스포|영화제|콘서트)", k) else "체험·활동·콘텐츠"
    if area == "기업·산업":
        return "기술·산업"
    if area == "정책·공공사업":
        return "정책·사업·제도"
    if area == "인물·기관·사회 현상":
        return "사회적 쟁점"
    return "미정(검토)"


def keyword_dictionary() -> pd.DataFrame:
    allc = pd.concat([CT[m][["target", "raw", "content_id"]].assign(매체=m) for m in MEDIA])
    n = allc.groupby("target")["content_id"].nunique()
    tg = n[n >= 3].index
    raws = allc[allc["target"].isin(tg)].assign(r=lambda x: x["raw"].astype(str).str.strip()).groupby("target")["r"] \
        .agg(lambda s: " | ".join(s.value_counts().head(5).index))
    ev = pd.concat([EVID[m][EVID[m]["원문일치검사"] == "일치"][["target", "content_id"]].assign(매체=m) for m in MEDIA]) \
        .drop_duplicates("target").set_index("target")
    syn_std = set(SYN_MAP.values())
    tset = set(tg)
    rows = []
    for t in tg:
        area = AREA_MAP.get(t, "미분류(검토필요)")
        parent = next((p for p in (t[:i] for i in range(len(t) - 2, 1, -1)) if p in tset and AREA_MAP.get(p) == area
                       and re.search(r"(점|본점|지점|분점|매장)$", t)), "")
        rows.append({
            "대상ID": target_id(t), "표준관심대상": t, "원어": raws.get(t, ""), "대상유형": target_kind(t, area),
            "상위대상ID": target_id(parent) if parent else "", "브랜드지점구분": "지점" if parent else "",
            "주분야": area, "보조분야": AREA_TABLE["보조분야"].get(t, "") if t in AREA_TABLE.index else "",
            "지역후보": HOME["대상소재시군"].get(t, ""), "포함상태": "포함" if area in AREAS else "검토",
            "동일성근거": "표현통합사전" if t in syn_std else "표기 정규화(공백·기호·대소문자·연도 접두)",
            "원문근거ID": (ev.at[t, "매체"] + ":" + ev.at[t, "content_id"]) if t in ev.index else "",
            "등장콘텐츠수_3매체합": int(n[t]),
            "사전버전": ANALYSIS_VERSION, "검토자": "규칙(자동)", "검토일": datetime.now().strftime("%Y-%m-%d"), "승인상태": "미승인(초기 기준)",
        })
    return pd.DataFrame(rows).sort_values("등장콘텐츠수_3매체합", ascending=False)


def exclusion_dictionary() -> pd.DataFrame:
    rule = {
        "정당명": ("주순위·분야·유형 모두 제외(정당명 단독)", "dict/제외_정당명.txt 부분 일치", "정당명이 들어간 표현", "정책·공약 대상은 별도 대상으로 유지"),
        "지역명": ("주순위 제외, 지역 메타데이터로 보존", "dict/지역명.txt 정확 일치, 하위 행정구역", "단독 지역명", "지역명이 일부인 고유명칭(전주비빔밥)은 유지"),
        "보조어": ("주순위 제외, 맥락 표지로 활용", "dict/보조어.txt, SNS일반어, 시군명+일반어, 범용어 자동 판정", "범용어·수집 유도 태그", "시군명+일반어 뒤의 상호·메뉴는 떼어 후보로 유지"),
        "지정인명": ("주순위 제외, 행위자 정보로 보존", "dict/제외_지정인명.txt(발주자 지정)", "이재명·윤석열·김관영·이원택", "발언 대상인 정책·사업은 유지"),
        "인명": ("주순위 제외, 행위자 정보로 보존(승인 필요)", "직함 동반 빈도 자동 판정, 수기 분류표 인물", "단독 인명", "인물의 작품·문화 활동은 검토 대상"),
        "기관·직함": ("주순위 제외, 기관/행위자 사전에 보존", "기관·직함 접미어로 끝나는 표현", "학교·대학·기관명 단독, 단체장 직함", "기관과 연결된 축제·프로그램·정책은 유지"),
        "주소·타지역": ("주순위 제외", "도로명 주소(○○길·○○로 + 번호), 시군+읍면 연속 표현, 원문에서 번지가 붙는 길·로, 도 밖 광역시", "업체·장소 안내 문구의 주소(학원농장길 150)", "주소가 아닌 길 이름(마실길·둘레길)과 장소 자체(학원농장)는 유지"),
        "선거구명": ("주순위 제외", "시군명+갑·을·병·정, ○○선거구", "전주병·익산갑·장수선거구", "선거·정책 쟁점(지방선거·선거구 획정 논의)은 대상명으로 남지 않으면 맥락으로만 반영"),
        "상호·지점명": ("주순위 제외", "지점을 뜻하는 '점'으로 끝나는 표현(일반명사 백화점·서점·반점 등 제외)", "전주한옥마을점·전주혁신점·○○본점", "상호 자체(풍년제과·이성당)는 유지"),
        "광고·홍보성": ("주순위 제외", "dict/제외_광고홍보어.txt + 해시태그 나열(지역명+같은 표현 4개 이상) 자동 판정 표현과 그 지역명 결합형", "전자담배·전북혁신도시전자담배·미용가위", "나열 콘텐츠는 콘텐츠 전체 분석 제외(02 S4b)"),
        "식품주제(지역연결불명)": ("주순위 제외, 식품 주제 보조·검토", "dict/지역연결필요식품.txt, 같은 콘텐츠에 시군명+식품명 없음", "치즈·사과·한우 단독", "임실치즈·장수사과처럼 명시 연결 시 표준 대상"),
    }
    parts = []
    for m in MEDIA:
        a = AUX[m]
        a = a[a["judge"].isin(rule)]
        g = a.groupby(["target", "judge"])["content_id"].nunique().rename("등장콘텐츠수").reset_index()
        parts.append(g.assign(매체=m))
    g = pd.concat(parts).groupby(["target", "judge"])["등장콘텐츠수"].sum().reset_index()
    g = g[(g["등장콘텐츠수"] >= 3) | g["judge"].isin(["정당명", "지정인명"])]
    g = g.sort_values("등장콘텐츠수", ascending=False)
    out = pd.DataFrame({
        "표현ID": [f"X{i:06d}" for i in range(1, len(g) + 1)], "원표현": g["target"].values, "표준표현": g["target"].values,
        "유형": g["judge"].values, "처리_주순위제외보조맥락화": g["judge"].map(lambda j: rule[j][0]).values,
        "콘텐츠전체제외여부": "아니오(키워드만 주순위 제외)", "규칙": g["judge"].map(lambda j: rule[j][1]).values,
        "예외조건": g["judge"].map(lambda j: rule[j][3]).values, "보존대상예시": g["judge"].map(lambda j: rule[j][2]).values,
        "등장콘텐츠수_3매체합": g["등장콘텐츠수"].values, "검토자": "규칙(자동)",
        "승인상태": np.where(g["judge"].isin(["인명"]), "승인 필요(자동 판정 인명)", np.where(g["judge"] == "지정인명", "발주자 지정", "미승인(초기 기준)")),
        "버전": ANALYSIS_VERSION,
    })
    return out


T04 = keyword_dictionary()
T05 = exclusion_dictionary()
print("04 사전", len(T04), "| 05 제외·보조", len(T05), T05["유형"].value_counts().to_dict())

In [ ]:
def review_list() -> pd.DataFrame:
    rows = []

    def add(typ, t, m, why, need, cid=""):
        rows.append({"유형": typ, "대상ID": target_id(t) if t else "", "표준관심대상": t, "매체": m, "콘텐츠ID": cid,
                     "원문근거": why, "현재처리": "", "필요확인사항": need})
    for m in MEDIA:
        top = T09[m][(T09[m]["최소지지충족"] == "충족")].head(400)
        for t in top.loc[~top["주분야"].isin(AREAS), "target"].head(60):
            add("미분류(상위 400 내)", t, m, f"등장 {int(top.set_index('target').at[t, '등장콘텐츠수'])}건", "6개 분야 중 주분야 확정 또는 보조어 전환")
        for t in top.loc[top["단일출처의존"], "target"].head(30):
            add("단일 출처 의존", t, m, f"최대출처점유율 {top.set_index('target').at[t, '최대출처점유율']:.2f}", "반복 홍보 여부 확인(원집계·반복군 보정 병기 중)")
        for t in RISE[m].loc[RISE[m]["표집변경가능성"].str.startswith("수집태그"), "target"].head(30):
            add("수집태그 대상의 RISING/신규", t, m, "수집 해시태그와 같은 표현", "새 현상인지 수집태그 영향인지 확인")
        miss = EVID[m][EVID[m]["원문일치검사"] == "미발견"]
        for t, n in miss["target"].value_counts().head(20).items():
            add("원문 인용 미발견", t, m, f"{n}건", "표기 변형(전각·이체자) 또는 정제 단계 변형 확인", miss.loc[miss["target"] == t, "content_id"].iloc[0])
        unk = REL[m][REL[m]["최종귀속시군"] == "지역미확정"]["target"].value_counts().head(20)
        for t, n in unk.items():
            add("지역미확정 상위", t, m, f"{n}건", "대상 소재지 확인(외부 확인은 발주자 승인 범위)")
    a = ACTOR_LOG[ACTOR_LOG["judge"] == "인명"].groupby("target")["콘텐츠수"].sum().sort_values(ascending=False).head(80)
    for t, n in a.items():
        add("인명 자동 판정(주순위 제외)", t, "전체", f"{n}건", "인명 여부·주순위 제외 승인")
    for t in [x for x in set().union(*[set(CT[m]["target"]) for m in MEDIA]) if re.search(r"^(장수|완주)(?!군)", normalize_unit(x))][:60]:
        add("동음이의 후보(장수·완주)", t, "전체", "시군 접미어 없는 '장수/완주' 시작 표현", "지명인지 일반어(오래 삶·끝까지 달림)인지 확인")
    out = pd.DataFrame(rows)
    out.insert(0, "오류ID", [f"E{i:05d}" for i in range(1, len(out) + 1)])
    out["집계ID"] = ""
    out["담당자"], out["발주자판정"], out["수정내용"], out["최종상태"], out["수정버전"] = "", "", "", "미해결", ANALYSIS_VERSION
    return out


T22 = review_list()
T22["유형"].value_counts()

In [ ]:
def qa_checks() -> pd.DataFrame:
    rows = []

    def add(item, target, actual, denom, ok, evidence="", follow=""):
        rows.append({"검수항목": item, "목표": target, "실측": actual, "분모": denom, "통과여부": "통과" if ok else "미통과",
                     "개발평가구분": "자동 검산", "모집단층": "전체", "표본추출법": "전수", "seed": "", "표본수": denom,
                     "독립검토주체": "분석 코드(자동)", "불확실성": "해당없음(전수)", "증빙파일": evidence, "후속조치": follow})
    after = {p.name: file_sha256(p) for p in (RAW_NEWS, RAW_SNS)}
    add("원본 파일 해시 작업 전후 동일", "100%", "동일" if after == INPUT_HASH else "변경됨", 2, after == INPUT_HASH, "00_안내(입력 해시)")
    for m in MEDIA:
        dup = REL[m].duplicated(["content_id", "target", "최종귀속시군"]).sum()
        add(f"{m}: 콘텐츠–대상–지역 중복 키 없음", "0건", f"{dup}건", len(REL[m]), dup == 0, f"06_지역근거_{m}.csv")
        dup2 = CT[m].duplicated(["content_id", "target"]).sum()
        add(f"{m}: 콘텐츠–대상 중복 없음(타임라인·제목·태그 다중 등장은 1건)", "0건", f"{dup2}건", len(CT[m]), dup2 == 0, f"07_키워드근거_{m}.csv")
        okq = (EVID[m]["원문일치검사"] == "일치").mean()
        add(f"{m}: 근거 인용이 원문 필드의 실제 부분 문자열", "100%(미발견은 검토 목록)", f"{okq:.2%}", len(EVID[m]), okq >= 0.95,
            f"07_키워드근거_{m}.csv", "미발견 행은 22_검토오류목록")
        bad = 0
        for i, r in EVID[m][EVID[m]["원문일치검사"] == "일치"].sample(min(2000, len(EVID[m])), random_state=CONFIG["SEED"]).iterrows():
            full = TEXT_IDX[m][r["content_id"]][0]
            if r["원문인용"] and r["원문인용"] not in full and not r["근거필드"].startswith("타임라인"):
                bad += 1
        add(f"{m}: 인용 문자열 원문 포함 재확인(표본 2,000)", "0건 불일치", f"{bad}건", 2000, bad == 0)
        top = report_list(T09[m]).head(100)
        leak = top["target"].map(lambda t: bool(actor_type(t)) or bool(_PARTY.search(normalize_unit(t))) or normalize_unit(t) in REGION_WORDS).sum()
        add(f"{m}: 보고용 상위 100에 행위자·정당·단독 지역명 없음", "0건", f"{leak}건", 100, leak == 0, "09_매체별키워드전체")
    yt_tl = CT["유튜브"]["content_id"].isin(set(TL_ITEMS)).sum()
    add("유튜브 타임라인 다중 행이 영상 빈도를 부풀리지 않음", "영상 1건으로 집계", "콘텐츠–대상 중복 0건(위 항목)", int(yt_tl), True, "07_키워드근거_유튜브.csv(타임라인구간수=보조)")
    for m in ("인스타", "유튜브"):
        z = DF[m]["좋아요상태"].value_counts().to_dict()
        add(f"{m}: 결측과 실제 0 구분", "숨김·미제공·적용불가를 0으로 치환하지 않음", str(z), len(DF[m]), True, "03_콘텐츠마스터")
    rows.append({"검수항목": "핵심 관심대상 후보 정밀도", "목표": "90% 이상(착수 협의 목표)", "실측": "미실시", "분모": 300, "통과여부": "미실시",
                 "개발평가구분": "최종 평가 표본", "모집단층": "매체×보고용 상위/경계/저빈도 고반응", "표본추출법": "층화 무작위", "seed": CONFIG["SEED"],
                 "표본수": 300, "독립검토주체": "발주자 또는 별도 검토자(미지정)", "불확실성": "검수 후 산출",
                 "증빙파일": "검수표본_관심대상.csv", "후속조치": "검토자 판정 입력 후 재집계"})
    rows.append({"검수항목": "확정 시군 귀속 정밀도", "목표": "95% 이상(착수 협의 목표)", "실측": "미실시", "분모": 300, "통과여부": "미실시",
                 "개발평가구분": "최종 평가 표본", "모집단층": "매체×연결등급×귀속확실성", "표본추출법": "층화 무작위", "seed": CONFIG["SEED"],
                 "표본수": 300, "독립검토주체": "발주자 또는 별도 검토자(미지정)", "불확실성": "검수 후 산출",
                 "증빙파일": "검수표본_시군귀속.csv", "후속조치": "검토자 판정 입력 후 재집계"})
    out = pd.DataFrame(rows)
    out.insert(0, "검수ID", [f"Q{i:03d}" for i in range(1, len(out) + 1)])
    return out


def review_samples():
    """독립 검수용 층화 표본(판정 칸은 비워 둠)."""
    rng = CONFIG["SEED"]
    a, b = [], []
    for m in MEDIA:
        rl = report_list(T09[m])
        strata = {"상위": rl.head(50), "경계": rl.iloc[150:400], "저빈도": T09[m][(T09[m]["등장콘텐츠수"].between(5, 15))]}
        if m != "뉴스":
            strata["저빈도고반응"] = T11[(T11["매체"] == m) & (T11["비교기간ID"] == "A_전체") & (T11["숨은관심대상후보"] == "예")]
        for s, df in strata.items():
            if df.empty:
                continue
            pick = df.sample(min(len(df), 25 if s != "상위" else 25), random_state=rng)
            for _, r in pick.iterrows():
                cid = r["대표콘텐츠ID"]
                fld, q, _ = quote_of(m, cid, r["target"])
                a.append({"매체": m, "층": s, "대상ID": target_id(r["target"]), "표준관심대상": r["target"], "주분야": r["주분야"],
                          "대표콘텐츠ID": cid, "근거필드": fld, "원문인용": q,
                          "판정_후보적절(Y/N)": "", "판정_표준명동일성(Y/N)": "", "판정_분야(Y/N)": "", "검토자": "", "메모": ""})
        rel = REL[m]
        for (g, cert), df in rel.groupby(["연결등급", "귀속확실성"]):
            pick = df.sample(min(len(df), 12), random_state=rng)
            for _, r in pick.iterrows():
                b.append({"매체": m, "연결등급": g, "귀속확실성": cert, "콘텐츠ID": r["content_id"], "표준관심대상": r["target"],
                          "최종귀속시군": r["최종귀속시군"], "귀속근거": r["귀속근거"], "원문인용": r["원문인용"],
                          "판정_시군귀속(Y/N)": "", "검토자": "", "메모": ""})
    return pd.DataFrame(a), pd.DataFrame(b)


T23 = qa_checks()
SAMPLE_TARGET, SAMPLE_REGION = review_samples()
print(T23[["검수항목", "실측", "통과여부"]].to_string())
print("검수 표본:", len(SAMPLE_TARGET), len(SAMPLE_REGION))

In [ ]:
METRIC_DEFS = [
    ("등장콘텐츠수", "정수", "건", "대상이 등장한 고유 콘텐츠 수(DF). 같은 콘텐츠의 제목·본문·태그·타임라인 중복 등장은 1건", "-", "-", "빈도 상위 = 도민 선호 아님"),
    ("문자열출현횟수_보조", "정수", "회", "원어가 원본 필드에 나온 횟수(TF) 합계", "-", "-", "보조 지표"),
    ("반복군보정등장수", "정수", "건", "뉴스 기사군(같은 제목·3일 이내) 수 / 인스타 동일 본문 묶음·출처 반복 생산 1건 / 유튜브 영상(출처 반복 1건)", "-", "-", "반복 홍보 영향 확인용"),
    ("기사군수", "정수", "건", "뉴스: 반복군보정등장수와 같음", "-", "SNS 적용불가", ""),
    ("유효콘텐츠분모", "정수", "건", "같은 매체·지역·기간·수집층의 유효 고유 콘텐츠 수", "-", "0이면 비중 산출불가", ""),
    ("분모정의", "문자", "-", "분모를 센 범위의 설명", "-", "-", ""),
    ("등장비중", "실수", "비율", "등장콘텐츠수 ÷ 유효콘텐츠분모", "유효콘텐츠분모", "분모 0이면 결측", "매체·층 간 원시 건수 대신 비교"),
    ("천건당언급", "실수", "건/1000", "등장비중 × 1000", "유효콘텐츠분모", "", ""),
    ("분야내콘텐츠분모", "정수", "건", "같은 층에서 같은 주분야 대상이 1개 이상 나온 콘텐츠 수", "-", "", ""),
    ("분야내등장비중", "실수", "비율", "등장콘텐츠수 ÷ 분야내콘텐츠분모", "분야내콘텐츠분모", "", "전체 비중과 바꿔 쓰지 않음"),
    ("고유출처수", "정수", "곳", "언론사(뉴스)·작성자 해시(인스타)·채널명(유튜브) 수", "-", "", ""),
    ("최대출처점유율", "실수", "비율", "가장 많이 게시한 출처의 콘텐츠 수 ÷ 등장콘텐츠수", "등장콘텐츠수", "", "0.5 이상이면 단일출처의존"),
    ("관측월수_대상", "정수", "개월", "대상이 등장한 서로 다른 달 수", "-", "", ""),
    ("관련조회수합계/중앙값/최댓값", "실수", "회", "대상 포함 콘텐츠의 조회수(유효·실제0만)", "조회수유효N", "숨김·미제공은 결측, 적용불가(인스타 사진·캐러셀, 뉴스)는 제외", "대상별 합계를 더해 전체 조회수·고유 이용자 수라고 하지 않음"),
    ("관련좋아요합계/중앙값/최댓값", "실수", "개", "대상 포함 콘텐츠의 좋아요(유효·실제0만)", "좋아요유효N", "좋아요숨김=예는 '숨김' 결측", ""),
    ("관련댓글수합계/중앙값/최댓값", "실수", "개", "대상 포함 콘텐츠의 댓글 수", "댓글수유효N", "", "댓글 많음 ≠ 긍정 호응"),
    ("조회수결측수/좋아요결측수/댓글수결측수", "정수", "건", "상태가 숨김·미제공인 콘텐츠 수", "-", "적용불가는 세지 않음", ""),
    ("조회수상위1콘텐츠기여율/좋아요상위1콘텐츠기여율", "실수", "비율", "최댓값 ÷ 합계", "합계", "합계 0이면 결측", "0.5 이상이면 단일 콘텐츠 의존"),
    ("대표반응점수", "실수", "0~100", "유튜브·릴스: (조회수 백분위+좋아요 백분위)/2, 사진·캐러셀: 좋아요 백분위. 백분위 = (평균순위−0.5)/N×100, 집단 = 플랫폼×형식×게시연도·분기(N≥30, 부족 시 연도)", "비교집단유효N", "한쪽 지표만 있으면 조회수전용·좋아요전용", "같은 플랫폼·형식 안에서만 해석, 선택 편향 교정 아님"),
    ("대표반응점수중앙값", "실수", "0~100", "대상 포함 콘텐츠 대표반응점수의 중앙값", "반응점수유효N", "", "상대 반응 순위 기준"),
    ("최소지지충족", "문자", "-", f"고유 콘텐츠 ≥ {CONFIG['MIN_SUPPORT_CONTENTS']} & 출처 ≥ {CONFIG['MIN_SUPPORT_SOURCES']}(뉴스 언론사 ≥ {CONFIG['NEWS_MIN_PRESS']})", "-", "", "미달은 소수근거·단일흥행 후보로 별도 표시"),
    ("빈도순위", "정수", "위", "층 안 등장콘텐츠수 내림차순(동점은 출처 수)", "-", "", ""),
    ("반응순위", "정수", "위", "지지 충족 대상 중 대표반응점수중앙값 내림차순", "-", "", ""),
    ("사분면", "문자", "-", "지지 충족 대상의 등장콘텐츠수·대표반응점수중앙값 중앙값 기준 4분면", "사분면후보수", "", "평균·상위사분위 기준 민감도 병기"),
    ("지역고유성비", "실수", "배", "해당 시군 등장비중 ÷ 다른 13개 시군 등장비중", "지역고유성분모", "다른 지역 0건이면 신규/비교불가", "높은 고유성 ≠ 지역 대표성"),
    ("비중비/비중차/천건당언급증가/건수차", "실수", "-", "현재(2026 동기간) 대 기준(2025 동기간) 비교", "현재분모·기준분모", "기준 0이면 비중비 결측(신규 관측)", "RISING은 등장 관심의 변화, 반응 증가 아님"),
    ("연결등급", "문자", "-", "A 명시(대상명 지명·인용 앞뒤 40자 지명), B 보조(같은 콘텐츠 다른 곳·대상 소재 추정·전북 광역), C 동반 출현(다중 지명 중 근접 지명 없음·수집태그뿐)", "-", "", "핵심 시군 순위는 A/B"),
]


def metric_defs() -> pd.DataFrame:
    return pd.DataFrame(METRIC_DEFS, columns=["필드명", "자료형", "단위", "정의및계산식", "분모", "결측규칙", "해석주의"])


def param_table() -> pd.DataFrame:
    notes = {"MIN_SUPPORT_CONTENTS": "과업지시서 15장 초기값", "MIN_SUPPORT_SOURCES": "과업지시서 15장 초기값", "RISING_MIN_RATIO": "16장 초기값",
             "RISING_MIN_PER1000_DELTA": "16장 초기값", "MIN_VIEWS_FOR_RATE": "13장 품질관리 초기값", "PCTL_MIN_GROUP": "14장",
             "EVENT_TOP1_SHARE": "16장 초기 점검 기준", "EVENT_MONTH_SHARE": "16장 초기 점검 기준", "BASE_YEAR": "17.3 트랙 B 기본 비교",
             "SUPP_SEGMENT_END": "수집기준 시트의 실제 경계(2023-10-30)"}
    rows = []
    for i, (k, v) in enumerate(CONFIG.items(), 1):
        rows.append({"설정ID": f"P{i:03d}", "설정명": k, "초기제안값": json.dumps(v, ensure_ascii=False) if not isinstance(v, str) else v,
                     "적용범위": "전체", "근거및주의": notes.get(k, ""), "승인값": "", "승인자": "", "승인일": "", "버전": ANALYSIS_VERSION})
    return pd.DataFrame(rows)


def change_log() -> pd.DataFrame:
    """1차 결과(output/csv_20261104_2146 또는 가장 최근 1차 결과)의 상위 50 → 이번 처리."""
    prev_dirs = sorted(d for d in OUT_DIR.glob("csv_2026110*") if (d / "3_빈도_뉴스.csv").exists())
    if not prev_dirs:
        return pd.DataFrame(columns=["변경ID"])
    prev = prev_dirs[-1]
    rows = []
    for m in MEDIA:
        old = pd.read_csv(prev / f"3_빈도_{m}.csv").head(50)
        new = report_list(T09[m]).set_index("target")
        aux = AUX[m].groupby("target")["judge"].agg(lambda s: s.value_counts().index[0])
        for _, r in old.iterrows():
            t = r["target"]
            if t in new.index:
                st, after, why = "유지", f"보고용 순위 {int(new.index.get_loc(t)) + 1}위", "구체 대상으로 유지(지지조건 충족)"
            elif t in set(T09[m]["target"]):
                x = T09[m].set_index("target").loc[t]
                st = "맥락화(분야 검토)" if x["주분야"] not in AREAS else "유지(보고용 조건 미충족)"
                after = f"전체 원장 {int(x['빈도순위'])}위"
                why = "단일출처의존" if x["단일출처의존"] else ("미분류" if x["주분야"] not in AREAS else "지지조건")
            elif t in aux.index:
                j = aux[t]
                st = "제외(행위자 단독)" if j in ("인명", "지정인명", "기관·직함") else ("보조어 전환" if j in ("보조어", "식품주제(지역연결불명)") else "제외")
                after, why = f"보조({j})", {"인명": "단독 인명(6.2)", "지정인명": "발주자 지정 인명(6.2)", "기관·직함": "기관·직함 단독(6.2)",
                                           "식품주제(지역연결불명)": "지역 연결 없는 식품명(6.3)"}.get(j, j)
            else:
                st, after, why = "통합·변경", "표준명 변경 또는 미등장", "표현 통합·식품 연결 등"
            rows.append({"매체": m, "기존키워드규칙": f"1차 상위 {int(r['순위'])}위: {t}", "변경후대상규칙": after, "유지맥락화보조제외": st, "변경사유": why})
        oldset = set(old["target"])
        for t in new.head(50).index:
            if t not in oldset:
                rows.append({"매체": m, "기존키워드규칙": "1차 상위 50 밖", "변경후대상규칙": f"보고용 순위 {int(new.index.get_loc(t)) + 1}위: {t}",
                             "유지맥락화보조제외": "신규 관심대상", "변경사유": "행위자·범용어 제외 후 상위 진입"})
    out = pd.DataFrame(rows)
    out.insert(0, "변경ID", [f"C{i:04d}" for i in range(1, len(out) + 1)])
    out["변경일"] = datetime.now().strftime("%Y-%m-%d")
    out["근거ID"] = "09_매체별키워드전체 / 05_제외보조어사전"
    out["변경전후집계영향"] = "빈도 기준: 1차 기사군·반복 상쇄 → 2차 고유 콘텐츠(DF), 반복군은 보조"
    out["승인자"], out["버전"] = "", ANALYSIS_VERSION
    return out


T24, T25, T26 = metric_defs(), param_table(), change_log()
T26["유지맥락화보조제외"].value_counts()

## 16. 자료 점검·전처리 집계와 결과 저장 (00~29번 시트, 원장 파일, manifest)

**저장 구조** (`output/v2_<실행시각>/`)
- `미지의전북_관심대상분석_v2_<실행시각>.xlsx`: 양식 29개 시트(00~28) + 부록 `29_기타민감도`. 모든 표는 병합 없는 단일 머리글·필터·첫 행 고정·숫자형 지표·ISO 날짜입니다. 빈 칸은 0이 아니라 결측이며, 상태 열(조회수상태 등)과 '적용불가'로 구분합니다.
- 원장(03 콘텐츠마스터, 06 지역근거, 07 키워드근거, 08 지역대상맥락)은 Excel 한도와 파일 크기 때문에 **Excel에는 색인과 앞부분 표본만** 싣고, 전체는 `원장/` 폴더의 UTF-8 BOM CSV(100만 행 단위 분할)와 Parquet로 둡니다. 09·14번도 Excel에는 요약 범위만 싣고 전체는 `csv/`에 있습니다.
- `manifest.csv`: 모든 파일의 행 수·고유 키 중복 수·SHA-256.

In [ ]:
def data_check_table() -> pd.DataFrame:
    """01_자료점검: 매체×연도×월×시군×형식×수집층 수록 현황과 결측·중복."""
    parts = []
    for m in MEDIA:
        d = DF[m].copy()
        d["시군"] = d["원본시군"].replace("", "불명확")
        d["_dup"] = d.duplicated(["source", "date", "title"], keep=False) if m == "뉴스" else d.duplicated("content_id", keep=False)
        d["_out"] = ~d["in_common"]
        d["_nobody"] = d["body"].fillna("").str.strip() == ""
        d["_notag"] = (d["tags"].fillna("").str.strip() == "") if m != "뉴스" else False
        d["_tl"] = (d["timeline"] != "") if m == "유튜브" else False
        g = d.groupby(["year", "month", "시군", "type", "수집층"], dropna=False)
        t = g.agg(원자료행수=("content_id", "size"), 고유ID수=("content_id", "nunique"), 시작일=("date", "min"), 종료일=("date", "max"),
                  기간외수=("_out", "sum"), 정확중복수=("_dup", "sum"), 본문설명결측=("_nobody", "sum"), 태그결측=("_notag", "sum"),
                  타임라인연결수=("_tl", "sum")).reset_index()
        for k in ("좋아요", "조회수", "댓글수"):
            st = d[k + "상태"]
            t[f"{k}유효N"] = g.apply(lambda x: int(x[k + "상태"].isin(["유효", "실제0"]).sum())).values
        t["좋아요숨김수"] = g.apply(lambda x: int((x["좋아요상태"] == "숨김").sum())).values
        t["좋아요결측수"] = g.apply(lambda x: int((x["좋아요상태"] == "미제공").sum())).values
        t["날짜결측"] = 0
        t["매체"] = m
        parts.append(t)
    t = pd.concat(parts, ignore_index=True)
    t["시작일"] = pd.to_datetime(t["시작일"]).dt.strftime("%Y-%m-%d")
    t["종료일"] = pd.to_datetime(t["종료일"]).dt.strftime("%Y-%m-%d")
    t["분석버전"], t["수집시각확인여부"] = ANALYSIS_VERSION, "미확보(구간별 수집일만)"
    t["입력파일"] = np.where(t["매체"] == "뉴스", RAW_NEWS.name, RAW_SNS.name)
    t["SHA256"] = t["입력파일"].map(INPUT_HASH)
    t["시트"] = t["매체"].map({"뉴스": "(CSV)", "인스타": "인스타", "유튜브": "유튜브"})
    note = {"뉴스": "날짜=검색표시날짜(실제 발행일 미확인), 제목+미리보기만 분석",
            "인스타": "2023.1.1~10.30 보완 수집(인기·릴스탭 포함) / 2026 수록량 급증(쪽수 상한·최근 편중)",
            "유튜브": "검색어×연도 상위 50 구조. 2023.1.1~10.30은 조회수 순, 이후 기본 순서. '추가'=시범 수집(2026 없음)"}
    t["편향및주의사항"] = t["매체"].map(note)
    t = t.rename(columns={"year": "연도", "month": "월", "type": "형식"})
    # 원자료 요약 시트 재검산(시군×구간)
    s = sns_raw["요약"].copy()
    rec = []
    for _, r in s.iterrows():
        plat, sg = r["플랫폼"], r["시군"]
        if plat not in ("인스타", "유튜브") or not isinstance(sg, str):
            continue
        d = DF[plat]
        dd = d[d["원본시군"] == sg]
        for col, (a, b) in {"2023년(1.1~10.30)": ("2023-01-01", "2023-10-30"), "2023년(10.31~12.31)": ("2023-10-31", "2023-12-31"),
                            "2024년(1.1~12.31)": ("2024-01-01", "2024-12-31"), "2025년(1.1~12.31)": ("2025-01-01", "2025-12-31"),
                            "2026년(1.1~10.30)": ("2026-01-01", "2026-10-30")}.items():
            actual = int(in_period(dd["date"], a, b).sum())
            claimed = pd.to_numeric(r.get(col), errors="coerce")
            rec.append({"매체": plat, "시군": sg, "형식": "요약시트재검산", "연도": int(a[:4]), "월": col, "원자료행수": actual,
                        "편향및주의사항": f"요약 시트 값 {int(claimed) if pd.notna(claimed) else '없음'} / 실제 행 {actual}" +
                                     ("" if pd.notna(claimed) and int(claimed) == actual else " → 불일치(요약 시트는 정제 전 값일 수 있음)")})
    t = pd.concat([t, pd.DataFrame(rec)], ignore_index=True)
    cols = ["분석버전", "입력파일", "SHA256", "시트", "매체", "연도", "월", "시군", "형식", "수집층", "원자료행수", "고유ID수", "시작일", "종료일",
            "날짜결측", "기간외수", "정확중복수", "본문설명결측", "태그결측", "타임라인연결수", "좋아요유효N", "좋아요숨김수", "좋아요결측수",
            "조회수유효N", "댓글수유효N", "수집시각확인여부", "편향및주의사항"]
    return t.reindex(columns=cols)


def preprocess_table() -> pd.DataFrame:
    """02_전처리집계: 단계별 입력/포함/제외/검토/중복과 검산."""
    rows = []
    for m in MEDIA:
        d = DF[m]
        n0 = len(d)
        n_date = int(d["date"].notna().sum())
        n_common = int(d["in_common"].sum())
        n_ref = int(in_period(d["date"], "2026-10-27", CONFIG["SNS_REF_END"]).sum()) if m != "뉴스" else 0
        n_valid = int((d["valid"] & d["in_common"]).sum())
        n_ad = int((d.get("광고나열", pd.Series(False, index=d.index)) & d["in_common"]).sum())
        n_rev = int(d["검토등급"].isin(["B", "C"]).sum())
        has_t = CT[m].loc[in_period(CT[m]["date"], CONFIG["COMMON_START"], CONFIG["COMMON_END"]), "content_id"].nunique()
        aux = AUX[m]["judge"].value_counts()
        steps = [
            ("S1", "원자료 적재", n0, n0, 0, n_rev, "", 0, "원본 수정 없음, 입력 해시 기록"),
            ("S2", "날짜 확인", n0, n_date, n0 - n_date, 0, "", 0, "날짜 변환 실패 제외"),
            ("S3", "공통 비교기간(2023-01-01~2026-10-26)", n_date, n_common, n_date - n_common, 0, "", 0,
             f"기간 밖은 주 결과에서 제외(SNS 10/27~10/30 {n_ref}건은 참고 보존)"),
            ("S4", "지역 관련성(A·B 유지)", n_common, n_valid + n_ad, n_common - n_valid - n_ad, 0, "", 0, "수집태그 외 지역 단서 없는 SNS 제외(C 수준)"),
            ("S4b", "광고·홍보성 해시태그 나열 콘텐츠 제외", n_valid + n_ad, n_valid, n_ad, 0, "", 0, "해시태그에 지역명+같은 상품·서비스어 4개 이상(2차 검토 반영)"),
            ("S5", "반복군 구성", n_valid, n_valid, 0, 0, str(d.loc[d["valid"] & d["in_common"], "rep_key"].nunique()),
             int(d.loc[d["valid"] & d["in_common"]].duplicated(["source", "date", "title"]).sum()) if m == "뉴스" else 0,
             "뉴스 기사군·인스타 동일 본문 묶음(보조 지표, 행 삭제 없음)"),
            ("S6", "관심대상 후보 보유 콘텐츠", n_valid, has_t, n_valid - has_t, 0, "", 0, "후보가 없는 콘텐츠는 분모에만 포함"),
        ]
        for sid, name, a, b, c, r, grp, exact, rule in steps:
            rows.append({"단계ID": sid, "매체": m, "분석버전": ANALYSIS_VERSION, "단계": name, "입력수": a, "포함수": b, "제외수": c, "검토수": r,
                         "중복군수": grp, "정확중복제거수": exact, "출력수": b, "단계간계수규칙": "입력수 = 포함수 + 제외수",
                         "처리기준": rule, "승인여부": "미승인(초기 기준)", "검산결과": "일치" if a == b + c else "불일치"})
        for j, n in aux.items():
            rows.append({"단계ID": "S7", "매체": m, "분석버전": ANALYSIS_VERSION, "단계": f"키워드 판정: {j}", "입력수": np.nan, "포함수": 0, "제외수": int(n),
                         "검토수": int(n) if j == "인명" else 0, "중복군수": "", "정확중복제거수": 0, "출력수": 0,
                         "단계간계수규칙": "콘텐츠×표현 행 수(콘텐츠는 삭제하지 않음)", "처리기준": "주순위 제외·보조 보존", "승인여부": "미승인(초기 기준)", "검산결과": "-"})
        rows.append({"단계ID": "S8", "매체": m, "분석버전": ANALYSIS_VERSION, "단계": "관심대상 원장(콘텐츠×대상)", "입력수": np.nan,
                     "포함수": len(CT[m]), "제외수": 0, "검토수": int((EVID[m]["원문일치검사"] == "미발견").sum()), "중복군수": "",
                     "정확중복제거수": 0, "출력수": len(CT[m]), "단계간계수규칙": "콘텐츠×대상 고유 행", "처리기준": f"고유 대상 {CT[m]['target'].nunique():,}개",
                     "승인여부": "미승인(초기 기준)", "검산결과": "중복 0" if not CT[m].duplicated(["content_id", "target"]).any() else "중복 있음"})
    return pd.DataFrame(rows)


T01, T02 = data_check_table(), preprocess_table()
print(len(T01), "행 / 02:", len(T02), "행")
T02[T02["단계ID"].isin(["S1", "S2", "S3", "S4", "S6"])][["매체", "단계", "입력수", "포함수", "제외수", "검산결과"]]

In [ ]:
COMMON_KEYS = ["집계ID", "분석버전", "매체", "콘텐츠형식", "수집층", "집계수준", "지역", "주분야", "기간유형", "기간시작", "기간종료", "계절연도",
               "관측월수", "기간완전성", "비교가능여부", "대상ID", "표준관심대상", "관심맥락"]
COMMON_METRICS = ["등장콘텐츠수", "문자열출현횟수_보조", "유효콘텐츠분모", "분모정의", "등장비중", "천건당언급", "분야내콘텐츠분모", "분야내등장비중",
                  "기사군수", "반복군보정등장수", "고유출처수", "최대출처점유율", "관측월수_대상",
                  "관련조회수합계", "관련조회수중앙값", "관련조회수최댓값", "조회수유효N", "조회수결측수",
                  "관련좋아요합계", "관련좋아요중앙값", "관련좋아요최댓값", "좋아요유효N", "좋아요결측수",
                  "관련댓글수합계", "관련댓글수중앙값", "관련댓글수최댓값", "댓글수유효N", "댓글수결측수",
                  "조회수상위1콘텐츠기여율", "좋아요상위1콘텐츠기여율", "대표반응점수중앙값", "반응점수유효N", "반응점수유형",
                  "빈도순위", "반복군보정순위", "반응순위", "최소지지충족", "단일출처의존", "민감도안정성", "검토상태", "대표콘텐츠ID", "근거원장파일",
                  "보조분야", "하위유형"]
TRACK_COLS = ["분석트랙", "보완수집포함여부", "행별출처확인수준", "비교그룹ID", "선택방식", "비교가능판정", "해석제약"]


def tidy(df: pd.DataFrame, sheet: str, extra: list) -> pd.DataFrame:
    """공통 집계키 → 공통 지표 → 시트별 열 → 트랙 열 순서로 정리하고 집계ID를 붙입니다."""
    df = df.copy()
    if "민감도안정성" not in df:
        df["민감도안정성"] = ""
    df["집계ID"] = [f"{sheet[:2]}-{i:07d}" for i in range(1, len(df) + 1)]
    cols = [c for c in COMMON_KEYS + COMMON_METRICS + extra + TRACK_COLS if c in df.columns]
    for c in ("단일출처의존",):
        if c in df:
            df[c] = np.where(df[c].astype(bool), "예", "")
    return df[cols]


def ledger_frames(m: str):
    """원장 3종(06 지역근거, 07 키워드근거, 08 지역대상맥락)을 매체별로 만듭니다."""
    e = EVID[m].copy()
    e["키워드근거ID"] = [f"K{m[0]}{i:08d}" for i in range(1, len(e) + 1)]
    src_map = {"본문": "형태소 명사 연쇄(본문)", "복합": "어절 결합 복합명사(빈도·결합강도 기준)", "태그": "해시태그·태그"}
    k07 = pd.DataFrame({
        "키워드근거ID": e["키워드근거ID"], "콘텐츠ID": e["content_id"], "대상ID": e["target"].map(target_id), "원어": e["raw"],
        "표준관심대상": e["target"], "근거필드": e["근거필드"], "문장구간ID": np.where(e["필드내위치"] >= 0, e["근거필드"] + ":" + e["필드내위치"].astype(str), ""),
        "타임라인출처": np.where(e["근거필드"].str.startswith("타임라인"), e["근거필드"].str.extract(r"\((.+)\)")[0], ""),
        "시간표시": e["시간표시"], "원문인용": e["원문인용"], "원문일치검사": e["원문일치검사"],
        "직접보조여부": np.where(e["src"] == "태그", "태그(작성자 표기)", "본문 직접"), "주분야": e["target"].map(AREA_MAP).fillna("미분류(검토필요)"),
        "보조분야": e["target"].map(AREA_TABLE["보조분야"]).fillna(""), "추출방법": e["src"].map(src_map),
        "모델프롬프트버전": "해당없음(LLM 미사용)", "사전버전": ANALYSIS_VERSION, "검토상태": np.where(e["원문일치검사"] == "일치", "자동", "검토(원문 미발견)"),
        "판정자": "규칙(자동)", "문자열출현횟수": e["문자열출현횟수"], "타임라인구간수": e["타임라인구간수"],
    })
    r = REL[m].copy()
    r["지역근거ID"] = [f"G{m[0]}{i:08d}" for i in range(1, len(r) + 1)]
    cm = DF[m].set_index("content_id")
    g06 = pd.DataFrame({
        "지역근거ID": r["지역근거ID"], "콘텐츠ID": r["content_id"], "대상ID": r["target"].map(target_id),
        "원본시군": r["content_id"].map(cm["원본시군"]), "검색수집후보시군": r["content_id"].map(cm["sigun_all"].map(lambda L: ",".join(L))),
        "콘텐츠근거시군": r["콘텐츠근거시군"], "대상소재시군": r["대상소재시군"], "최종귀속시군": r["최종귀속시군"],
        "다중귀속여부": np.where(r["다중귀속여부"], "예", "아니오"), "근거필드": r["근거필드"], "원문인용": r["원문인용"],
        "문장구간ID": np.where(r["필드내위치"] >= 0, r["근거필드"] + ":" + r["필드내위치"].astype(str), ""),
        "외부확인출처_승인시": "", "귀속확실성": r["귀속확실성"], "귀속근거": r["귀속근거"], "검토상태": "자동", "판정자": "규칙(자동)",
    })
    ctx = r["content_id"].map(CTX[m])
    co = CT[m].groupby("content_id")["target"].agg(list)
    r08 = pd.DataFrame({
        "관계ID": [f"R{m[0]}{i:08d}" for i in range(1, len(r) + 1)], "콘텐츠ID": r["content_id"], "대상ID": g06["대상ID"],
        "표준관심대상": r["target"], "최종귀속시군": r["최종귀속시군"],
        "경험메뉴정책쟁점": ctx.map(lambda L: ", ".join(L) if isinstance(L, list) else ""), "연결등급": r["연결등급"],
        "지역근거ID": r["지역근거ID"], "키워드근거ID": r[["content_id", "target"]].merge(e[["content_id", "target", "키워드근거ID"]], on=["content_id", "target"], how="left")["키워드근거ID"].values,
        "동반대상ID": [",".join([target_id(x) for x in co.get(c, []) if x != t][:3]) for c, t in zip(r["content_id"], r["target"])],
        "방향성": "대상→지역", "검토상태": "자동", "집계포함여부": np.where(r["연결등급"].isin(["A", "B"]), "핵심 집계", "C 별도 표"),
    })
    return g06, k07, r08

In [ ]:
def write_csv_split(df: pd.DataFrame, path: Path, max_rows: int = 1_000_000) -> list:
    """UTF-8 BOM CSV로 저장(100만 행 단위 분할). 저장한 파일 목록을 돌려줍니다."""
    files = []
    if len(df) <= max_rows:
        df.to_csv(path, index=False, encoding="utf-8-sig")
        return [path]
    for i, a in enumerate(range(0, len(df), max_rows), 1):
        p = path.with_name(f"{path.stem}_part{i}{path.suffix}")
        df.iloc[a:a + max_rows].to_csv(p, index=False, encoding="utf-8-sig")
        files.append(p)
    return files


def guide_sheet(out_dir: Path) -> pd.DataFrame:
    rows = [
        ("파일 버전", ANALYSIS_VERSION),
        ("과업 기준", "SNS_키워드분석_상세과업지시서_v2_수집편향반영(발주자 확인 전 '초기 기준 결과')"),
        ("입력 파일(SHA-256)", "; ".join(f"{k} = {v}" for k, v in INPUT_HASH.items())),
        ("주 비교기간", f"{CONFIG['COMMON_START']} ~ {CONFIG['COMMON_END']}(세 매체 공통). SNS 2026-10-27~10-30은 02_전처리집계에 건수만 참고 보존"),
        ("연도 비교", "각 연도 1/1~10/26 동기간(민감도 1~9월). 2023~2025 연간 전체는 13번 시트 '연간 전체' 기간유형으로만 제공"),
        ("분석 트랙", "A=전체 관측자료 발굴(2023 보완 포함), B=2024~2026 비교 가능한 층(뉴스 전체, IG-기본, YT-기본순-기본) 주 추세, C=2023 포함 민감도"),
        ("2023 보완 수집", "2023년 1~10월(정확히는 2023-01-01~10-30 작성분)에만 보완 수집이 이루어져 연도별 수록량과 선택 방식이 다르다. 전체 자료는 관심대상 발굴에 활용하고, 연도별 변화는 비교 가능한 수집층·동기간을 중심으로 분석했다. 수집 비중 정규화만으로 선택 편향이 완전히 제거되지는 않으며, 2023년 포함 여부에 민감한 결론은 조건부 결과로 표시했다."),
        ("빈도 단위", "등장콘텐츠수 = 고유 콘텐츠 수(DF). 반복군보정등장수(뉴스 기사군, SNS 동일 본문·출처 반복 1건)를 보조로 병기"),
        ("보고용 상위 목록 조건", "최소지지충족(콘텐츠 5·출처 3, 뉴스 언론사 2) & 단일출처의존 아님(최대출처점유율 < 0.5) & 6개 분야 분류. 전체 표는 조건과 무관하게 전부 제공"),
        ("주순위 제외(보조 보존)", "단독 지역명, 범용어·SNS 일반어, 정당명, 지정인명(이재명·윤석열·김관영·이원택), 자동 판정 인명(승인 필요), 기관·직함 단독, 지역 연결 없는 식품명. 콘텐츠는 삭제하지 않음"),
        ("시군 귀속", "대상명 지명 > 대상 소재(본문 분포 추정) > 콘텐츠 본문 지명 > 전북전체 > 지역미확정. 핵심 순위는 연결등급 A/B, C는 별도 열"),
        ("반응", "조회수·좋아요·댓글을 분리. 결측 상태(숨김·미제공·적용불가)를 0과 구분. 대표반응점수는 플랫폼×형식×분기 백분위. 인스타·유튜브 원시 지표 합산 없음"),
        ("원장 파일", "원장/ 폴더: 03_콘텐츠마스터_*, 06_지역근거_*, 07_키워드근거_*, 08_지역대상맥락_*(CSV는 100만 행 단위 분할, 같은 이름의 Parquet 전체본)"),
        ("Excel 요약 범위", "03·06·07·08: 색인 + 앞부분 표본 / 09·11: 등장 5건 이상 / 10: 공통기간 전체·연도 동기간, 5건 이상 / 13: 10건 이상 / 14: 유형·보고용 상위 대상. 전체(조건 없음)는 csv/ 폴더"),
        ("부록 시트", "29_기타민감도(양식 29개 외 추가): 1~9월, 반복군, 출처 유형, 설명란 타임라인, 상위 1개 제외 반응"),
        ("검수 상태", "자동 검산은 23번 시트 실측값. 독립 검수(정밀도)는 미실시 — 검수표본_*.csv에 판정 입력 필요. AI 검토를 사람 검수로 표시하지 않음"),
        ("금지 해석", "빈도 상위=도민 선호, 조회수 합계=고유 관심자, 좋아요 결측=0, 누적 조회수로 과거 반응 성장 복원, 검색태그만으로 시군 확정, 시군 데이터량 차이=관심도 우열"),
        ("저장 위치", str(out_dir)),
    ]
    return pd.DataFrame(rows, columns=["항목", "내용"])


def index_sheet(name: str, files: list, df_sample: pd.DataFrame, total: int, note: str) -> pd.DataFrame:
    head = pd.DataFrame({"구분": ["원장 색인"] * len(files), "파일명": [f"원장/{p.name}" for p in files],
                         "내용": [note] * len(files), "전체행수": [total] * len(files)})
    s = df_sample.copy()
    s.insert(0, "구분", f"표본(앞 {len(s):,}행)")
    return pd.concat([head, s], ignore_index=True)


def export_all():
    out_dir = OUT_DIR / f"v2_{RUN_TS}"
    led_dir, csv_dir = out_dir / "원장", out_dir / "csv"
    for p in (out_dir, led_dir, csv_dir):
        p.mkdir(parents=True, exist_ok=True)
    manifest = []

    def note_file(p: Path, df: pd.DataFrame, key: list, sheet: str = ""):
        dup = int(df.duplicated(key).sum()) if key and all(k in df for k in key) else ""
        manifest.append({"파일": str(p.relative_to(out_dir)), "시트·범위": sheet, "행수": len(df), "고유키": "+".join(key), "고유키중복수": dup})

    # ── 원장 ──
    idx = {}
    for m in MEDIA:
        t0 = time.time()
        g06, k07, r08 = ledger_frames(m)
        for code_, df, key in (("03_콘텐츠마스터", MASTER[m], ["콘텐츠ID"]), ("06_지역근거", g06, ["콘텐츠ID", "대상ID", "최종귀속시군"]),
                               ("07_키워드근거", k07, ["콘텐츠ID", "대상ID"]), ("08_지역대상맥락", r08, ["콘텐츠ID", "대상ID", "최종귀속시군"])):
            fs = write_csv_split(df, led_dir / f"{code_}_{m}.csv")
            df.to_parquet(led_dir / f"{code_}_{m}.parquet", index=False)
            dup = int(df.duplicated(key).sum())
            for k_, f in enumerate(fs):
                part_len = len(df) if len(fs) == 1 else min(1_000_000, len(df) - k_ * 1_000_000)
                manifest.append({"파일": str(f.relative_to(out_dir)), "시트·범위": f"분할 {k_ + 1}/{len(fs)}", "행수": part_len,
                                 "고유키": "+".join(key), "고유키중복수": dup if k_ == 0 else ""})
            manifest.append({"파일": f"원장/{code_}_{m}.parquet", "시트·범위": "전체본", "행수": len(df), "고유키": "+".join(key), "고유키중복수": dup})
            idx.setdefault(code_, []).append((m, fs, df.head(2000 if code_ != "03_콘텐츠마스터" else 5000), len(df)))
        print(f"  원장 저장 {m} {time.time()-t0:.0f}초")

    # ── 시트 ──
    sheets = {
        "00_안내": guide_sheet(out_dir), "01_자료점검": T01, "02_전처리집계": T02,
        "04_표준키워드사전": T04, "05_제외보조어사전": T05,
        "09_매체별키워드전체": tidy(pd.concat(T09.values(), ignore_index=True), "09", []),
        "10_시군별빈도": tidy(T10, "10", ["시군내빈도순위", "지역고유성비", "지역고유성분모", "지역고유성판정", "다중귀속여부", "다중귀속콘텐츠수", "C동반출현수"]),
        "11_시군별SNS반응": tidy(T11, "11", ["시군내절대반응순위", "시군내상대반응순위", "사분면", "사분면빈도기준선", "사분면반응기준선", "사분면후보수",
                                         "사분면_평균기준", "사분면_상위사분위기준", "숨은관심대상후보", "독립문구수", "대상소재시군", "반응의존구분", "원문검토상태", "원문검토판정", "원문검토사유",
                                         "숨은관심대상여부", "소수근거여부", "비교기간ID", "분석기간"]),
        "12_분야별빈도반응": tidy(T12, "12", ["분야", "분야내순위", "분류가능콘텐츠비율", "주분야복수분야구분"]),
        "13_연도계절키워드": tidy(T13, "13", ["비교기간ID", "대응동기간", "불완전기간사유"]),
        "14_월별추이": None,
        "15_HOT": tidy(pd.concat(HOT.values(), ignore_index=True), "15", ["HOT", "최신기간", "HOT선정기준", "고반응여부", "선정근거", "미충족사유"]),
        "16_RISING": tidy(pd.concat(RISE.values(), ignore_index=True), "16", ["유형", "유형내순위", "비교기준연도", "기준기간시작", "기준기간종료", "기준등장수", "기준분모",
                       "기준등장비중", "현재등장수", "현재분모", "현재등장비중", "비중비", "비중차", "천건당언급증가", "건수차", "2024등장수", "2024등장비중",
                       "과거연도별비중_원장참조", "과거동일가중평균비중", "평균대비비중비", "2023등장비중_C", "C기준_2023포함평균비중", "C기준_비중비",
                       "판정기준", "신규관측여부", "표집변경가능성", "지지조건", "RISING_B", "RISING_C", "RISING_1~9월", "2023보완수집민감", "선정근거", "미충족사유"]),
        "17_STEADY": tidy(pd.concat(STEADY.values(), ignore_index=True), "17", ["STEADY", "충족연도수", "연도별등장수_원장참조", "연도별비중_원장참조", "전체등장수",
                       "비중변동지표", "연도내월분산", "상투반복점검", "선정근거", "미충족사유"]),
        "18_SEASONAL": tidy(pd.concat([v for v in SEASONAL.values() if len(v)], ignore_index=True), "18", ["SEASONAL", "피크계절", "반복확인완전연도수", "확인계절연도",
                       "계절별분모_원장참조", "계절별비중_원장참조", "기간완전성점검", "행사일정근거", "선정근거", "미충족사유"]),
        "19_EVENT_VIRAL": tidy(pd.concat([v for v in EVENT.values() if len(v)], ignore_index=True), "19", ["EVENT_VIRAL", "세부유형", "피크월", "피크월콘텐츠기여율",
                       "상위1콘텐츠기여율", "관련사건행사", "연례반복여부", "사건맥락근거", "단일출처점검", "선정근거", "미충족사유"]),
        "20_매체비교": T20, "21_대표콘텐츠": T21, "22_검토오류목록": T22, "23_검수결과": T23, "24_지표정의": T24, "25_파라미터": T25,
        "26_변경이력": T26, "27_수집층비교가능성": LAYER27, "28_보완수집민감도": T28, "29_기타민감도": T29,
    }
    # 14: 전체는 CSV, Excel에는 유형·보고용 상위 대상
    t14 = T14.copy()
    t14.insert(0, "집계ID", [f"14-{i:07d}" for i in range(1, len(t14) + 1)])
    focus = set()
    for m in MEDIA:
        focus |= set(report_list(T09[m]).head(30)["target"]) | set(HOT[m].loc[HOT[m]["HOT"], "target"]) | set(RISE[m]["target"].head(60))
        focus |= set(STEADY[m].loc[STEADY[m]["STEADY"], "target"].head(30))
        if len(SEASONAL[m]):
            focus |= set(SEASONAL[m].loc[SEASONAL[m]["SEASONAL"], "target"].head(30))
    fid = {target_id(t) for t in focus}
    for name in list(sheets):
        df = sheets[name] if name != "14_월별추이" else t14
        if df is None:
            continue
        p = csv_dir / f"{name}.csv"
        files = write_csv_split(df, p)
        for f in files:
            note_file(f, df, [], "전체")
    sheets["14_월별추이"] = t14[t14["대상ID"].isin(fid)]
    # Excel 요약 범위(전체는 csv/): 09·11 등장 5건 이상, 10 공통기간 전체·연도 동기간 5건 이상, 13 10건 이상
    s09 = sheets["09_매체별키워드전체"]
    sheets["09_매체별키워드전체"] = s09[s09["등장콘텐츠수"] >= 5]
    s10 = sheets["10_시군별빈도"]
    sheets["10_시군별빈도"] = s10[(s10["등장콘텐츠수"] >= 5) & s10["기간유형"].isin(["공통기간 전체", "연도 동기간(1/1~10/26)"])]
    s11 = sheets["11_시군별SNS반응"]
    sheets["11_시군별SNS반응"] = s11[s11["등장콘텐츠수"] >= 5]
    s13 = sheets["13_연도계절키워드"]
    sheets["13_연도계절키워드"] = s13[s13["등장콘텐츠수"] >= 10]
    for code_, lst in idx.items():
        parts = [index_sheet(code_, fs, smp, tot, f"{m} 전체 {tot:,}행") for m, fs, smp, tot in lst]
        sheets[code_] = pd.concat(parts, ignore_index=True)
    order = ["00_안내", "01_자료점검", "02_전처리집계", "03_콘텐츠마스터", "04_표준키워드사전", "05_제외보조어사전", "06_지역근거", "07_키워드근거",
             "08_지역대상맥락", "09_매체별키워드전체", "10_시군별빈도", "11_시군별SNS반응", "12_분야별빈도반응", "13_연도계절키워드", "14_월별추이",
             "15_HOT", "16_RISING", "17_STEADY", "18_SEASONAL", "19_EVENT_VIRAL", "20_매체비교", "21_대표콘텐츠", "22_검토오류목록", "23_검수결과",
             "24_지표정의", "25_파라미터", "26_변경이력", "27_수집층비교가능성", "28_보완수집민감도", "29_기타민감도"]
    xlsx = out_dir / f"미지의전북_관심대상분석_v2_{RUN_TS}.xlsx"
    t0 = time.time()
    with pd.ExcelWriter(xlsx, engine="xlsxwriter", engine_kwargs={"options": {"strings_to_urls": False, "nan_inf_to_errors": True}}) as w:
        wb = w.book
        hdr = wb.add_format({"bold": True, "bg_color": "#E8EEF4", "border": 1, "text_wrap": True, "valign": "top"})
        for name in order:
            df = sheets[name]
            df = df.replace([np.inf, -np.inf], np.nan)
            if len(df) > 1_048_000:
                raise ValueError(f"{name}: Excel 한도 초과({len(df):,}행) — CSV로만 저장 필요")
            df.to_excel(w, sheet_name=name, index=False)
            ws = w.sheets[name]
            for j, c in enumerate(df.columns):
                ws.write(0, j, c, hdr)
                try:
                    q = float(df.iloc[:300, j].astype(str).str.len().quantile(0.9))
                    q = 8 if np.isnan(q) else q
                except Exception:
                    q = 12
                width = int(min(48, max(8, q + 2, len(str(c)) + 2)))
                ws.set_column(j, j, width)
            ws.freeze_panes(1, 0)
            ws.autofilter(0, 0, max(len(df), 1), max(len(df.columns) - 1, 0))
            manifest.append({"파일": xlsx.name, "시트·범위": name, "행수": len(df), "고유키": "", "고유키중복수": ""})
    print(f"  Excel 저장 {time.time()-t0:.0f}초")
    SAMPLE_TARGET.to_csv(out_dir / "검수표본_관심대상.csv", index=False, encoding="utf-8-sig")
    SAMPLE_REGION.to_csv(out_dir / "검수표본_시군귀속.csv", index=False, encoding="utf-8-sig")
    note_file(out_dir / "검수표본_관심대상.csv", SAMPLE_TARGET, [])
    note_file(out_dir / "검수표본_시군귀속.csv", SAMPLE_REGION, [])
    ENV_INFO.to_csv(out_dir / "재현정보.csv", index=False, encoding="utf-8-sig")
    man = pd.DataFrame(manifest)
    man["SHA256"] = [file_sha256(out_dir / f) if (out_dir / f).exists() and f.endswith((".csv", ".parquet")) else "" for f in man["파일"]]
    man.to_csv(out_dir / "manifest.csv", index=False, encoding="utf-8-sig")
    return out_dir, xlsx


t0 = time.time()
OUT_V2, XLSX_V2 = export_all()
print(f"저장 완료 {time.time()-t0:.0f}초 →", XLSX_V2)

## 17. 보고서용 그림 (과업지시서 20장 시각화 기본)

빈도와 반응을 분리한 산점도, 매체별 월별 비중(RISING 대상), 시군×분야 열지도, 연도별 비중 패턴을 PNG로 저장합니다. 워드클라우드는 만들지 않습니다.

In [ ]:
FIG_DIR = OUT_V2 / "그림"
FIG_DIR.mkdir(exist_ok=True)
C_BLUE, C_ORANGE, C_AQUA, C_GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#b8b6ad"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e5e0",
                     "grid.linewidth": 0.6, "axes.edgecolor": "#8a8984", "font.size": 9})


def fig_scatter(m: str):
    """빈도(x, 로그) × 상대 반응(y) 산점도. 숨은 관심대상은 주황, 단일출처의존은 회색."""
    d = T11[(T11["매체"] == m) & (T11["비교기간ID"] == "A_전체") & (T11["지역"] == "매체 전체") & (T11["최소지지충족"] == "충족")]
    d = d[d["주분야"].isin(AREAS)]
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    base = d[(d["숨은관심대상여부"] != "예")]
    ax.scatter(base["등장콘텐츠수"], base["대표반응점수중앙값"], s=10, color=C_BLUE, alpha=0.35, linewidths=0, label="보고용 대상")
    hid = d[d["숨은관심대상여부"] == "예"]
    ax.scatter(hid["등장콘텐츠수"], hid["대표반응점수중앙값"], s=18, color=C_ORANGE, alpha=0.9, linewidths=0, label="숨은 관심대상(언급 적음·반응 높음)")
    ax.set_xscale("log")
    ax.axhline(d["사분면반응기준선"].iloc[0], color="#8a8984", lw=0.8, ls="--")
    ax.axvline(d["사분면빈도기준선"].iloc[0], color="#8a8984", lw=0.8, ls="--")
    for _, r in pd.concat([hid.nlargest(5, "등장콘텐츠수"), d.nlargest(5, "등장콘텐츠수")]).drop_duplicates("target").iterrows():
        ax.annotate(r["target"], (r["등장콘텐츠수"], r["대표반응점수중앙값"]), fontsize=7.5, xytext=(3, 3), textcoords="offset points", color="#2b2b2b")
    ax.set_xlabel("등장 콘텐츠 수(로그 축)")
    ax.set_ylabel("대표반응점수 중앙값(0~100)")
    ax.set_title(f"{ {'인스타': '인스타그램', '유튜브': '유튜브'}[m] }: 언급 빈도와 상대 반응(지지조건 충족 대상, 점선 = 중앙값)", loc="left", fontsize=10)
    ax.legend(loc="lower right", fontsize=8, frameon=False)
    fig.tight_layout()
    p = FIG_DIR / f"그림_산점도_{m}.png"
    fig.savefig(p, dpi=200)
    plt.close(fig)
    return p


def fig_rising_monthly(m: str, k: int = 3):
    """RISING 상위 k개 대상의 월별 등장 비중(트랙 B 수집층)."""
    r = RISE[m]
    tg = r.loc[r["유형"] == "RISING", "target"].head(k).tolist()
    if not tg:
        return None
    track = "B"
    t14 = T14[(T14["매체"] == m) & (T14["분석트랙"] == ("A" if m != "유튜브" else "B"))]
    fig, ax = plt.subplots(figsize=(7.2, 3.4))
    for t, col in zip(tg, (C_BLUE, C_ORANGE, C_AQUA)):
        x = t14[t14["표준관심대상"] == t]
        x = x[x["월"] >= ("2024-01" if m != "뉴스" else "2023-01")]
        ax.plot(x["월"], x["천건당언급"], color=col, lw=2, label=t)
    ax.set_xticks(ax.get_xticks()[::6])
    ax.set_ylabel("천 건당 등장 수")
    ax.set_title(f"{ {'뉴스': '뉴스', '인스타': '인스타그램', '유튜브': '유튜브'}[m] }: RISING 상위 대상의 월별 등장 비중", loc="left", fontsize=10)
    ax.legend(fontsize=8, frameon=False, loc="upper left")
    fig.tight_layout()
    p = FIG_DIR / f"그림_RISING월별_{m}.png"
    fig.savefig(p, dpi=200)
    plt.close(fig)
    return p


def fig_heatmap():
    """시군×분야 열지도: 시군별(A/B 연결) 대상 등장 중 각 분야의 비중, 세 매체 각각."""
    fig, axes = plt.subplots(1, 3, figsize=(10, 4.6), sharey=True)
    for ax, m in zip(axes, MEDIA):
        x = T10[(T10["매체"] == m) & (T10["기간유형"] == "공통기간 전체") & T10["지역"].isin(SIGUN) & T10["주분야"].isin(AREAS)
                & (T10["최소지지충족"] == "충족") & ~T10["단일출처의존"].astype(bool)]
        pv = x.pivot_table(index="지역", columns="주분야", values="등장콘텐츠수", aggfunc="sum").reindex(index=SIGUN, columns=AREAS).fillna(0)
        pv = pv.div(pv.sum(axis=1), axis=0)
        im = ax.imshow(pv.values, cmap="Blues", vmin=0, vmax=0.6, aspect="auto")
        ax.set_xticks(range(len(AREAS)))
        ax.set_xticklabels([a.split("·")[0] for a in AREAS], rotation=45, ha="right", fontsize=7.5)
        ax.set_yticks(range(len(SIGUN)))
        ax.set_yticklabels(SIGUN, fontsize=7.5)
        ax.set_title({"뉴스": "뉴스", "인스타": "인스타그램", "유튜브": "유튜브"}[m], fontsize=9)
        ax.grid(False)
        for i in range(pv.shape[0]):
            for j in range(pv.shape[1]):
                v = pv.values[i, j]
                ax.text(j, i, f"{v*100:.0f}", ha="center", va="center", fontsize=6, color="white" if v > 0.35 else "#2b2b2b")
    fig.colorbar(im, ax=axes, shrink=0.7, label="시군 내 분야 비중")
    fig.suptitle("시군별 관심대상 등장의 분야 구성(%, A/B 연결, 지지조건 충족·단일 출처 의존 제외, 공통기간 전체)", x=0.01, ha="left", fontsize=10)
    p = FIG_DIR / "그림_시군분야열지도.png"
    fig.savefig(p, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return p


def fig_patterns():
    """유형별 예시: STEADY·SEASONAL 대표 대상의 연도별(동기간) 비중."""
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
    for ax, m in zip(axes, MEDIA):
        yb = YS[(m, "B", "")]
        st = STEADY[m].loc[STEADY[m]["STEADY"], "target"].head(2).tolist()
        rs = RISE[m].loc[RISE[m]["유형"] == "RISING", "target"].head(2).tolist()
        for t, col, ls in zip(st + rs, (C_BLUE, C_BLUE, C_ORANGE, C_ORANGE), ("-", "--", "-", "--")):
            ys = sorted(yb)
            ax.plot(ys, [yb[y]["천건당언급"].get(t, 0) for y in ys], color=col, ls=ls, marker="o", ms=4, lw=1.8,
                    label=f"{t}({'STEADY' if t in st else 'RISING'})")
        ax.set_xticks(sorted(yb))
        ax.set_title({"뉴스": "뉴스", "인스타": "인스타그램", "유튜브": "유튜브"}[m], fontsize=9)
        ax.legend(fontsize=6.5, frameon=False)
    axes[0].set_ylabel("천 건당 등장 수(1/1~10/26)")
    fig.suptitle("지속(STEADY)과 상승(RISING) 대상의 연도별 동기간 비중(트랙 B)", x=0.01, ha="left", fontsize=10)
    fig.tight_layout()
    p = FIG_DIR / "그림_유형패턴.png"
    fig.savefig(p, dpi=200)
    plt.close(fig)
    return p


FIGS = {"산점도_인스타": fig_scatter("인스타"), "산점도_유튜브": fig_scatter("유튜브"), "열지도": fig_heatmap(), "패턴": fig_patterns()}
for m in MEDIA:
    FIGS[f"RISING월별_{m}"] = fig_rising_monthly(m)
print({k: (v.name if v else None) for k, v in FIGS.items()})

## 다음 단계(발주자 확인 후 재실행)

1. `22_검토오류목록`과 `05_제외보조어사전`의 '승인 필요' 항목(자동 판정 인명, 미분류 상위, 동음이의 후보)을 확인해 `dict/` 사전에 반영합니다.
2. `검수표본_관심대상.csv`, `검수표본_시군귀속.csv`에 독립 검토자가 판정을 입력하면 23번 시트의 정밀도를 산출할 수 있습니다.
3. `25_파라미터`의 지지조건·유형 임계값을 승인받아 `CONFIG`에 고정한 뒤 처음부터 다시 실행합니다(형태소 분석은 캐시 사용).

## 18. 보고서 연계형 결과 Excel (2차 검토 반영)

09~29번 양식 시트는 분석 과정과 세부 지표를 모두 담고 있어 보고서 표를 바로 찾기 어렵습니다. 그래서 **보고서 표 번호와 시트가 1:1로 대응하는 별도 파일** `미지의전북_보고서연계결과_v2_<실행시각>.xlsx`을 만듭니다.

- **표6 매체별 / 표7 분야별 / 표8 시군별 / 표9 SNS 반응**: `전체`(2023~2026 누적, 트랙 A)와 `2023`~`2026`(각 연도 1/1~10/26 동기간, 트랙 Y) 시트로 나눕니다.
- 보고서에는 상위 10~15개만 싣지만, 시트에는 **보고용 기준을 통과한 대상 전부**를 싣습니다. 순위마다 뉴스·인스타그램·유튜브의 키워드·콘텐츠 수·비중을 나란히 놓고, 맨 아래 `전체` 행에 분모(유효 콘텐츠 수)를 둡니다.
- 보고용 기준: 최소지지(콘텐츠 5건·출처 3곳, 뉴스는 언론사 2곳), 단일 출처 의존 아님(최대출처점유율 < 0.5), 6개 분야로 분류됨. 제외·보조 처리된 표현은 `제외보조_키워드` 시트에 따로 둡니다.
- SNS 반응 시트에는 조회수·좋아요·댓글 합계·중앙값, 대표반응점수, 사분면, 숨은 관심대상 판정과 반응 의존 구분, 대표 원문·URL을 함께 둡니다.
- `원문근거_키워드별`: 보고용 대상·숨은 관심대상·신규 관측·지역미확정 상위 대상마다 대표 콘텐츠 최대 3건의 원문 인용과 URL(뉴스는 언론사·원자료 행 ID).

In [ ]:
YEARS = [2023, 2024, 2025, 2026]
PERIOD_LABEL = {"전체": "2023-01-01~2026-10-26 누적(트랙 A: 2023 보완 수집 포함 전체 자료)"}
for _y in YEARS:
    PERIOD_LABEL[str(_y)] = (f"{_y}-01-01~{_y}-10-26 동기간. " +
                             ("SNS는 2023 보완 수집층(인스타 인기·릴스 탭 포함, 유튜브 조회수순 상위 50) → 2024~2026과 수집 방식이 달라 참고용" if _y == 2023
                              else "뉴스 전체, 인스타 기본 구간(최근 게시물), 유튜브 기본순·기본 수집(트랙 B와 같은 수집층)"))
MK = {"뉴스": "뉴스", "인스타": "인스타그램", "유튜브": "유튜브"}


def view_media(m: str, per: str) -> pd.DataFrame:
    """매체 전체 수준의 집계(전체 = T09, 연도 = T13 트랙 Y)."""
    if per == "전체":
        return T09[m]
    return T13[(T13["매체"] == m) & (T13["분석트랙"] == "Y") & (T13["비교기간ID"] == f"Y{per}")]


def view_region(m: str, per: str) -> pd.DataFrame:
    if per == "전체":
        return T10[(T10["매체"] == m) & (T10["기간유형"] == "공통기간 전체")]
    return T10[(T10["매체"] == m) & (T10["기간유형"].str.startswith("연도 동기간")) & (T10["기간시작"] == f"{per}-01-01")]


def view_reaction(m: str, per: str) -> pd.DataFrame:
    pid = "A_전체" if per == "전체" else f"Y{per}"
    return T11[(T11["매체"] == m) & (T11["비교기간ID"] == pid)]


def denom(m: str, per: str) -> int:
    v = view_media(m, per)
    return int(v["유효콘텐츠분모"].iloc[0]) if len(v) else 0


def ranked(df: pd.DataFrame) -> pd.DataFrame:
    return report_list(df).sort_values(["등장콘텐츠수", "고유출처수"], ascending=False).reset_index(drop=True)


def side_by_side(frames: dict, dens: dict) -> pd.DataFrame:
    """순위 × (매체별 키워드·콘텐츠 수·비중) 가로 배열 + '전체'(분모) 행."""
    n = max([len(f) for f in frames.values()] + [0])
    out = pd.DataFrame({"순위": list(range(1, n + 1))})
    for m in MEDIA:
        f = frames.get(m, pd.DataFrame())
        k = f["표준관심대상"].tolist() if len(f) else []
        c = f["등장콘텐츠수"].astype(int).tolist() if len(f) else []
        s = (f["등장비중"] * 100).round(3).tolist() if len(f) else []
        out[f"{MK[m]} 키워드"] = k + [""] * (n - len(k))
        out[f"{MK[m]} 콘텐츠 수"] = c + [None] * (n - len(c))
        out[f"{MK[m]} 비중(%)"] = s + [None] * (n - len(s))
    tot = {"순위": "전체(분모)"}
    for m in MEDIA:
        tot[f"{MK[m]} 키워드"] = "유효 콘텐츠 수"
        tot[f"{MK[m]} 콘텐츠 수"] = dens.get(m)
        tot[f"{MK[m]} 비중(%)"] = 100.0 if dens.get(m) else None
    return pd.concat([out, pd.DataFrame([tot])], ignore_index=True)


# ── 원문 근거 ────────────────────────────────────────────
_DFI = {m: DF[m].set_index("content_id") for m in MEDIA}


def quotes(m: str, targets, k: int = 3, start=None, end=None) -> pd.DataFrame:
    """대상별 대표 콘텐츠 k건의 원문 인용·URL. SNS는 대표반응점수, 뉴스는 최신 순(같은 날은 기사군 큰 순)."""
    e = EVID[m][EVID[m]["target"].isin(set(targets))][["content_id", "target", "근거필드", "원문인용"]]
    cu = CONT[m].drop_duplicates("content_id").set_index("content_id")
    e = e.assign(date=e["content_id"].map(cu["date"]), score=e["content_id"].map(cu["대표반응점수"]),
                 source=e["content_id"].map(cu["source"]))
    if start is not None:
        e = e[in_period(e["date"], start, end)]
    if m == "뉴스":
        rk = _DFI[m]["rep_key"]
        e = e.assign(sz=e["content_id"].map(rk).map(rk.value_counts())).sort_values(["date", "sz"], ascending=False)
    else:
        e = e.sort_values("score", ascending=False)
    e = e.groupby("target", sort=False).head(k).copy()
    d = _DFI[m]
    e["순번"] = e.groupby("target").cumcount() + 1
    e["제목"] = e["content_id"].map(lambda c: str(d.at[c, "title"] or d.at[c, "body"] or "")[:100].replace("\n", " "))
    e["URL"] = e["content_id"].map(d["url"]).fillna("") if m != "뉴스" else ""
    e["원자료ID"] = e["content_id"].map(d["원본ID"])
    e["출처"] = e["source"].astype(str) if m != "인스타" else "(작성자 해시)"
    e["조회수"] = e["content_id"].map(cu["조회수"])
    e["좋아요"] = e["content_id"].map(cu["좋아요"])
    e["댓글수"] = e["content_id"].map(cu["댓글수"])
    e["원문인용"] = e["원문인용"].astype(str).str.replace("\n", " ").str.slice(0, 200)
    return e


# ── 시트 데이터 ──────────────────────────────────────────
RPT = {}          # 시트명 → (제목, [블록]), 블록 = (소제목 또는 None, DataFrame)
NOTE_COMMON = "보고용 기준(최소지지·단일 출처 의존 아님·6개 분야 분류)을 통과한 대상 전부. 비중 = 등장콘텐츠수 ÷ 같은 매체·기간 유효 콘텐츠 수."
for per in ["전체"] + [str(y) for y in YEARS]:
    fr = {m: ranked(view_media(m, per)) for m in MEDIA}
    dens = {m: denom(m, per) for m in MEDIA}
    RPT[f"표6_매체별_{per}"] = (f"표 6{'' if per == '전체' else '-' + per}. 매체별 다빈도 관심대상 ({per})", PERIOD_LABEL[per], [(None, side_by_side(fr, dens))])
    blocks = []
    for ar in AREAS:
        fa = {m: fr[m][fr[m]["주분야"] == ar].reset_index(drop=True) for m in MEDIA}
        blocks.append((f"■ {ar}", side_by_side(fa, dens)))
    RPT[f"표7_분야별_{per}"] = (f"표 7{'' if per == '전체' else '-' + per}. 6개 분야별 관심대상 ({per})", PERIOD_LABEL[per] + " 분야 = 주분야.", blocks)
    blocks = []
    for sg in SIGUN + ["전북전체", "지역미확정"]:
        fs, ds = {}, {}
        for m in MEDIA:
            v = view_region(m, per)
            v = v[v["지역"] == sg]
            fs[m] = ranked(v)
            ds[m] = int(v["유효콘텐츠분모"].iloc[0]) if len(v) else 0
        blocks.append((f"■ {sg}", side_by_side(fs, ds)))
    RPT[f"표8_시군별_{per}"] = (f"표 8{'' if per == '전체' else '-' + per}. 시군별 관심대상 ({per})", PERIOD_LABEL[per] +
                                " 시군 귀속 연결등급 A·B만. 분모 = 그 시군 지명이 본문에 있거나 그 시군 대상이 귀속된 콘텐츠 수(다중 귀속 콘텐츠는 각 시군에 1번씩).", blocks)

REACT_OUT = ["순위", "표준관심대상", "주분야", "등장콘텐츠수", "등장비중(%)", "고유출처수", "최대출처점유율", "관련조회수합계", "관련조회수중앙값",
             "관련좋아요합계", "관련좋아요중앙값", "관련댓글수합계", "관련댓글수중앙값", "대표반응점수중앙값", "반응점수유효N", "반응순위",
             "상위1콘텐츠기여율", "사분면", "숨은관심대상후보", "독립문구수", "대상소재시군", "반응의존구분", "원문검토판정", "숨은관심대상여부", "대표원문", "대표URL"]


def reaction_block(m: str, per: str) -> pd.DataFrame:
    v = view_reaction(m, per)
    v = ranked(v[v["지역"] == "매체 전체"])
    if v.empty:
        return pd.DataFrame(columns=REACT_OUT)
    v = v.copy()
    v["순위"] = np.arange(1, len(v) + 1)
    v["등장비중(%)"] = (v["등장비중"] * 100).round(3)
    v["상위1콘텐츠기여율"] = v["좋아요상위1콘텐츠기여율" if m == "인스타" else "조회수상위1콘텐츠기여율"]
    start, end = (None, None) if per == "전체" else (f"{per}-01-01", f"{per}-10-26")
    q = quotes(m, v["표준관심대상"], 1, start, end).set_index("target")
    v["대표원문"] = v["표준관심대상"].map(q["원문인용"]).fillna("")
    v["대표URL"] = v["표준관심대상"].map(q["URL"]).fillna("")
    return v[REACT_OUT]


for per in ["전체"] + [str(y) for y in YEARS]:
    RPT[f"표9_SNS반응_{per}"] = (f"표 9{'' if per == '전체' else '-' + per}. SNS 반응: 고반응·숨은 관심대상 ({per})", PERIOD_LABEL[per] +
        " 사분면 = 지지조건 충족 대상의 빈도·대표반응점수 중앙값 기준. 반응은 수집 시점(2026-10-31, 2023 보완분은 2026-11-05) 한 번의 누적값이라 "
        "오래된 게시물일수록 누적 기간이 길고, 2023년은 인기·조회수순 표집이라 반응이 높게 잡힘 → 연도 간 반응 수준 비교 금지.",
        [(f"■ {MK[m]}", reaction_block(m, per)) for m in ("인스타", "유튜브")])

# 표9-1 숨은 관심대상 검증(모든 기간, 매체 전체 수준 후보)
_h = T11[(T11["지역"] == "매체 전체") & (T11["숨은관심대상후보"] == "예") & T11["주분야"].isin(AREAS)].copy()
_h["상위1콘텐츠기여율"] = np.where(_h["매체"] == "인스타", _h["좋아요상위1콘텐츠기여율"], _h["조회수상위1콘텐츠기여율"])
_h = _h.sort_values(["매체", "비교기간ID", "숨은관심대상여부", "대표반응점수중앙값"], ascending=[True, True, False, False])
_qs = []
for m in ("인스타", "유튜브"):
    q = quotes(m, _h.loc[_h["매체"] == m, "표준관심대상"].unique(), 3)
    q = q.assign(매체=m)
    _qs.append(q)
_qs = pd.concat(_qs)
for i in (1, 2, 3):
    qi = _qs[_qs["순번"] == i].set_index(["매체", "target"])
    key = list(zip(_h["매체"], _h["표준관심대상"]))
    _h[f"원문{i}"] = [qi["원문인용"].get(k, "") for k in key]
    _h[f"URL{i}"] = [qi["URL"].get(k, "") for k in key]
H91 = _h[["매체", "분석기간", "표준관심대상", "주분야", "등장콘텐츠수", "고유출처수", "최대출처점유율", "대표반응점수중앙값", "반응점수유효N",
          "상위1콘텐츠기여율", "관련조회수합계", "관련좋아요합계", "관련댓글수합계", "독립문구수", "대상소재시군", "반응의존구분",
          "원문검토상태", "원문검토판정", "원문검토사유", "숨은관심대상여부",
          "원문1", "URL1", "원문2", "URL2", "원문3", "URL3"]]
RPT["표9-1_숨은관심대상_검증"] = ("표 9-1. 숨은 관심대상 후보와 검증(전체·연도별)",
    "후보 = 언급적음·반응높음 사분면 & 지지조건 충족 & 단일 출처 의존 아님. 확정(예) = ① 대상 소재 시군 확인(이름의 지명 또는 본문 지명 70% 이상) "
    "② 같은 안내문 반복을 하나로 본 독립 문구 5개 이상 ③ 반응이 여러 게시물·계정에 분산(상위 1개 기여율 < 0.5, 반응점수 유효 5건 이상, 한 계정 점유율 < 0.3, 계정 5곳 이상) "
    "④ 원문 검토에서 제외되지 않음. 하나라도 어긋나면 보류(사유 표시).", [(None, H91)])

# 표9-2 시군별 숨은 관심대상
_hs = T11[(T11["지역"] != "매체 전체") & (T11["숨은관심대상후보"] == "예") & T11["주분야"].isin(AREAS)]
RPT["표9-2_시군별숨은관심대상"] = ("표 9-2. 시군별 숨은 관심대상 후보(전체·연도별)", "시군 안에서의 빈도·반응 중앙값 기준 사분면. 열 정의는 표 9-1과 같음.",
    [(None, _hs[["매체", "분석기간", "지역", "표준관심대상", "주분야", "등장콘텐츠수", "고유출처수", "최대출처점유율", "대표반응점수중앙값",
                 "관련조회수합계", "관련좋아요합계", "반응의존구분", "숨은관심대상여부"]].sort_values(["매체", "분석기간", "지역", "대표반응점수중앙값"], ascending=[True, True, True, False]))])
print("보고서 연계 시트 데이터:", len(RPT))

In [ ]:
# ── 유형(HOT·RISING·STEADY·SEASONAL·EVENT) 전체 목록 ───────────────
def type_sheet(frames: dict, flag: str, cols: list, sort: list) -> pd.DataFrame:
    parts = []
    for m, f in frames.items():
        if f is None or not len(f):
            continue
        x = f.copy()
        x["매체"] = m
        parts.append(x)
    x = pd.concat(parts, ignore_index=True)
    if flag:
        x = x[x[flag] == True]
    x["등장비중(%)"] = (x["등장비중"] * 100).round(3)
    return x.sort_values(sort, ascending=[True] + [False] * (len(sort) - 1))[[c for c in cols if c in x.columns]]


RPT["표10_HOT"] = ("표 10. HOT(2026-01-01~10-26, 트랙 B) 전체", "비교 가능한 수집층의 2026년 동기간 다빈도 상위 30 중 최소 건수·출처 다양성 충족. 고반응여부는 SNS만.",
    [(None, type_sheet(HOT, "HOT", ["매체", "빈도순위", "표준관심대상", "주분야", "등장콘텐츠수", "등장비중(%)", "고유출처수", "최대출처점유율",
                                    "대표반응점수중앙값", "고반응여부", "선정근거"], ["매체", "등장콘텐츠수"]))])
_ris = type_sheet(RISE, "", ["매체", "유형", "유형내순위", "표준관심대상", "주분야", "기준등장수", "기준등장비중", "현재등장수", "현재등장비중", "비중비",
                             "천건당언급증가", "2024등장수", "평균대비비중비", "C기준_비중비", "민감도안정성", "2023보완수집민감", "표집변경가능성", "선정근거"],
                  ["매체", "현재등장수"])
_ris = _ris.sort_values(["매체", "유형", "유형내순위"])
RPT["표11_RISING"] = ("표 11. RISING·신규 관측·후보 전체(2025 동기간 → 2026 동기간, 트랙 B)",
    "유형: RISING(비중비 ≥ 1.5, 천건당 증가 ≥ 1, 관측월 ≥ 2, 지지조건) / 신규 관측(2024·2025 동기간 0건) / 단일 언론사 후보 / C 민감도 후보. "
    "표집변경가능성 = 인스타 수집 해시태그와 같은 표현.", [(None, _ris)])
RPT["표12_STEADY"] = ("표 12. STEADY(지속 관심형) 전체", "비교 가능한 3개 연도 이상, 연도별 3건 이상, 합계 20건 이상, 연도별 비중 변동계수 ≤ 0.5.",
    [(None, type_sheet(STEADY, "STEADY", ["매체", "표준관심대상", "주분야", "충족연도수", "전체등장수", "비중변동지표", "연도별등장수_원장참조", "선정근거"], ["매체", "전체등장수"]))])
RPT["표12-1_SEASONAL"] = ("표 12-1. SEASONAL(계절 반복형) 전체", "완전 관측·동일 표집 계절 2개 연도 이상에서 같은 계절 지수 ≥ 1.5.",
    [(None, type_sheet(SEASONAL, "SEASONAL", ["매체", "표준관심대상", "주분야", "피크계절", "반복확인완전연도수", "확인계절연도", "행사일정근거", "선정근거"], ["매체", "반복확인완전연도수"]))])
RPT["표12-2_EVENT_VIRAL"] = ("표 12-2. EVENT/VIRAL(사건형·단일콘텐츠형) 전체", "사건형 = 한 해 등장 비중의 50% 이상이 한 달에 집중. 단일콘텐츠형 = 반응 상위 1개 콘텐츠 기여율 ≥ 50%.",
    [(None, type_sheet(EVENT, "EVENT_VIRAL", ["매체", "표준관심대상", "주분야", "세부유형", "피크월", "피크월콘텐츠기여율", "상위1콘텐츠기여율", "연례반복여부", "사건맥락근거", "선정근거"], ["매체", "피크월콘텐츠기여율"]))])
RPT["참고_매체비교"] = ("참고. 매체 간 공통·특화 관심대상(보고서 3.7)", "세 매체 보고용 상위 100의 합집합.", [(None, T20)])

# ── 제외·보조 키워드 ──────────────────────────────────────
_ex = []
for m in MEDIA:
    a = AUX[m]
    g = a.groupby(["target", "judge"])["content_id"].nunique().rename("등장콘텐츠수").reset_index()
    g["매체"] = m
    _ex.append(g)
_ex = pd.concat(_ex)
_ex = _ex[_ex["등장콘텐츠수"] >= 3].rename(columns={"target": "표현", "judge": "처리유형"})
_rule = T05.drop_duplicates("유형").set_index("유형")
_ex["처리"] = _ex["처리유형"].map(_rule["처리_주순위제외보조맥락화"]).fillna("주순위 제외(보조)")
_ex["판정규칙"] = _ex["처리유형"].map(_rule["규칙"]).fillna("")
_ex = _ex.sort_values(["처리유형", "등장콘텐츠수"], ascending=[True, False])
_ad = pd.DataFrame([{"매체": r["매체"], "표현": "(콘텐츠 단위) " + r["나열표현"][:200], "처리유형": "광고·홍보성 나열 콘텐츠 제외",
                     "등장콘텐츠수": r["제외콘텐츠수"], "처리": "콘텐츠 전체 분석 제외", "판정규칙": "해시태그에 지역명+같은 표현 4개 이상"} for _, r in AD_LOG.iterrows()])
RPT["제외보조_키워드"] = ("제외·보조 처리 키워드(3건 이상)", "주순위에서 뺀 표현과 그 사유. 콘텐츠는 지우지 않고(광고 나열 콘텐츠만 예외) 같은 콘텐츠의 다른 대상은 집계됨.",
    [(None, pd.concat([_ad, _ex], ignore_index=True)[["매체", "처리유형", "표현", "등장콘텐츠수", "처리", "판정규칙"]])])

# ── 특산품 연결 기준 ─────────────────────────────────────
_fl = FOOD_LOG.copy()
_sp = []
for m in MEDIA:
    v = T09[m]
    x = v[v["표준관심대상"].isin(set(FOOD_LOG["연결대상"]) - {""})][["표준관심대상", "등장콘텐츠수", "고유출처수", "최소지지충족"]].assign(매체=m)
    _sp.append(x)
_sp = pd.concat(_sp).sort_values(["표준관심대상", "매체"])
RPT["특산품_연결기준"] = ("지역 특산품 표현의 연결·통합 기준",
    "식품명 단독(사과·치즈·고추장·홍삼·복분자 등, dict/지역연결필요식품.txt)은 어디서나 쓰여 지역 대상이라 단정하지 않음. 같은 콘텐츠에 '장수 사과', '임실치즈', '#순창 #고추장'처럼 "
    "시군명 바로 뒤에 식품명이 오면 '시군명+식품명' 표준 대상으로 연결(주요 관심대상 유지). 원래 '장수사과'로 붙여 쓴 표현은 처음부터 표준 대상. 연결 근거가 없는 단독 식품명만 보조 처리.",
    [("■ 처리 내역(원표현 → 연결 대상, 콘텐츠×표현 행 수)", _fl), ("■ 특산품 표준 대상의 전체 기간 등장(트랙 A)", _sp)])

# ── 시군 귀속 점검 ───────────────────────────────────────
_homo = {"무주(무주상보시·무주택 등)": r"무주(?=택|상보시|상|공처)", "지평선(일반어)": r"지평선(?!\s?(축제|쌀|산단|산업단지|농협|학교|중|고))",
         "청보리(타 지역)": r"청보리(?!\s?(밭|축제))", "선유도공원·선유도역(서울)": r"선유도(?=공원|역)", "춘향전(고전 작품)": r"춘향(?=전)",
         "군산복합체": r"군산(?=복합체)"}
_hrows = []
for name, pat in _homo.items():
    rx = re.compile(pat, re.I)
    for m in MEDIA:
        n = sum(1 for v in TEXT_IDX[m].values() if rx.search(v[0]))
        _hrows.append({"동음이의 패턴": name, "매체": m, "해당 표현이 나온 콘텐츠 수": n, "처리": "지명 단서에서 제외(다른 지명이 있으면 그 지명으로 귀속)"})
_g = pd.concat([REL[m].assign(매체=m).groupby(["매체", "귀속근거", "연결등급"]).size().rename("콘텐츠×대상 행 수").reset_index() for m in MEDIA])
RPT["시군귀속_점검"] = ("시군 귀속 점검(동음이의 지명·광역 접두·동반 출현)",
    "핵심 시군 순위는 연결등급 A(대상 이름에 지명 또는 인용 앞뒤 40자 안 지명)·B(같은 콘텐츠의 단일 시군, 분포 추정)만 사용. "
    "C(여러 시군 중 근접 지명 없음 = 단순 동반 출현)는 제외. 대상 바로 앞에 '전북'이 붙고 시군 지명이 떨어져 있으면 전북전체로 귀속(2차 검토 반영).",
    [("■ 동음이의 지명 패턴(2차 검토에서 지명 단서에서 제외)", pd.DataFrame(_hrows)), ("■ 귀속 근거·연결등급 분포", _g)])
print("유형·제외·특산품·귀속 시트 준비")

In [ ]:
# ── 2차 검토 변경 내역(이전 버전 대비) ─────────────────────────
PREV_DIR = OUT_DIR / CONFIG["PREV_VERSION"]
CHECK = pd.read_csv(DICT_DIR / "2차검토_원문확인.csv") if (DICT_DIR / "2차검토_원문확인.csv").exists() else pd.DataFrame()


def _rank_map(df: pd.DataFrame) -> pd.DataFrame:
    x = report_list(df).sort_values(["등장콘텐츠수", "고유출처수"], ascending=False).reset_index(drop=True)
    return pd.DataFrame({"보고용순위": np.arange(1, len(x) + 1), "등장콘텐츠수": x["등장콘텐츠수"].values}, index=x["표준관심대상"].values)


_AUX_JUDGE = {m: AUX[m].groupby("target")["judge"].agg(lambda s: s.mode().iat[0]) for m in MEDIA}
_RL_SET = {m: set(report_list(T09[m])["표준관심대상"]) for m in MEDIA}
_ALL_SET = {m: set(T09[m]["표준관심대상"]) for m in MEDIA}


def _status(t: str, m: str) -> str:
    """이번 결과에서 그 표현의 처리 상태."""
    if t in _AUX_JUDGE[m].index:
        return "제외·보조: " + _AUX_JUDGE[m][t]
    if t in _RL_SET[m]:
        return "보고용"
    if t in _ALL_SET[m]:
        x = T09[m].set_index("표준관심대상").loc[t]
        why = []
        if x["최소지지충족"] != "충족":
            why.append(f"지지조건 {x['최소지지충족']}")
        if bool(x["단일출처의존"]):
            why.append(f"단일 출처 의존(최대출처점유율 {x['최대출처점유율']:.2f})")
        if x["주분야"] not in AREAS:
            why.append("6개 분야 미분류")
        return "보고용 기준 미달: " + ", ".join(why)
    return "후보 없음(광고 나열 콘텐츠 제외·표현 통합 등)"


chg = []
if PREV_DIR.exists():
    prev09 = pd.read_csv(PREV_DIR / "csv" / "09_매체별키워드전체.csv", usecols=["매체", "표준관심대상", "등장콘텐츠수", "고유출처수", "최소지지충족", "단일출처의존", "주분야"])
    prev09["단일출처의존"] = prev09["단일출처의존"].astype(str).str.lower().isin(["true", "예"])
    for m in MEDIA:
        before = _rank_map(prev09[prev09["매체"] == m])
        after = _rank_map(T09[m])
        allt = T09[m].set_index("표준관심대상")["등장콘텐츠수"]
        top_b = set(before.index[:100])
        top_a = set(after.index[:100])
        manual = set(CHECK.loc[CHECK["매체"] == m, "대상"]) if len(CHECK) else set()
        for t in sorted((top_b ^ top_a) | manual, key=lambda t: before["보고용순위"].get(t, 9999)):
            rb, ra = before["보고용순위"].get(t), after["보고용순위"].get(t)
            row = {"매체": m, "키워드": t,
                   "변경 전 보고용순위": rb, "변경 전 콘텐츠 수": before["등장콘텐츠수"].get(t),
                   "변경 후 보고용순위": ra, "변경 후 콘텐츠 수": allt.get(t), "변경 후 처리": _status(t, m)}
            row["변경 구분"] = ("보고용 목록에서 제외" if (rb is not None and ra is None) else ("보고용 목록 신규" if (rb is None and ra is not None) else
                            ("상위 100 진입" if (ra or 9999) <= 100 < (rb or 9999) else ("상위 100 이탈" if (rb or 9999) <= 100 < (ra or 9999) else "순위 변동"))))
            if t in manual:
                c = CHECK[(CHECK["매체"] == m) & (CHECK["대상"] == t)].iloc[0]
                row["수정 사유"], row["원문 확인 결과"] = c["조치"], c["원문확인결과"]
            else:
                row["수정 사유"] = ("제외·보조 규칙 적용" if row["변경 후 처리"].startswith("제외") else ("표현 통합·광고 콘텐츠 제외" if row["변경 후 처리"].startswith("후보 없음") else
                                 ("광고 나열 콘텐츠 제외·재집계로 순위 하락" if row["변경 구분"] == "상위 100 이탈" else
                                  ("제외된 대상의 빈자리로 순위 상승" if row["변경 구분"] == "상위 100 진입" else "재집계"))))
                row["원문 확인 결과"] = ""
            chg.append(row)
CHG_COLS = ["매체", "키워드", "변경 구분", "변경 전 보고용순위", "변경 전 콘텐츠 수", "변경 후 보고용순위", "변경 후 콘텐츠 수", "변경 후 처리", "수정 사유", "원문 확인 결과"]
CHG = pd.DataFrame(chg, columns=CHG_COLS)
if len(CHECK):
    _done = set(zip(CHG["매체"], CHG["키워드"]))
    _mo = CHECK[[(a, b) not in _done for a, b in zip(CHECK["매체"], CHECK["대상"])]]
    if len(_mo):
        _mo = _mo.rename(columns={"대상": "키워드", "조치": "수정 사유", "원문확인결과": "원문 확인 결과"}).assign(**{"변경 구분": "검토(순위표 밖)"})
        _mo["변경 후 처리"] = [_status(t, m) if m in MEDIA else "" for t, m in zip(_mo["키워드"], _mo["매체"])]
        CHG = pd.concat([CHG, _mo.reindex(columns=CHG_COLS)], ignore_index=True)
CHG["_수기"] = CHG["원문 확인 결과"].fillna("").astype(str).ne("")
CHG = CHG.sort_values(["_수기", "매체"], ascending=[False, True]).drop(columns="_수기").reset_index(drop=True)
print("2차 검토 변경 내역", len(CHG), "행", CHG["변경 구분"].value_counts().to_dict() if len(CHG) else {})
RPT["2차검토_변경내역"] = ("2차 검토 반영: 수정·제외된 주요 키워드의 변경 전후", f"변경 전 = {CONFIG['PREV_VERSION']} 결과, 변경 후 = 이번 결과. 보고용 상위 100 진입·이탈 대상과 발주자 지적 대상을 모두 실음.",
                         [(None, CHG)])

# ── 원문 근거(키워드별 대표 콘텐츠 3건) ──────────────────────────
_need = {m: set() for m in MEDIA}
for m in MEDIA:
    for per in ["전체"] + [str(y) for y in YEARS]:
        _need[m] |= set(ranked(view_media(m, per))["표준관심대상"].head(300))
        v = view_region(m, per)
        _need[m] |= set(ranked(v).groupby("지역").head(30)["표준관심대상"])
    _need[m] |= set(RISE[m]["표준관심대상"].head(400))
    if m != "뉴스":
        _need[m] |= set(T11[(T11["매체"] == m) & (T11["숨은관심대상후보"] == "예")]["표준관심대상"])
    um = REL[m][REL[m]["최종귀속시군"] == "지역미확정"]["target"].value_counts()
    _need[m] |= set(um.head(100).index)
EVQ = []
for m in MEDIA:
    q = quotes(m, _need[m], 3)
    q["매체"] = m
    EVQ.append(q)
EVQ = pd.concat(EVQ, ignore_index=True).rename(columns={"target": "키워드", "date": "날짜"})
EVQ["날짜"] = pd.to_datetime(EVQ["날짜"]).dt.strftime("%Y-%m-%d")
EVQ = EVQ[["매체", "키워드", "순번", "날짜", "출처", "제목", "근거필드", "원문인용", "URL", "원자료ID", "조회수", "좋아요", "댓글수"]].sort_values(["매체", "키워드", "순번"])
RPT["원문근거_키워드별"] = ("키워드별 대표 원문(최대 3건)", "SNS는 대표반응점수 상위, 뉴스는 최신 기사. 원문인용은 원본 필드의 실제 문자열(앞뒤 25자). 뉴스는 URL이 원자료에 없어 언론사·원자료ID(입력 해시:행 번호)로 조회.",
                         [(None, EVQ)])
print("원문 근거", len(EVQ), "행")

In [ ]:
def write_report_workbook(path: Path):
    """RPT를 시트별로 씁니다. 시트 머리에 표 제목·분석기간·기준을 적고, 블록(소제목+표)을 아래로 쌓습니다."""
    order = (["표6_매체별_" + p for p in ["전체"] + [str(y) for y in YEARS]] + ["표7_분야별_" + p for p in ["전체"] + [str(y) for y in YEARS]]
             + ["표8_시군별_" + p for p in ["전체"] + [str(y) for y in YEARS]] + ["표9_SNS반응_" + p for p in ["전체"] + [str(y) for y in YEARS]]
             + ["표9-1_숨은관심대상_검증", "표9-2_시군별숨은관심대상", "표10_HOT", "표11_RISING", "표12_STEADY", "표12-1_SEASONAL", "표12-2_EVENT_VIRAL",
                "참고_매체비교", "특산품_연결기준", "시군귀속_점검", "제외보조_키워드", "2차검토_변경내역", "원문근거_키워드별"])
    with pd.ExcelWriter(path, engine="xlsxwriter", engine_kwargs={"options": {"strings_to_urls": False, "nan_inf_to_errors": True}}) as w:
        wb = w.book
        f_title = wb.add_format({"bold": True, "font_size": 13})
        f_note = wb.add_format({"font_color": "#555555", "text_wrap": False})
        f_sec = wb.add_format({"bold": True, "font_size": 11, "bg_color": "#E8EEF7"})
        f_head = wb.add_format({"bold": True, "bg_color": "#F2F2F2", "border": 1, "text_wrap": True, "valign": "vcenter"})
        f_tot = wb.add_format({"bold": True, "top": 1})
        # 목차
        toc = pd.DataFrame([{"시트": s, "보고서 표": RPT[s][0], "분석기간·기준": RPT[s][1][:250]} for s in order if s in RPT])
        toc.to_excel(w, sheet_name="00_목차", index=False, startrow=3)
        ws = wb.get_worksheet_by_name("00_목차")
        ws.write(0, 0, "「미지의 전북」 관심대상 분석 — 보고서 연계형 결과표", f_title)
        ws.write(1, 0, f"분석 버전 {ANALYSIS_VERSION}. {NOTE_COMMON} 전체 세부 지표는 미지의전북_관심대상분석_v2_*.xlsx(00~29 시트)와 csv/, 원장/에 있음.", f_note)
        ws.set_column(0, 0, 26); ws.set_column(1, 1, 52); ws.set_column(2, 2, 120)
        for s in order:
            if s not in RPT:
                continue
            title, note, blocks = RPT[s]
            r = 3
            first = True
            wb.add_worksheet(s)
            for sub, df in blocks:
                df = df.copy()
                for c in df.columns:
                    if df[c].dtype == bool:
                        df[c] = df[c].map({True: "예", False: ""})
                if sub:
                    wb.get_worksheet_by_name(s).write(r, 0, sub, f_sec)
                    r += 1
                df.to_excel(w, sheet_name=s, index=False, startrow=r)
                ws = wb.get_worksheet_by_name(s)
                for j, c in enumerate(df.columns):
                    ws.write(r, j, c, f_head)
                if len(df) and str(df.iloc[-1, 0]).startswith("전체"):
                    ws.set_row(r + len(df), None, f_tot)
                r += len(df) + 2
                first = False
            ws = wb.get_worksheet_by_name(s)
            ws.write(0, 0, title, f_title)
            ws.write(1, 0, "분석기간·기준: " + note, f_note)
            ws.freeze_panes(4 if not blocks[0][0] else 0, 1 if s.startswith(("표6", "표7", "표8")) else 0)
            cols = blocks[0][1].columns
            for j, c in enumerate(cols):
                width = 7 if c in ("순위", "순번") else (24 if ("키워드" in c or c in ("표준관심대상", "표현")) else
                         (60 if c.startswith(("원문", "대표원문", "제목", "수정 사유", "원문 확인")) else (14 if len(c) <= 8 else 16)))
                ws.set_column(j, j, width)
    return path


RPT_XLSX = write_report_workbook(OUT_V2 / f"미지의전북_보고서연계결과_v2_{RUN_TS}.xlsx")
(OUT_V2 / "csv" / "보고서연계").mkdir(exist_ok=True)
for s, (_, _, blocks) in RPT.items():
    df = pd.concat([b.assign(구분=(sub or "").strip("■ ")) for sub, b in blocks], ignore_index=True)
    df.to_csv(OUT_V2 / "csv" / "보고서연계" / f"{s}.csv", index=False, encoding="utf-8-sig")
_man = pd.read_csv(OUT_V2 / "manifest.csv")
_add = [{"파일": RPT_XLSX.name, "시트·범위": s, "행수": int(sum(len(b) for _, b in RPT[s][2])), "고유키": "", "고유키중복수": "", "SHA256": ""} for s in RPT]
_add += [{"파일": f"csv/보고서연계/{s}.csv", "시트·범위": "전체", "행수": int(sum(len(b) for _, b in RPT[s][2])), "고유키": "", "고유키중복수": "",
          "SHA256": file_sha256(OUT_V2 / "csv" / "보고서연계" / f"{s}.csv")} for s in RPT]
pd.concat([_man, pd.DataFrame(_add)], ignore_index=True).to_csv(OUT_V2 / "manifest.csv", index=False, encoding="utf-8-sig")
print("보고서 연계 결과:", RPT_XLSX, f"{RPT_XLSX.stat().st_size/1e6:.1f} MB")